Proses Prapipeline dan Integritas Data

NST

In [ ]:
"""
1 | Skor sentimen per dokumen (Pers. 3.3) dan NSt harian (Pers. 3.4)
====================================================================
Input  : FINAL 2.csv  (kolom wajib: label_final, tanggal_wib)
Output : dataset_skor.csv   — skor sentimen per dokumen
         nst_harian.csv     — NSt harian (rata-rata skor per hari)
         nst_periode.csv    — NSt per periode + total keseluruhan

Rumus:
  Skor dokumen (Pers. 3.3):  pos → +1,  neu → 0,  neg → −1
  NSt harian   (Pers. 3.4):  NSt_t = (1/N_t) * Σ skor_i
  NSt periode  :              NSt_p = (1/N_p) * Σ skor_i  untuk semua dokumen di periode p
"""

import sys
import numpy as np
import pandas as pd
from pathlib import Path

# ── konfigurasi ──────────────────────────────────────────────────────
from pathlib import Path

FILE_IN = Path(
    r"G:\My Drive\skripsi\proje\proje\data\hasil\hasil_visualisasi\FINAL 3.csv"
)

DIR_OUT = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil\Hasil Akhir")
OUT_DOK = DIR_OUT / "01_dataset_skor_sentimen.csv"
OUT_NST = DIR_OUT / "02_nst_harian.csv"
OUT_PERIODE = DIR_OUT / "03_nst_per_periode.csv"

PETA_SKOR = {"pos": 1.0, "neu": 0.0, "neg": -1.0}
VALID = set(PETA_SKOR.keys())

PERIODE = [
    ("Pra-peristiwa", "2025-06-01", "2025-08-24"),
    ("Puncak peristiwa", "2025-08-25", "2025-09-09"),
    ("Pasca-peristiwa", "2025-09-10", "2025-10-31"),
]

# ── 0. validasi file input ───────────────────────────────────────────
if not FILE_IN.exists():
    sys.exit(f"GAGAL: '{FILE_IN.name}' tidak ditemukan di {FILE_IN.parent}")

# ── 1. baca data ─────────────────────────────────────────────────────
df = pd.read_csv(FILE_IN, encoding="utf-8-sig", low_memory=False)
df.columns = [c.strip().lstrip("\ufeff") for c in df.columns]
print(f"Kolom yang ditemukan: {list(df.columns)}\n")

for kol in ["label_final", "tanggal_wib"]:
    if kol not in df.columns:
        sys.exit(f"GAGAL: kolom '{kol}' tidak ditemukan di {FILE_IN.name}")

# ── 2. bersihkan label_final ─────────────────────────────────────────
df["label_final"] = df["label_final"].astype(str).str.strip().str.lower()

PETA_NAMA = {
    "positive": "pos",
    "positif": "pos",
    "neutral": "neu",
    "netral": "neu",
    "negative": "neg",
    "negatif": "neg",
}
df["label_final"] = df["label_final"].replace(PETA_NAMA)

invalid = df[~df["label_final"].isin(VALID)]
if len(invalid) > 0:
    print(
        f"PERINGATAN: {len(invalid)} baris punya label_final tidak valid, akan dibuang:"
    )
    print(invalid["label_final"].value_counts().to_string(), "\n")
    df = df[df["label_final"].isin(VALID)].copy()

# ── 3. parse tanggal ─────────────────────────────────────────────────
df["tanggal"] = pd.to_datetime(df["tanggal_wib"], errors="coerce").dt.normalize()

n_tanggal_gagal = df["tanggal"].isna().sum()
if n_tanggal_gagal > 0:
    pct = n_tanggal_gagal / len(df) * 100
    print(
        f"PERINGATAN: {n_tanggal_gagal} baris ({pct:.2f}%) tanggal_wib gagal diparse, akan dibuang.\n"
    )
    if pct > 5:
        print("  → Lebih dari 5% gagal. Periksa format tanggal_wib di CSV.")
    df = df.dropna(subset=["tanggal"]).copy()

df = df.reset_index(drop=True)
print(f"Dokumen valid: {len(df)}")

# ── 4. hitung skor sentimen per dokumen (Pers. 3.3) ──────────────────
df["skor_sentimen"] = df["label_final"].map(PETA_SKOR)

# ── 5. tentukan periode setiap dokumen ────────────────────────────────
df["periode"] = "Di luar periode"
for nama, mulai, selesai in PERIODE:
    mask = (df["tanggal"] >= pd.Timestamp(mulai)) & (
        df["tanggal"] <= pd.Timestamp(selesai)
    )
    df.loc[mask, "periode"] = nama

n_luar = (df["periode"] == "Di luar periode").sum()
if n_luar > 0:
    print(
        f"PERINGATAN: {n_luar} dokumen berada di luar ketiga periode, tetap dihitung dalam total.\n"
    )

# ── 6. simpan skor per dokumen ────────────────────────────────────────
KOL_SIMPAN = [
    "tanggal",
    "tanggal_wib",
    "label_final",
    "sumber",
    "skor_sentimen",
    "periode",
]
kol_ada = [c for c in KOL_SIMPAN if c in df.columns]

for kol_id in ["row_key", "judul", "text_clean", "text_clean_bert"]:
    if kol_id in df.columns and kol_id not in kol_ada:
        kol_ada.insert(0, kol_id)

df[kol_ada].to_csv(OUT_DOK, index=False, encoding="utf-8-sig")
print(f"Disimpan: {OUT_DOK.name} ({len(df)} baris)")

# ── 7. hitung NSt harian (Pers. 3.4) ─────────────────────────────────
daily = (
    df.groupby("tanggal")
    .agg(
        n_total=("skor_sentimen", "size"),
        n_pos=("label_final", lambda x: (x == "pos").sum()),
        n_neu=("label_final", lambda x: (x == "neu").sum()),
        n_neg=("label_final", lambda x: (x == "neg").sum()),
        NSt=("skor_sentimen", "mean"),
    )
    .reset_index()
    .sort_values("tanggal")
    .reset_index(drop=True)
)

# tambahkan kolom periode ke daily
daily["periode"] = "Di luar periode"
for nama, mulai, selesai in PERIODE:
    mask = (daily["tanggal"] >= pd.Timestamp(mulai)) & (
        daily["tanggal"] <= pd.Timestamp(selesai)
    )
    daily.loc[mask, "periode"] = nama

daily.to_csv(OUT_NST, index=False, encoding="utf-8-sig")
print(f"Disimpan: {OUT_NST.name} ({len(daily)} hari)")

# ── 8. hitung NSt per periode + total ─────────────────────────────────
rows_periode = []
for nama, mulai, selesai in PERIODE:
    sub = df[df["periode"] == nama]
    rows_periode.append(
        {
            "periode": nama,
            "mulai": mulai,
            "selesai": selesai,
            "n_total": len(sub),
            "n_pos": (sub["label_final"] == "pos").sum(),
            "n_neu": (sub["label_final"] == "neu").sum(),
            "n_neg": (sub["label_final"] == "neg").sum(),
            "n_hari": sub["tanggal"].nunique(),
            "NSt": sub["skor_sentimen"].mean() if len(sub) > 0 else np.nan,
        }
    )

# baris total keseluruhan
rows_periode.append(
    {
        "periode": "TOTAL",
        "mulai": PERIODE[0][1],
        "selesai": PERIODE[-1][2],
        "n_total": len(df),
        "n_pos": (df["label_final"] == "pos").sum(),
        "n_neu": (df["label_final"] == "neu").sum(),
        "n_neg": (df["label_final"] == "neg").sum(),
        "n_hari": df["tanggal"].nunique(),
        "NSt": df["skor_sentimen"].mean(),
    }
)

df_periode = pd.DataFrame(rows_periode)
df_periode.to_csv(OUT_PERIODE, index=False, encoding="utf-8-sig")
print(f"Disimpan: {OUT_PERIODE.name} ({len(df_periode)} baris)")

# ── 9. ringkasan ──────────────────────────────────────────────────────
print("\n" + "=" * 60)
print("RINGKASAN")
print("=" * 60)

print(f"\nDistribusi label_final:")
print(df["label_final"].value_counts().to_string())

if "sumber" in df.columns:
    print(f"\nSumber label:")
    print(df["sumber"].value_counts().to_string())

print(
    f"\nRentang tanggal : {daily['tanggal'].min().date()} s.d. {daily['tanggal'].max().date()}"
)
print(f"Jumlah hari     : {len(daily)}")

print(f"\n{'=' * 60}")
print("NSt PER PERIODE")
print("=" * 60)
print(df_periode.to_string(index=False))

print(f"\nDeskriptif NSt harian:")
print(daily["NSt"].describe().round(4).to_string())

print(f"\nHari dengan sentimen negatif (NSt < 0) : {(daily['NSt'] < 0).sum()}")
print(f"Hari dengan sentimen netral  (NSt = 0) : {(daily['NSt'] == 0).sum()}")
print(f"Hari dengan sentimen positif (NSt > 0) : {(daily['NSt'] > 0).sum()}")

print(f"\n5 hari NSt terendah:")
print(
    daily.nsmallest(5, "NSt")[
        ["tanggal", "periode", "n_total", "n_pos", "n_neu", "n_neg", "NSt"]
    ].to_string(index=False)
)

print(f"\n5 hari NSt tertinggi:")
print(
    daily.nlargest(5, "NSt")[
        ["tanggal", "periode", "n_total", "n_pos", "n_neu", "n_neg", "NSt"]
    ].to_string(index=False)
)

print(f"\nLanjut → 2_volatilitas_ihsg.py")

Kolom yang ditemukan: ['row_key', 'tanggal_wib', 'text_clean_bert', 'label_final', 'sumber', 'status', 'n_pos', 'n_neg', 'n_neu', 'pemicu', 'label_lama', 'sumber_lama', 'berubah_dari_final2']

Dokumen valid: 14050
Disimpan: 01_dataset_skor_sentimen.csv (14050 baris)
Disimpan: 02_nst_harian.csv (153 hari)
Disimpan: 03_nst_per_periode.csv (4 baris)

RINGKASAN

Distribusi label_final:
label_final
neg    5930
pos    5114
neu    3006

Sumber label:
sumber
leksikon         9200
voting_3model    4850

Rentang tanggal : 2025-06-01 s.d. 2025-10-31
Jumlah hari     : 153

NSt PER PERIODE
         periode      mulai    selesai  n_total  n_pos  n_neu  n_neg  n_hari       NSt
   Pra-peristiwa 2025-06-01 2025-08-24     7605   2820   1672   3113      85 -0.038527
Puncak peristiwa 2025-08-25 2025-09-09     1998    553    356   1089      16 -0.268268
 Pasca-peristiwa 2025-09-10 2025-10-31     4447   1741    978   1728      52  0.002923
           TOTAL 2025-06-01 2025-10-31    14050   5114   3006   5930

Visualisasi untuk NST

In [ ]:
"""
Visualisasi NSt per periode
Input  : 02_nst_harian.csv, 03_nst_per_periode.csv, 01_dataset_skor_sentimen.csv (opsional)
Output : 6 grafik PNG di subfolder Visual/ dalam folder data
"""

from datetime import datetime
from pathlib import Path

import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

# ── konfigurasi ──────────────────────────────────────────────────────
DIR_MANUAL = None
AKAR_PENCARIAN = [
    Path(r"G:\My Drive\skripsi"),
    Path(r"G:\Drive Saya\skripsi"),
]
F_HARIAN = "02_nst_harian.csv"
F_PERIODE = "03_nst_per_periode.csv"
F_DOK = "01_dataset_skor_sentimen.csv"
KOLOM_LABEL_DOK = "label_final"
KOLOM_SUMBER_DOK = "sumber"
SUBFOLDER_OUTPUT = "Visual"

KOLOM_NST = "NSt"
SATUAN = "dokumen"
TAMPILKAN_JUDUL = True
FORMAT_ANGKA_ID = True
HENTIKAN_JIKA_TIDAK_KONSISTEN = True
DPI = 300

PERIODE = [
    ("Pra-peristiwa", "2025-06-01", "2025-08-24"),
    ("Puncak peristiwa", "2025-08-25", "2025-09-09"),
    ("Pasca-peristiwa", "2025-09-10", "2025-10-31"),
]
PERIODE_SOROTAN = "Puncak peristiwa"

WARNA_SENTIMEN = {"pos": "#2a78d6", "neu": "#9a9993", "neg": "#e34948"}
NAMA_SENTIMEN = {"pos": "Positif", "neu": "Netral", "neg": "Negatif"}
WARNA_MA = "#eb6834"
WARNA_VOLUME = "#7a7973"
WARNA_SOROTAN = "#9a9993"
SEL_NST = {"pos": "#dbe8f8", "neu": "#f0efec", "neg": "#f9dada"}
TINTA = "#0b0b0b"
TINTA_2 = "#52514e"
GARIS_BANTU = "#c3c2b7"
BULAN = [
    "Jan",
    "Feb",
    "Mar",
    "Apr",
    "Mei",
    "Jun",
    "Jul",
    "Agu",
    "Sep",
    "Okt",
    "Nov",
    "Des",
]

plt.rcParams.update(
    {
        "figure.dpi": 100,
        "figure.facecolor": "white",
        "axes.facecolor": "white",
        "axes.edgecolor": GARIS_BANTU,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "axes.grid": True,
        "axes.grid.axis": "y",
        "axes.axisbelow": True,
        "grid.color": "#e1e0d9",
        "grid.linewidth": 0.6,
        "font.size": 10,
        "axes.titlesize": 13,
        "axes.titleweight": "bold",
        "legend.frameon": False,
    }
)


# ── format angka dan tanggal ─────────────────────────────────────────
def desimal(v, d):
    s = f"{v:.{d}f}"
    return s.replace(".", ",") if FORMAT_ANGKA_ID else s


def ribuan(n):
    s = f"{int(round(float(n))):,}"
    return s.replace(",", ".") if FORMAT_ANGKA_ID else s


def _tick_desimal(v, _):
    if abs(v) < 1e-12:
        return "0"
    s = f"{v:g}"
    return s.replace(".", ",") if FORMAT_ANGKA_ID else s


def _tick_tanggal(x, _):
    t = mdates.num2date(x)
    return f"{t.day:02d} {BULAN[t.month - 1]}"


FMT_DESIMAL = mticker.FuncFormatter(_tick_desimal)
FMT_RIBUAN = mticker.FuncFormatter(lambda v, _: ribuan(v))
FMT_TANGGAL = mticker.FuncFormatter(_tick_tanggal)


# ── lokasi data ──────────────────────────────────────────────────────
def csv(nama):
    return nama if nama.lower().endswith(".csv") else f"{nama}.csv"


def cari_folder_data():
    harian, periode = csv(F_HARIAN), csv(F_PERIODE)
    if DIR_MANUAL is not None:
        folder = Path(DIR_MANUAL)
        kurang = [n for n in (harian, periode) if not (folder / n).is_file()]
        if kurang:
            raise FileNotFoundError(f"Di {folder} tidak ada: {kurang}")
        return folder

    kandidat = []
    for akar in AKAR_PENCARIAN:
        if not akar.is_dir():
            print(f"[lewati] folder tidak ada: {akar}")
            continue
        for f in sorted(akar.rglob(f"{Path(harian).stem}*")):
            if f.name.lower() != harian.lower():
                print(f"[perhatian] nama tidak persis '{harian}': {f}")
                continue
            if (f.parent / periode).is_file():
                kandidat.append(f.parent)
            else:
                print(f"[perhatian] {periode} tidak ada di {f.parent}")

    if not kandidat:
        daftar = "\n".join(f"  {a}" for a in AKAR_PENCARIAN)
        raise FileNotFoundError(
            f"{harian} dan {periode} tidak ditemukan bersama di:\n{daftar}\n"
            "Isi DIR_MANUAL dengan folder tempat kedua file itu berada."
        )
    if len(kandidat) > 1:
        daftar = "\n".join(f"  {k}" for k in kandidat)
        raise RuntimeError(
            f"Ditemukan lebih dari satu folder data:\n{daftar}\n"
            "Isi DIR_MANUAL dengan folder yang dimaksud."
        )
    return kandidat[0]


# ── baca dan validasi ────────────────────────────────────────────────
def cek_kolom(df, wajib, nama):
    kurang = [k for k in wajib if k not in df.columns]
    if kurang:
        raise KeyError(
            f"{nama}: kolom {kurang} tidak ada. Kolom tersedia: {list(df.columns)}"
        )


def label_periode(index):
    label = pd.Series(pd.NA, index=index, dtype="object")
    for nama, mulai, selesai in PERIODE:
        label[(index >= pd.Timestamp(mulai)) & (index <= pd.Timestamp(selesai))] = nama
    return label


def baca_harian(folder):
    nama = csv(F_HARIAN)
    df = pd.read_csv(folder / nama, encoding="utf-8-sig")
    df.columns = df.columns.str.strip()
    cek_kolom(df, ["tanggal", KOLOM_NST, "n_total"], nama)
    df["tanggal"] = pd.to_datetime(df["tanggal"]).dt.normalize()
    ganda = df.loc[df["tanggal"].duplicated(), "tanggal"]
    if not ganda.empty:
        raise ValueError(
            f"{nama}: ada tanggal ganda, contoh {ganda.dt.date.tolist()[:5]}"
        )
    df = df.sort_values("tanggal").set_index("tanggal")
    awal = min(df.index.min(), pd.Timestamp(PERIODE[0][1]))
    akhir = max(df.index.max(), pd.Timestamp(PERIODE[-1][2]))
    rentang = pd.date_range(awal, akhir, freq="D", name="tanggal")
    kosong = rentang.difference(df.index)
    return df.reindex(rentang), kosong


def baca_periode(folder):
    nama = csv(F_PERIODE)
    df = pd.read_csv(folder / nama, encoding="utf-8-sig")
    df.columns = df.columns.str.strip()
    cek_kolom(
        df, ["periode", KOLOM_NST, "n_pos", "n_neu", "n_neg", "n_total", "n_hari"], nama
    )
    df["periode"] = df["periode"].astype(str).str.strip()

    is_total = df["periode"].str.upper() == "TOTAL"
    if is_total.sum() > 1:
        raise ValueError(f"{nama}: baris TOTAL lebih dari satu")
    total = df[is_total].iloc[0] if is_total.any() else None

    per = df[~is_total]
    urutan = [p[0] for p in PERIODE]
    asing = sorted(set(per["periode"]) - set(urutan))
    hilang = [p for p in urutan if p not in set(per["periode"])]
    if asing or hilang or per["periode"].duplicated().any():
        raise ValueError(
            f"{nama}: nama periode tidak cocok dengan PERIODE. "
            f"Tidak dikenal: {asing}; tidak ada: {hilang}"
        )
    per = per.set_index("periode").loc[urutan].reset_index()

    if {"mulai", "selesai"} <= set(per.columns):
        beda = [
            f"{p}: file {b} s.d. {s}, konfigurasi {m} s.d. {e}"
            for (p, m, e), b, s in zip(PERIODE, per["mulai"], per["selesai"])
            if (pd.Timestamp(b), pd.Timestamp(s)) != (pd.Timestamp(m), pd.Timestamp(e))
        ]
        if beda:
            raise ValueError(
                f"{nama}: batas periode berbeda dengan PERIODE:\n  " + "\n  ".join(beda)
            )
    return per, total


def baca_dokumen(folder):
    f = folder / csv(F_DOK)
    if not f.is_file():
        print(
            f"[info] {csv(F_DOK)} tidak ditemukan, pemeriksaan tingkat dokumen dilewati"
        )
        return None
    kolom = [c.strip() for c in pd.read_csv(f, encoding="utf-8-sig", nrows=0).columns]
    pakai = {"tanggal", KOLOM_LABEL_DOK, KOLOM_SUMBER_DOK} & set(kolom)
    if not pakai:
        return pd.read_csv(f, encoding="utf-8-sig", usecols=[0])
    dok = pd.read_csv(f, encoding="utf-8-sig", usecols=lambda c: c.strip() in pakai)
    dok.columns = dok.columns.str.strip()
    return dok


def cek_konsistensi(daily, per, total, kosong, dok):
    masalah, rekap = [], []
    komponen = ["n_pos", "n_neu", "n_neg"]
    ada = daily["n_total"].notna()

    for (nama, mulai, selesai), (_, baris) in zip(PERIODE, per.iterrows()):
        potong = daily.loc[mulai:selesai]
        n_harian = int(potong["n_total"].sum())
        rekap.append(
            {
                "periode": nama,
                "n_total periode": int(baris["n_total"]),
                "n_total harian": n_harian,
                "n_hari periode": int(baris["n_hari"]),
                "hari kalender": len(potong),
                "hari berisi NSt": int(potong[KOLOM_NST].notna().sum()),
            }
        )
        if float(baris[komponen].astype(float).sum()) != float(baris["n_total"]):
            masalah.append(f"{nama}: n_pos + n_neu + n_neg tidak sama dengan n_total")
        if n_harian != int(baris["n_total"]):
            masalah.append(
                f"{nama}: n_total periode berbeda dengan jumlah n_total harian ({mulai} s.d. {selesai})"
            )

    if total is not None:
        for kol in komponen + ["n_total", "n_hari"]:
            if float(per[kol].sum()) != float(total[kol]):
                masalah.append(f"TOTAL: {kol} tidak sama dengan jumlah ketiga periode")

    if set(komponen) <= set(daily.columns):
        salah = (
            daily.loc[ada, komponen].sum(axis=1) != daily.loc[ada, "n_total"]
        ).sum()
        if salah:
            masalah.append(
                f"{csv(F_HARIAN)}: {salah} hari dengan n_pos + n_neu + n_neg tidak sama dengan n_total"
            )

    if "periode" in daily.columns:
        harapan = label_periode(daily.index)
        salah = (
            daily.loc[ada, "periode"].astype(str).str.strip() != harapan[ada]
        ).sum()
        if salah:
            masalah.append(
                f"{csv(F_HARIAN)}: {salah} hari berlabel periode yang tidak sesuai PERIODE"
            )

    luar = (ada & label_periode(daily.index).isna()).sum()
    if luar:
        masalah.append(f"{luar} hari data berada di luar rentang PERIODE")

    if dok is not None:
        nama_dok = csv(F_DOK)
        if total is not None and len(dok) != int(total["n_total"]):
            masalah.append(
                f"{nama_dok}: {ribuan(len(dok))} baris, sedangkan n_total TOTAL {ribuan(total['n_total'])}"
            )
        if {"tanggal", KOLOM_LABEL_DOK} <= set(dok.columns) and set(komponen) <= set(
            daily.columns
        ):
            label = dok[KOLOM_LABEL_DOK].astype(str).str.strip()
            asing = sorted(set(label) - {"pos", "neu", "neg"})
            if asing:
                masalah.append(f"{nama_dok}: nilai label tidak dikenal {asing}")
            tgl = pd.to_datetime(dok["tanggal"]).dt.normalize()
            acuan = daily.loc[ada, komponen].set_axis(["pos", "neu", "neg"], axis=1)
            hitung = pd.crosstab(tgl, label).reindex(
                index=acuan.index, columns=["pos", "neu", "neg"], fill_value=0
            )
            beda = (hitung != acuan).any(axis=1).sum()
            if beda:
                masalah.append(
                    f"{nama_dok}: {beda} hari dengan hitungan label berbeda dari {csv(F_HARIAN)}"
                )
            di_luar = (~tgl.isin(acuan.index)).sum()
            if di_luar:
                masalah.append(
                    f"{nama_dok}: {di_luar} dokumen bertanggal di luar {csv(F_HARIAN)}"
                )

    print("\nRekap konsistensi:")
    print(pd.DataFrame(rekap).to_string(index=False))
    print(f"Tanggal tanpa baris di {csv(F_HARIAN)}: {len(kosong)}")
    if dok is not None:
        print(f"Baris {csv(F_DOK)}: {ribuan(len(dok))}")
        if {KOLOM_SUMBER_DOK, KOLOM_LABEL_DOK} <= set(dok.columns):
            print(f"\nAsal label (kolom '{KOLOM_SUMBER_DOK}' x '{KOLOM_LABEL_DOK}'):")
            print(
                pd.crosstab(
                    dok[KOLOM_SUMBER_DOK],
                    dok[KOLOM_LABEL_DOK],
                    margins=True,
                    margins_name="Jumlah",
                ).to_string()
            )

    if masalah:
        print("\nPERINGATAN KONSISTENSI:")
        for m in masalah:
            print(f"  - {m}")
        if HENTIKAN_JIKA_TIDAK_KONSISTEN:
            raise ValueError(
                "Data tidak konsisten, grafik tidak dibuat. Lihat peringatan di atas."
            )
    else:
        print("\nSemua pemeriksaan konsistensi lolos.")


# ── komponen grafik ──────────────────────────────────────────────────
def batas(mulai, selesai):
    return (
        pd.Timestamp(mulai) - pd.Timedelta(hours=12),
        pd.Timestamp(selesai) + pd.Timedelta(hours=12),
    )


def tandai_periode(ax, tulis_nama=True):
    for i, (nama, mulai, selesai) in enumerate(PERIODE):
        a, b = batas(mulai, selesai)
        if nama == PERIODE_SOROTAN:
            ax.axvspan(a, b, color=WARNA_SOROTAN, alpha=0.15, linewidth=0, zorder=0)
        if i > 0:
            ax.axvline(a, color=TINTA_2, linewidth=0.8, linestyle="--", zorder=1)
        if tulis_nama:
            ax.text(
                mdates.date2num(a + (b - a) / 2),
                1.01,
                nama,
                transform=ax.get_xaxis_transform(),
                ha="center",
                va="bottom",
                fontsize=9,
                color=TINTA_2,
            )


def sumbu_tanggal(ax, daily):
    ax.set_xlim(
        daily.index.min() - pd.Timedelta(hours=12),
        daily.index.max() + pd.Timedelta(hours=12),
    )
    ax.xaxis.set_major_locator(mdates.WeekdayLocator(byweekday=mdates.MO, interval=2))
    ax.xaxis.set_major_formatter(FMT_TANGGAL)
    ax.set_xlabel("Tanggal (2025)")


def area_nst(ax, daily):
    nst = daily[KOLOM_NST]
    ax.fill_between(
        daily.index,
        nst,
        0,
        where=nst >= 0,
        color=WARNA_SENTIMEN["pos"],
        alpha=0.25,
        interpolate=True,
        linewidth=0,
    )
    ax.fill_between(
        daily.index,
        nst,
        0,
        where=nst < 0,
        color=WARNA_SENTIMEN["neg"],
        alpha=0.25,
        interpolate=True,
        linewidth=0,
    )
    ax.plot(daily.index, nst, color=TINTA, linewidth=1.0, label="NSt harian", zorder=3)
    ax.axhline(0, color=TINTA, linewidth=0.8, zorder=2)
    ax.yaxis.set_major_formatter(FMT_DESIMAL)


def judul(ax, teks, pad=10):
    if TAMPILKAN_JUDUL:
        ax.set_title(teks, pad=pad)


def simpan(fig, out, nama):
    fig.savefig(out / nama, dpi=DPI, bbox_inches="tight")
    plt.close(fig)
    print(f"Disimpan: {nama}")


# ── grafik ───────────────────────────────────────────────────────────
def grafik_1(daily, out):
    fig, ax = plt.subplots(figsize=(14, 5))
    tandai_periode(ax)
    area_nst(ax, daily)
    ma7 = daily[KOLOM_NST].rolling(7, center=True).mean()
    ax.plot(
        daily.index,
        ma7,
        color=WARNA_MA,
        linewidth=2,
        linestyle="--",
        label="Rata-rata bergerak 7 hari",
        zorder=4,
    )
    ax.set_ylabel("Net Sentiment (NSt)")
    sumbu_tanggal(ax, daily)
    judul(ax, "NSt Harian per Periode", pad=24)
    ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.13), ncol=2)
    simpan(fig, out, "1_nst_harian_timeseries.png")


def grafik_2(per, total, out):
    label = per["periode"].tolist()
    nilai = per[KOLOM_NST].astype(float).tolist()
    posisi = list(range(len(per)))
    if total is not None:
        label.append("Total")
        nilai.append(float(total[KOLOM_NST]))
        posisi.append(len(per) + 0.5)
    warna = [
        WARNA_SENTIMEN["pos"]
        if v > 0
        else WARNA_SENTIMEN["neg"]
        if v < 0
        else WARNA_SENTIMEN["neu"]
        for v in nilai
    ]
    fig, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(
        posisi, nilai, color=warna, width=0.6, edgecolor="white", linewidth=1.5
    )
    if total is not None:
        bars[-1].set_hatch("///")
        ax.axvline(len(per) - 0.25, color=GARIS_BANTU, linewidth=0.8, linestyle=":")
    ax.bar_label(
        bars,
        labels=[desimal(v, 4) for v in nilai],
        padding=3,
        fontsize=10,
        fontweight="bold",
        color=TINTA,
    )
    ax.axhline(0, color=TINTA, linewidth=0.8)
    ax.margins(y=0.15)
    ax.set_xticks(posisi)
    ax.set_xticklabels(label, fontsize=9)
    ax.set_ylabel("Net Sentiment (NSt)")
    ax.yaxis.set_major_formatter(FMT_DESIMAL)
    judul(ax, "NSt per Periode")
    simpan(fig, out, "2_nst_per_periode.png")


def grafik_3(per, out):
    n = per["n_total"].astype(float).replace(0, np.nan)
    x = np.arange(len(per))
    dasar = np.zeros(len(per))
    fig, ax = plt.subplots(figsize=(8, 5))
    for k in ("pos", "neu", "neg"):
        v = (per[f"n_{k}"].astype(float) / n * 100).fillna(0).to_numpy()
        b = ax.bar(
            x,
            v,
            0.55,
            bottom=dasar,
            color=WARNA_SENTIMEN[k],
            label=NAMA_SENTIMEN[k],
            edgecolor="white",
            linewidth=1.5,
        )
        ax.bar_label(
            b,
            labels=[f"{desimal(t, 1)}%" if t > 5 else "" for t in v],
            label_type="center",
            fontsize=9,
            fontweight="bold",
            color=TINTA if k == "neu" else "white",
        )
        dasar += v
    ax.set_ylim(0, 100)
    ax.set_xticks(x)
    ax.set_xticklabels(per["periode"], fontsize=9)
    ax.set_ylabel("Proporsi (%)")
    ax.yaxis.set_major_formatter(FMT_DESIMAL)
    judul(ax, "Distribusi Sentimen per Periode")
    ax.legend(loc="upper left", bbox_to_anchor=(1.01, 1))
    simpan(fig, out, "3_proporsi_sentimen_periode.png")


def grafik_4(daily, out):
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(14, 7), sharex=True, gridspec_kw={"hspace": 0.08}
    )
    tandai_periode(ax1)
    tandai_periode(ax2, tulis_nama=False)
    ax1.bar(
        daily.index,
        daily["n_total"].fillna(0),
        width=0.8,
        color=WARNA_VOLUME,
        linewidth=0,
    )
    ax1.set_ylabel(f"Jumlah {SATUAN}")
    ax1.yaxis.set_major_formatter(FMT_RIBUAN)
    area_nst(ax2, daily)
    ax2.set_ylabel("NSt")
    sumbu_tanggal(ax2, daily)
    judul(ax1, f"Volume {SATUAN.capitalize()} Harian dan NSt", pad=24)
    simpan(fig, out, "4_volume_dan_nst.png")


def grafik_5(per, out):
    x = np.arange(len(per))
    w = 0.26
    fig, ax = plt.subplots(figsize=(9, 5))
    for off, k in zip((-w, 0, w), ("pos", "neu", "neg")):
        v = per[f"n_{k}"].astype(float)
        b = ax.bar(
            x + off,
            v,
            w,
            color=WARNA_SENTIMEN[k],
            label=NAMA_SENTIMEN[k],
            edgecolor="white",
            linewidth=1.5,
        )
        ax.bar_label(
            b, labels=[ribuan(t) for t in v], padding=2, fontsize=9, color=TINTA
        )
    ax.margins(y=0.12)
    ax.set_xticks(x)
    ax.set_xticklabels(per["periode"], fontsize=9)
    ax.set_ylabel(f"Jumlah {SATUAN}")
    ax.yaxis.set_major_formatter(FMT_RIBUAN)
    judul(ax, f"Jumlah {SATUAN.capitalize()} per Sentimen dan Periode")
    ax.legend()
    simpan(fig, out, "5_jumlah_sentimen_periode.png")


def grafik_6(per, total, out):
    baris = [r for _, r in per.iterrows()]
    if total is not None:
        baris.append(total)

    def jumlah_persen(a, n):
        return (
            f"{ribuan(a)} ({desimal(float(a) / n * 100, 1)}%)" if n > 0 else ribuan(a)
        )

    isi, nilai_nst = [], []
    for r in baris:
        n = float(r["n_total"])
        v = float(r[KOLOM_NST])
        nilai_nst.append(v)
        isi.append(
            [
                "Total" if str(r["periode"]).upper() == "TOTAL" else r["periode"],
                ribuan(n),
                jumlah_persen(r["n_pos"], n),
                jumlah_persen(r["n_neu"], n),
                jumlah_persen(r["n_neg"], n),
                ribuan(r["n_hari"]),
                desimal(v, 4),
                "Positif" if v > 0 else "Negatif" if v < 0 else "Netral",
            ]
        )
    kolom = [
        "Periode",
        f"N {SATUAN}",
        "Positif",
        "Netral",
        "Negatif",
        "Hari",
        "NSt",
        "Arah",
    ]

    fig, ax = plt.subplots(figsize=(11, 0.4 * (len(isi) + 1)))
    ax.axis("off")
    ax.grid(False)
    tbl = ax.table(cellText=isi, colLabels=kolom, cellLoc="center", bbox=[0, 0, 1, 1])
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(9)
    tbl.auto_set_column_width(list(range(len(kolom))))
    for j in range(len(kolom)):
        tbl[0, j].set_facecolor("#37474F")
        tbl[0, j].set_text_props(color="white", fontweight="bold")
        if total is not None:
            tbl[len(isi), j].set_facecolor("#ECEFF1")
            tbl[len(isi), j].set_text_props(fontweight="bold")
    for i, v in enumerate(nilai_nst, start=1):
        tbl[i, 6].set_facecolor(
            SEL_NST["pos"] if v > 0 else SEL_NST["neg"] if v < 0 else SEL_NST["neu"]
        )
    judul(ax, "Ringkasan NSt per Periode")
    simpan(fig, out, "6_tabel_ringkasan.png")


# ── jalankan ─────────────────────────────────────────────────────────
def main():
    folder = cari_folder_data()
    diubah = datetime.fromtimestamp((folder / csv(F_HARIAN)).stat().st_mtime)
    print(f"Folder data : {folder}")
    print(f"{csv(F_HARIAN)} terakhir diubah: {diubah:%d-%m-%Y %H:%M}")

    daily, kosong = baca_harian(folder)
    per, total = baca_periode(folder)
    dok = baca_dokumen(folder)
    cek_konsistensi(daily, per, total, kosong, dok)

    out = folder / SUBFOLDER_OUTPUT
    out.mkdir(exist_ok=True)
    print()
    grafik_1(daily, out)
    grafik_2(per, total, out)
    grafik_3(per, out)
    grafik_4(daily, out)
    grafik_5(per, out)
    grafik_6(per, total, out)
    print(f"\nSemua grafik disimpan di: {out}")


if __name__ == "__main__":
    main()

[lewati] folder tidak ada: G:\Drive Saya\skripsi
Folder data : G:\My Drive\skripsi\proje\proje\data\hasil\Hasil Akhir
02_nst_harian.csv terakhir diubah: 25-09-2026 18:17

Rekap konsistensi:
         periode  n_total periode  n_total harian  n_hari periode  hari kalender  hari berisi NSt
   Pra-peristiwa             7605            7605              85             85               85
Puncak peristiwa             1998            1998              16             16               16
 Pasca-peristiwa             4447            4447              52             52               52
Tanggal tanpa baris di 02_nst_harian.csv: 0
Baris 01_dataset_skor_sentimen.csv: 14.050

Asal label (kolom 'sumber' x 'label_final'):
label_final     neg   neu   pos  Jumlah
sumber                                 
leksikon       4467   232  4501    9200
voting_3model  1463  2774   613    4850
Jumlah         5930  3006  5114   14050

Semua pemeriksaan konsistensi lolos.

Disimpan: 1_nst_harian_timeseries.png
Disimpan

Volatiliti IHSG

In [ ]:
"""2 | Return logaritmik dan realized volatility IHSG (Pers. 3.1 dan 3.2)."""
import numpy as np
import pandas as pd
from pathlib import Path

DIR = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil\Hasil Akhir")
DIR.mkdir(parents=True, exist_ok=True)

FILE_OUT = DIR / "ihsg_harian.csv"
# ── Parameter model ───────────────────────────────────────────────────
TICKER = "^JKSE"
WINDOW_VOL = 5  # jendela rolling std untuk σ
MULAI = "2025-03-01"  # awal unduhan (yfinance)
AKHIR = "2025-11-15"  # akhir unduhan (yfinance)
PERIODE = ("2025-06-01", "2025-10-31")  # batas periode analisis
MIN_LEADIN = 15  # minimal lead-in hari bursa

# ══════════════════════════════════════════════════════════════════════
# 1.  Muat harga penutupan
# ══════════════════════════════════════════════════════════════════════
if SUMBER == "arsip":
    if not FILE_ARSIP.exists():
        raise SystemExit(
            f"GAGAL: {FILE_ARSIP} tidak ada.\n"
            f"       Pastikan file ada di folder 'arsip/', atau ganti SUMBER = 'yfinance'."
        )
    px = pd.read_csv(FILE_ARSIP, parse_dates=["date"]).rename(
        columns={"date": "tanggal"}
    )
    if "close" not in px.columns:
        raise SystemExit(
            f"GAGAL: kolom 'close' tidak ditemukan di {FILE_ARSIP.name}.\n"
            f"       Kolom yang ada: {px.columns.tolist()}"
        )
    px = px[["tanggal", "close"]]

elif SUMBER == "yfinance":
    import yfinance as yf

    raw = yf.download(TICKER, start=MULAI, end=AKHIR, auto_adjust=False, progress=False)
    if raw.empty:
        raise SystemExit(
            "GAGAL: unduhan yfinance kosong. Periksa koneksi atau rentang tanggal."
        )
    if isinstance(raw.columns, pd.MultiIndex):
        raw.columns = raw.columns.get_level_values(0)
    px = raw[["Close"]].dropna().reset_index()
    px.columns = ["tanggal", "close"]

else:
    raise SystemExit(
        f"GAGAL: SUMBER '{SUMBER}' tidak dikenali. Pilih 'arsip' atau 'yfinance'."
    )

# ══════════════════════════════════════════════════════════════════════
# 2.  Bersihkan kolom tanggal
# ══════════════════════════════════════════════════════════════════════
t = pd.to_datetime(px["tanggal"], errors="coerce")
if getattr(t.dt, "tz", None) is not None:
    t = t.dt.tz_localize(None)
px["tanggal"] = t.dt.normalize()

px = px.dropna(subset=["tanggal", "close"])
px = px[px["close"] > 0].sort_values("tanggal").reset_index(drop=True)

if px["tanggal"].duplicated().any():
    raise SystemExit("GAGAL: ada tanggal ganda pada deret harga.")

# ══════════════════════════════════════════════════════════════════════
# 3.  Hitung return logaritmik (R) dan realized volatility (σ)
# ══════════════════════════════════════════════════════════════════════
px["R"] = np.log(px["close"]) - np.log(px["close"].shift(1))
px["sigma"] = px["R"].rolling(WINDOW_VOL).std(ddof=1)

# ══════════════════════════════════════════════════════════════════════
# 4.  Simpan output
# ══════════════════════════════════════════════════════════════════════
out = px[["tanggal", "close", "R", "sigma"]].copy()  # ← .copy() ditambahkan
out.to_csv(FILE_OUT, index=False, encoding="utf-8-sig")

# ══════════════════════════════════════════════════════════════════════
# 5.  Ringkasan & validasi
# ══════════════════════════════════════════════════════════════════════
a, b = pd.Timestamp(PERIODE[0]), pd.Timestamp(PERIODE[1])
inti = out[(out["tanggal"] >= a) & (out["tanggal"] <= b)]
leadin = int((out["tanggal"] < a).sum())
nan_inti = int(inti["sigma"].isna().sum())

print("=" * 60)
print("  RINGKASAN — 2_volatilitas_ihsg.py")
print("=" * 60)
print(f"  Sumber data              : {SUMBER}")
print(f"  Ticker / jendela σ       : {TICKER} / {WINDOW_VOL} hari, ddof=1")
print(f"  Hari bursa pada berkas   : {len(out)}")
print(
    f"  Rentang                  : {out['tanggal'].min().date()} s.d. "
    f"{out['tanggal'].max().date()}"
)
print(
    f"  NaN sigma seluruh deret  : {int(out['sigma'].isna().sum())}  "
    f"(wajar {WINDOW_VOL - 1} baris pertama)"
)
print(
    f"  NaN R seluruh deret      : {int(out['R'].isna().sum())}  "
    f"(wajar 1 baris pertama)"
)
print(f"  Hari bursa {PERIODE[0]} s.d. {PERIODE[1]} : {len(inti)}")
print(f"  NaN sigma dalam periode  : {nan_inti}")
print(f"  Lead-in sebelum periode  : {leadin} hari bursa")
print("-" * 60)

if leadin < MIN_LEADIN:
    print(
        f"  ⚠ PERINGATAN: lead-in {leadin} hari < {MIN_LEADIN}. "
        f"Observasi awal periode akan gugur saat lag dibentuk."
    )
if nan_inti:
    raise SystemExit("GAGAL: masih ada sigma kosong di dalam periode analisis.")

print(f"\n  Deskriptif seluruh deret:")
print(out[["R", "sigma"]].describe().to_string())

print(f"\n  Deskriptif periode analisis:")
print(inti[["R", "sigma"]].describe().to_string())

print(f"\n  Lima baris pertama periode analisis:")
print(inti.head().to_string(index=False))

ekstrem = inti.reindex(inti["R"].abs().sort_values(ascending=False).index).head(5)
print(
    f"\n  Lima hari dengan |R| terbesar "
    f"(bahan verifikasi trading halt / pengumuman BEI):"
)
print(ekstrem[["tanggal", "close", "R", "sigma"]].to_string(index=False))

print(f"\n  ✓ Disimpan → {FILE_OUT}")
print("  Lanjut  → 3_panel_final.py")

  RINGKASAN — 2_volatilitas_ihsg.py
  Sumber data              : yfinance
  Ticker / jendela σ       : ^JKSE / 5 hari, ddof=1
  Hari bursa pada berkas   : 167
  Rentang                  : 2025-03-03 s.d. 2025-11-14
  NaN sigma seluruh deret  : 5  (wajar 4 baris pertama)
  NaN R seluruh deret      : 1  (wajar 1 baris pertama)
  Hari bursa 2025-06-01 s.d. 2025-10-31 : 105
  NaN sigma dalam periode  : 0
  Lead-in sebelum periode  : 52 hari bursa
------------------------------------------------------------

  Deskriptif seluruh deret:
                R       sigma
count  166.000000  162.000000
mean     0.001505    0.010189
std      0.012726    0.008221
min     -0.082318    0.001137
25%     -0.003686    0.005542
50%      0.002627    0.008111
75%      0.008617    0.011881
max      0.046815    0.051019

  Deskriptif periode analisis:
                R       sigma
count  105.000000  105.000000
mean     0.001229    0.008358
std      0.009414    0.003952
min     -0.026073    0.001137
25%     -0.

In [20]:
"""2b | Visualisasi return logaritmik dan realized volatility IHSG."""
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import matplotlib.ticker as mticker
from pathlib import Path

# ── Konfigurasi path ──────────────────────────────────────────────────
DIR = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil\Hasil Akhir")
FILE_IN = DIR / "ihsg_harian.csv"
DIR_VIS = DIR / "Visual" / "Volatilitas ihsg"
DIR_VIS.mkdir(parents=True, exist_ok=True)

# ── Periode peristiwa ─────────────────────────────────────────────────
PERIODE = {
    "Pra-peristiwa": ("2025-06-01", "2025-08-24"),
    "Puncak peristiwa": ("2025-08-25", "2025-09-09"),
    "Pasca-peristiwa": ("2025-09-10", "2025-10-31"),
}
WARNA_PERIODE = {
    "Pra-peristiwa": "#2196F3",
    "Puncak peristiwa": "#F44336",
    "Pasca-peristiwa": "#4CAF50",
}
ALPHA_SHADE = 0.12

# ── Muat data ────────────────────────────────────────────────────────
df = pd.read_csv(FILE_IN, parse_dates=["tanggal"])
df = df.sort_values("tanggal").reset_index(drop=True)

print(f"Memuat {FILE_IN.name}: {len(df)} baris")
print(f"Rentang: {df['tanggal'].min().date()} s.d. {df['tanggal'].max().date()}")


# ── Helper ───────────────────────────────────────────────────────────
def tambah_shading(ax):
    for label, (t0, t1) in PERIODE.items():
        ax.axvspan(
            pd.Timestamp(t0),
            pd.Timestamp(t1),
            alpha=ALPHA_SHADE,
            color=WARNA_PERIODE[label],
            label=label,
        )


def format_sumbu_x(ax):
    ax.xaxis.set_major_locator(mdates.MonthLocator())
    ax.xaxis.set_major_formatter(mdates.DateFormatter("%b\n%Y"))
    ax.tick_params(axis="x", labelsize=8)


# ── Gaya global ──────────────────────────────────────────────────────
plt.rcParams.update(
    {
        "figure.dpi": 150,
        "savefig.dpi": 150,
        "font.size": 9,
        "axes.titlesize": 11,
        "axes.labelsize": 9,
        "legend.fontsize": 7.5,
        "figure.facecolor": "white",
    }
)

# ══════════════════════════════════════════════════════════════════════
# Grafik 1 — Harga Penutupan IHSG
# ══════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(df["tanggal"], df["close"], color="#1565C0", linewidth=0.9, label="Close")
tambah_shading(ax)
format_sumbu_x(ax)
ax.set_title("Harga Penutupan IHSG Harian")
ax.set_ylabel("Harga (IDR)")
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:,.0f}"))
ax.legend(loc="upper left", framealpha=0.9)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fpath = DIR_VIS / "ihsg_1_close.png"
fig.savefig(fpath)
plt.close(fig)
print(f"  ✓ {fpath.name}")

# ══════════════════════════════════════════════════════════════════════
# Grafik 2 — Return Logaritmik Harian (R)
# ══════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 4))
ax.bar(
    df["tanggal"],
    df["R"],
    width=1.0,
    color="#1E88E5",
    alpha=0.7,
    label="R (log return)",
)
ax.axhline(0, color="black", linewidth=0.5)
tambah_shading(ax)
format_sumbu_x(ax)
ax.set_title("Return Logaritmik Harian IHSG")
ax.set_ylabel("R = ln(Pₜ / Pₜ₋₁)")
ax.legend(loc="upper left", framealpha=0.9)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fpath = DIR_VIS / "ihsg_2_return.png"
fig.savefig(fpath)
plt.close(fig)
print(f"  ✓ {fpath.name}")

# ══════════════════════════════════════════════════════════════════════
# Grafik 3 — Realized Volatility (σ)
# ══════════════════════════════════════════════════════════════════════
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    df["tanggal"],
    df["sigma"],
    color="#E53935",
    linewidth=0.9,
    label="σ (rolling 5 hari)",
)
tambah_shading(ax)
format_sumbu_x(ax)
ax.set_title("Realized Volatility IHSG (Rolling 5 Hari, ddof=1)")
ax.set_ylabel("σ")
ax.legend(loc="upper left", framealpha=0.9)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fpath = DIR_VIS / "ihsg_3_volatility.png"
fig.savefig(fpath)
plt.close(fig)
print(f"  ✓ {fpath.name}")

# ══════════════════════════════════════════════════════════════════════
# Grafik 4 — Histogram + KDE Return Logaritmik
# ══════════════════════════════════════════════════════════════════════
r_clean = df["R"].dropna()

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.hist(
    r_clean,
    bins=50,
    density=True,
    color="#42A5F5",
    alpha=0.65,
    edgecolor="white",
    linewidth=0.4,
    label="Histogram R",
)

try:
    from scipy.stats import gaussian_kde, norm
    import numpy as np

    kde = gaussian_kde(r_clean)
    x_kde = np.linspace(r_clean.min() - 0.005, r_clean.max() + 0.005, 500)
    ax.plot(x_kde, kde(x_kde), color="#E53935", linewidth=1.2, label="KDE")

    mu, std = r_clean.mean(), r_clean.std()
    ax.plot(
        x_kde,
        norm.pdf(x_kde, mu, std),
        color="#9E9E9E",
        linewidth=1.0,
        linestyle="--",
        label=f"Normal(μ={mu:.5f}, σ={std:.5f})",
    )
except ImportError:
    print("    (scipy tidak tersedia — KDE & normal overlay dilewati)")

ax.axvline(0, color="black", linewidth=0.5, linestyle=":")
ax.set_title("Distribusi Return Logaritmik Harian IHSG")
ax.set_xlabel("R")
ax.set_ylabel("Density")
ax.legend(loc="upper right", framealpha=0.9)
ax.grid(axis="y", alpha=0.3)
fig.tight_layout()
fpath = DIR_VIS / "ihsg_4_histogram_return.png"
fig.savefig(fpath)
plt.close(fig)
print(f"  ✓ {fpath.name}")

# ══════════════════════════════════════════════════════════════════════
# Grafik 5 — Overlay Return + Volatility dengan Shading Periode
# ══════════════════════════════════════════════════════════════════════
fig, ax1 = plt.subplots(figsize=(11, 5))

color_r = "#1565C0"
ax1.bar(
    df["tanggal"], df["R"], width=1.0, color=color_r, alpha=0.45, label="R (log return)"
)
ax1.axhline(0, color="black", linewidth=0.4)
ax1.set_ylabel("R = ln(Pₜ / Pₜ₋₁)", color=color_r)
ax1.tick_params(axis="y", labelcolor=color_r)
format_sumbu_x(ax1)

ax2 = ax1.twinx()
color_s = "#E53935"
ax2.plot(
    df["tanggal"], df["sigma"], color=color_s, linewidth=1.1, label="σ (rolling 5 hari)"
)
ax2.set_ylabel("σ (realized volatility)", color=color_s)
ax2.tick_params(axis="y", labelcolor=color_s)

tambah_shading(ax1)

lines1, labels1 = ax1.get_legend_handles_labels()
lines2, labels2 = ax2.get_legend_handles_labels()
ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left", framealpha=0.9)

ax1.set_title("Return Logaritmik & Realized Volatility IHSG — Overlay per Periode")
ax1.grid(axis="y", alpha=0.2)
fig.tight_layout()
fpath = DIR_VIS / "ihsg_5_overlay_return_vol.png"
fig.savefig(fpath)
plt.close(fig)
print(f"  ✓ {fpath.name}")

# ══════════════════════════════════════════════════════════════════════
# Selesai
# ══════════════════════════════════════════════════════════════════════
print(f"\n{'=' * 50}")
print(f"  5 grafik disimpan di: {DIR_VIS}")
print(f"{'=' * 50}")
print("  Lanjut → 3_panel_final.py")

Memuat ihsg_harian.csv: 167 baris
Rentang: 2025-03-03 s.d. 2025-11-14
  ✓ ihsg_1_close.png
  ✓ ihsg_2_return.png
  ✓ ihsg_3_volatility.png
  ✓ ihsg_4_histogram_return.png
  ✓ ihsg_5_overlay_return_vol.png

  5 grafik disimpan di: G:\My Drive\skripsi\proje\proje\data\hasil\Hasil Akhir\Visual\Volatilitas ihsg
  Lanjut → 3_panel_final.py


Panel Final (Gabung panel, bentuk fitur lag, tetapkan N.)

In [ ]:
"""3 | Gabung panel, bentuk fitur lag, tetapkan N. Versi perbaikan Tahap 1."""
import shutil
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd

BASE = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil")
DIR = BASE / "Hasil Akhir"
IN_NST = DIR / "02_nst_harian.csv"
IN_IHSG = DIR / "ihsg_harian.csv"
OUT_PANEL = DIR / "panel_final.csv"
OUT_RINGKAS = DIR / "ringkasan_panel.csv"
DIR_ARSIP = DIR / "arsip"

LAG_VOL = 5
LAG_RET = 2
AWAL = pd.Timestamp("2025-06-01")
AKHIR = pd.Timestamp("2025-10-31")
BTS_PRA = pd.Timestamp("2025-08-24")
BTS_PUNCAK = pd.Timestamp("2025-09-09")
KOL_TEKS = ["n_total", "n_pos", "n_neu", "n_neg"]


def baca(p, wajib):
    if not p.exists():
        raise SystemExit(f"GAGAL: {p} tidak ditemukan.")
    df = pd.read_csv(p, encoding="utf-8-sig")
    df.columns = df.columns.str.strip()
    kurang = [k for k in wajib if k not in df.columns]
    if kurang:
        raise SystemExit(f"GAGAL: {p.name} tidak memuat kolom {kurang}.")
    if df.empty:
        raise SystemExit(f"GAGAL: {p.name} kosong.")
    df["tanggal"] = pd.to_datetime(df["tanggal"]).dt.normalize()
    if df["tanggal"].duplicated().any():
        raise SystemExit(f"GAGAL: {p.name} memuat tanggal ganda.")
    return df.sort_values("tanggal").reset_index(drop=True)


ihsg = baca(IN_IHSG, ["tanggal", "close", "R", "sigma"])[
    ["tanggal", "close", "R", "sigma"]
]
nst = baca(IN_NST, ["tanggal", "NSt"] + KOL_TEKS)

lama, cadangan = None, None
if OUT_PANEL.exists():
    lama = pd.read_csv(OUT_PANEL, encoding="utf-8-sig")
    lama.columns = lama.columns.str.strip()
    lama["tanggal"] = pd.to_datetime(lama["tanggal"]).dt.normalize()
    DIR_ARSIP.mkdir(parents=True, exist_ok=True)
    cadangan = DIR_ARSIP / f"panel_final_sebelum_{datetime.now():%Y%m%d_%H%M%S}.csv"
    shutil.copy2(OUT_PANEL, cadangan)

n_dok = int(nst["n_total"].sum())
teks_luar = ~nst["tanggal"].between(AWAL, AKHIR)

bursa = ihsg[["tanggal"]].rename(columns={"tanggal": "tgl_bursa"})
peta = pd.merge_asof(
    nst, bursa, left_on="tanggal", right_on="tgl_bursa", direction="forward"
)
gugur = peta["tgl_bursa"].isna()
keluar = peta["tgl_bursa"].notna() & ~peta["tgl_bursa"].between(AWAL, AKHIR)

peta = peta.loc[~gugur].copy()
peta["_bobot"] = peta["NSt"] * peta["n_total"]
agr = peta.groupby("tgl_bursa", as_index=False)[["_bobot"] + KOL_TEKS].sum()
agr["NSt"] = agr["_bobot"] / agr["n_total"]
agr = agr.drop(columns="_bobot").rename(columns={"tgl_bursa": "tanggal"})

panel = ihsg.merge(
    agr[["tanggal", "NSt"] + KOL_TEKS], on="tanggal", how="left", validate="one_to_one"
)

for i in range(1, LAG_VOL + 1):
    panel[f"sigma_lag{i}"] = panel["sigma"].shift(i)
for i in range(1, LAG_RET + 1):
    panel[f"R_lag{i}"] = panel["R"].shift(i)
panel["NSt_lag1"] = panel["NSt"].shift(1)

n_bursa_total = len(panel)
leadin = int((panel["tanggal"] < AWAL).sum())
leadout = int((panel["tanggal"] > AKHIR).sum())

panel = panel.loc[panel["tanggal"].between(AWAL, AKHIR)].reset_index(drop=True)
n_bursa_periode = len(panel)

tanpa_teks = panel.loc[panel["n_total"].isna(), "tanggal"]
if len(tanpa_teks):
    print("Hari bursa dalam periode tanpa dokumen:")
    print([str(t.date()) for t in tanpa_teks])
    raise SystemExit(
        "BERHENTI: perlakuan hari bursa tanpa dokumen belum diputuskan. "
        "NSt tidak diisi 0 secara otomatis."
    )
panel[KOL_TEKS] = panel[KOL_TEKS].astype(int)

panel["periode"] = np.select(
    [panel["tanggal"] <= BTS_PRA, panel["tanggal"] <= BTS_PUNCAK],
    ["Pra", "Puncak"],
    default="Pasca",
)

kolom_lag = (
    [f"sigma_lag{i}" for i in range(1, LAG_VOL + 1)]
    + [f"R_lag{i}" for i in range(1, LAG_RET + 1)]
    + ["NSt_lag1"]
)
wajib = kolom_lag + ["sigma", "R"]
kosong = panel[wajib].isna()
baris_gugur = panel.loc[kosong.any(axis=1)]
bersih = panel.loc[~kosong.any(axis=1)].reset_index(drop=True)

if (
    not bersih["tanggal"].is_monotonic_increasing
    or bersih["tanggal"].duplicated().any()
):
    raise SystemExit("GAGAL: tanggal panel tidak monoton atau ganda.")
if bersih[wajib + ["NSt"] + KOL_TEKS].isna().any().any():
    raise SystemExit("GAGAL: masih ada NaN di panel.")
if (bersih["n_total"] <= 0).any():
    raise SystemExit("GAGAL: ada baris dengan n_total <= 0.")
if (bersih["tanggal"].dt.dayofweek >= 5).any():
    raise SystemExit("GAGAL: ada tanggal akhir pekan di panel.")
if not bersih["tanggal"].between(AWAL, AKHIR).all():
    raise SystemExit("GAGAL: ada tanggal di luar periode.")

bersih.to_csv(OUT_PANEL, index=False, encoding="utf-8-sig")

hari_kerja = pd.bdate_range(AWAL, AKHIR)
libur = hari_kerja.difference(ihsg["tanggal"])
dok_panel = int(bersih["n_total"].sum())
dok_gugur_lag = int(baris_gugur["n_total"].sum())

ringkasan = {
    "hari_bursa_berkas_ihsg": n_bursa_total,
    "lead_in_hari_bursa": leadin,
    "lead_out_hari_bursa": leadout,
    "hari_kerja_periode": len(hari_kerja),
    "hari_bursa_periode": n_bursa_periode,
    "hari_kalender_teks": len(nst),
    "dokumen_total": n_dok,
    "dokumen_tanggal_di_luar_periode": int(nst.loc[teks_luar, "n_total"].sum()),
    "dokumen_tanpa_hari_bursa": int(nst.loc[gugur.values, "n_total"].sum()),
    "dokumen_dipetakan_ke_luar_periode": int(nst.loc[keluar.values, "n_total"].sum()),
    "baris_gugur_lag": len(baris_gugur),
    "dokumen_pada_baris_gugur_lag": dok_gugur_lag,
    "N_panel_final": len(bersih),
    "dokumen_masuk_panel": dok_panel,
    "rentang_awal": str(bersih["tanggal"].min().date()),
    "rentang_akhir": str(bersih["tanggal"].max().date()),
}
pd.DataFrame([ringkasan]).to_csv(OUT_RINGKAS, index=False, encoding="utf-8-sig")

print("=" * 66)
print("  LANGKAH 3 - PANEL FINAL (perbaikan Tahap 1)")
print("=" * 66)
print(
    f"Berkas IHSG                      : {n_bursa_total} hari bursa, "
    f"{ihsg['tanggal'].min().date()} s.d. {ihsg['tanggal'].max().date()}"
)
print(f"Lead-in / lead-out               : {leadin} / {leadout} hari bursa")
print(f"Hari kerja {AWAL.date()} s.d. {AKHIR.date()} : {len(hari_kerja)}")
print(f"Hari bursa dalam periode         : {n_bursa_periode}")
print(f"Hari kerja tanpa perdagangan     : {len(libur)}")
print(f"  {[str(t.date()) for t in libur]}")
print(f"Hari kalender ber-teks           : {len(nst)} ({n_dok} dokumen)")
print(
    f"Dokumen bertanggal luar periode  : {ringkasan['dokumen_tanggal_di_luar_periode']}"
)
print(f"Dokumen tanpa hari bursa         : {ringkasan['dokumen_tanpa_hari_bursa']}")
print(
    f"Dokumen dipetakan ke luar periode: {ringkasan['dokumen_dipetakan_ke_luar_periode']}"
)
print(f"Hari bursa periode tanpa dokumen : {len(tanpa_teks)}")

print(f"\nBaris gugur karena lag           : {len(baris_gugur)}")
for _, b in baris_gugur.iterrows():
    kol = [k for k in wajib if pd.isna(b[k])]
    print(f"  {b['tanggal'].date()}  n_total={int(b['n_total'])}  kolom kosong: {kol}")

print(f"\n--- N PANEL FINAL                : {len(bersih)}")
print(
    f"Rentang                          : {ringkasan['rentang_awal']} s.d. {ringkasan['rentang_akhir']}"
)
print(f"Dokumen masuk panel (NSt_t)      : {dok_panel} dari {n_dok}")
print(f"Dokumen pada baris gugur lag     : {dok_gugur_lag}")
print(
    f"Tanggal ganda / akhir pekan      : {int(bersih['tanggal'].duplicated().sum())} / "
    f"{int((bersih['tanggal'].dt.dayofweek >= 5).sum())}"
)
print(f"Baris dengan n_total = 0         : {int((bersih['n_total'] == 0).sum())}")

print("\nBaris per bulan:")
print(bersih.groupby(bersih["tanggal"].dt.to_period("M")).size().to_string())
print("\nBaris per periode:")
print(bersih["periode"].value_counts().reindex(["Pra", "Puncak", "Pasca"]).to_string())
for p in ["Pra", "Puncak", "Pasca"]:
    sub = bersih.loc[bersih["periode"] == p, "tanggal"]
    print(f"  {p:6s}: {sub.min().date()} s.d. {sub.max().date()}")

print("\nDeskriptif:")
print(bersih[["sigma", "R", "NSt", "NSt_lag1", "n_total"]].describe().to_string())

print("\nKorelasi Pearson:")
print(f"  NSt      vs sigma : {bersih['NSt'].corr(bersih['sigma']):+.4f}")
print(f"  NSt_lag1 vs sigma : {bersih['NSt_lag1'].corr(bersih['sigma']):+.4f}")
print("\nKorelasi per periode:")
for p in ["Pra", "Puncak", "Pasca"]:
    sub = bersih[bersih["periode"] == p]
    if len(sub) > 2:
        print(f"  {p:6s}: N={len(sub):3d}, r={sub['NSt'].corr(sub['sigma']):+.4f}")
    else:
        print(f"  {p:6s}: N={len(sub):3d}, (terlalu sedikit)")

print("\n" + "=" * 66)
print("  PEMBANDING PANEL LAMA")
print("=" * 66)
if lama is None:
    print("panel_final.csv lama tidak ada, pembanding dilewati.")
else:
    print(f"Cadangan panel lama              : {cadangan}")
    print(
        f"Panel lama                       : {len(lama)} baris, "
        f"{lama['tanggal'].min().date()} s.d. {lama['tanggal'].max().date()}"
    )
    print(f"  sebelum {AWAL.date()}           : {int((lama['tanggal'] < AWAL).sum())}")
    print(
        f"  dalam periode                  : {int(lama['tanggal'].between(AWAL, AKHIR).sum())}"
    )
    print(
        f"  setelah {AKHIR.date()}           : {int((lama['tanggal'] > AKHIR).sum())}"
    )
    if "n_total" in lama.columns:
        print(f"  baris lama dengan n_total = 0  : {int((lama['n_total'] == 0).sum())}")
    hanya_lama = lama.loc[~lama["tanggal"].isin(bersih["tanggal"]), "tanggal"]
    hanya_baru = bersih.loc[~bersih["tanggal"].isin(lama["tanggal"]), "tanggal"]
    print(f"Tanggal hanya di panel lama      : {len(hanya_lama)}")
    print(
        f"  dalam periode: {[str(t.date()) for t in hanya_lama if AWAL <= t <= AKHIR]}"
    )
    print(f"Tanggal hanya di panel baru      : {[str(t.date()) for t in hanya_baru]}")
    gabung = bersih.merge(lama, on="tanggal", suffixes=("", "_lama"))
    kol_banding = [
        k
        for k in bersih.columns
        if k != "tanggal"
        and f"{k}_lama" in gabung.columns
        and pd.api.types.is_numeric_dtype(bersih[k])
    ]
    print(f"Selisih mutlak maksimum pada {len(gabung)} tanggal yang sama:")
    for k in kol_banding:
        d = (gabung[k] - gabung[f"{k}_lama"]).abs().max()
        print(f"  {k:10s}: {d:.3e}")
    if "periode_lama" in gabung.columns:
        print(
            f"  periode   : {int((gabung['periode'] != gabung['periode_lama']).sum())} label berbeda"
        )

print(f"\nDisimpan: {OUT_PANEL}")
print(f"Disimpan: {OUT_RINGKAS}")

  LANGKAH 3 - PANEL FINAL (perbaikan Tahap 1)
Berkas IHSG                      : 167 hari bursa, 2025-03-03 s.d. 2025-11-14
Lead-in / lead-out               : 52 / 10 hari bursa
Hari kerja 2025-06-01 s.d. 2025-10-31 : 110
Hari bursa dalam periode         : 105
Hari kerja tanpa perdagangan     : 5
  ['2025-06-06', '2025-06-09', '2025-06-27', '2025-08-18', '2025-09-05']
Hari kalender ber-teks           : 153 (14050 dokumen)
Dokumen bertanggal luar periode  : 0
Dokumen tanpa hari bursa         : 0
Dokumen dipetakan ke luar periode: 0
Hari bursa periode tanpa dokumen : 0

Baris gugur karena lag           : 1
  2025-06-02  n_total=85  kolom kosong: ['NSt_lag1']

--- N PANEL FINAL                : 104
Rentang                          : 2025-06-03 s.d. 2025-10-31
Dokumen masuk panel (NSt_t)      : 13965 dari 14050
Dokumen pada baris gugur lag     : 85
Tanggal ganda / akhir pekan      : 0 / 0
Baris dengan n_total = 0         : 0

Baris per bulan:
tanggal
2025-06    17
2025-07    23
2025-08    

"4 | Statistik deskriptif dan visualisasi — Bab IV.

In [ ]:
"""4 | Statistik deskriptif dan visualisasi, Bab IV.

Output CSV : hasil_visualisasi/Tabel Deskriptif/tabel_4_1 ... tabel_4_8
Output PNG : hasil_visualisasi/Deskriptif/gambar_4_1 ... gambar_4_4
"""
import matplotlib

matplotlib.use("Agg")
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from scipy import stats as sp_stats

# ── Jalur ────────────────────────────────────────────────────
BASE = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil")
DIR = BASE / "Hasil Akhir"
DIR_VIS = BASE / "hasil_visualisasi"
DIR_TBL = DIR_VIS / "Tabel Deskriptif"
DIR_GBR = DIR_VIS / "Deskriptif"

for _d in (DIR_TBL, DIR_GBR):
    _d.mkdir(parents=True, exist_ok=True)

IN_KORPUS = DIR_VIS / "FINAL 3.csv"
IN_SKOR = DIR / "01_dataset_skor_sentimen.csv"
IN_NST = DIR / "02_nst_harian.csv"
IN_PANEL = DIR / "panel_final.csv"
IN_IHSG = DIR / "ihsg_harian.csv"

OUT_T1 = DIR_TBL / "tabel_4_1_penyaringan_data.csv"
OUT_T2 = DIR_TBL / "tabel_4_2_distribusi_subperiode.csv"
OUT_T3 = DIR_TBL / "tabel_4_3_komposisi_sumber_label.csv"
OUT_T4 = DIR_TBL / "tabel_4_4_deskriptif_variabel.csv"
OUT_T5 = DIR_TBL / "tabel_4_5_rata2_subperiode.csv"
OUT_T6 = DIR_TBL / "tabel_4_6_distribusi_sentimen.csv"
OUT_T7 = DIR_TBL / "tabel_4_7_korelasi_nst_sigma.csv"
OUT_T8 = DIR_TBL / "tabel_4_8_uji_normalitas.csv"

OUT_G1 = DIR_GBR / "gambar_4_1_volume_unggahan_harian.png"
OUT_G2 = DIR_GBR / "gambar_4_2_nst_vs_volatilitas.png"
OUT_G3 = DIR_GBR / "gambar_4_3_histogram_nst.png"
OUT_G4 = DIR_GBR / "gambar_4_4_scatter_nst_sigma.png"

# ── Arsip PNG versi lama (folder lokal lama) ─────────────────
DIR_LEGACY = Path(r"c:\Users\EZRA\Downloads\tempat\Visual")
if DIR_LEGACY.exists():
    _arsip = DIR_LEGACY.parent / "arsip"
    _arsip.mkdir(exist_ok=True)
    for _png in DIR_LEGACY.glob("*.png"):
        _dest = _arsip / _png.name
        if not _dest.exists():
            _png.rename(_dest)
            print(f"ARSIP: {_png.name} -> arsip/")

# ── Konstanta ────────────────────────────────────────────────
SUBPERIODE = [
    ("Pra-peristiwa", "2025-06-01", "2025-08-24"),
    ("Peristiwa/Demonstrasi", "2025-08-25", "2025-09-09"),
    ("Pasca-peristiwa", "2025-09-10", "2025-10-31"),
]
URUT = [n for n, _, _ in SUBPERIODE]
LUAR = "Di luar sub-periode"
PETA = {"pos": "Positif", "neu": "Netral", "neg": "Negatif"}
COL_LABEL = "label_final"
COL_SUMBER = "sumber"
ALPHA = 0.05

_BATAS = [(n, pd.Timestamp(a), pd.Timestamp(b)) for n, a, b in SUBPERIODE]

plt.rcParams.update(
    {
        "figure.dpi": 150,
        "savefig.dpi": 300,
        "axes.grid": True,
        "grid.alpha": 0.3,
        "font.size": 10,
    }
)


# ── Validasi ─────────────────────────────────────────────────
def _cek_berkas(p: Path):
    if not p.exists():
        raise SystemExit(f"GAGAL: {p.name} belum ada. Jalankan langkah sebelumnya.")
    if pd.read_csv(p, encoding="utf-8-sig", nrows=1).empty:
        raise SystemExit(f"GAGAL: {p.name} kosong (hanya header).")


def _cek_kolom(df: pd.DataFrame, kolom, nama: str):
    hilang = [c for c in kolom if c not in df.columns]
    if hilang:
        raise SystemExit(
            f"GAGAL: kolom {hilang} tidak ditemukan di {nama}.\n"
            f"Kolom tersedia: {list(df.columns)}"
        )


for _f in [IN_KORPUS, IN_SKOR, IN_NST, IN_PANEL, IN_IHSG]:
    _cek_berkas(_f)

# ── Muat data ────────────────────────────────────────────────
dok = pd.read_csv(IN_SKOR, encoding="utf-8-sig", low_memory=False)
nst = pd.read_csv(IN_NST, encoding="utf-8-sig")
panel = pd.read_csv(IN_PANEL, encoding="utf-8-sig")
ihsg = pd.read_csv(IN_IHSG, encoding="utf-8-sig")

_cek_kolom(dok, ["tanggal", COL_LABEL, COL_SUMBER], IN_SKOR.name)
_cek_kolom(nst, ["tanggal"], IN_NST.name)
_cek_kolom(panel, ["tanggal", "NSt", "sigma", "R", "n_total"], IN_PANEL.name)
_cek_kolom(ihsg, ["tanggal"], IN_IHSG.name)

nst["tanggal"] = pd.to_datetime(nst["tanggal"])
panel["tanggal"] = pd.to_datetime(panel["tanggal"])
panel = panel.sort_values("tanggal").reset_index(drop=True)

dok["tanggal"] = pd.to_datetime(dok["tanggal"])
ada_jam = bool((dok["tanggal"] != dok["tanggal"].dt.normalize()).any())
dok["tanggal"] = dok["tanggal"].dt.normalize()

print(f"INFO  Kolom label  : {COL_LABEL}")
print(f"INFO  Kolom sumber : {COL_SUMBER}")
if ada_jam:
    print(
        "PERINGATAN: kolom tanggal di skor sentimen memuat komponen jam; "
        "dinormalisasi ke tanggal kalender sebelum dipetakan ke hari bursa."
    )


# ── Fungsi bantu ─────────────────────────────────────────────
def periode(ts):
    for nama, a, b in _BATAS:
        if a <= ts <= b:
            return nama
    return LUAR


def garis_peristiwa(ax):
    _, a, b = _BATAS[1]
    ax.axvspan(a, b, color="#e0e0e0", zorder=0, label="Periode peristiwa")


def tabel_frekuensi(seri: pd.Series, judul: str, peta=None, urutan=None):
    vc = seri.value_counts(dropna=False)
    if urutan:
        idx = [k for k in urutan if k in vc.index] + [
            k for k in vc.index if k not in urutan
        ]
        vc = vc.reindex(idx)
    peta = peta or {}
    nama = [peta.get(k, k) if pd.notna(k) else "(kosong)" for k in vc.index]
    t = pd.DataFrame(
        {
            judul: nama,
            "Jumlah": vc.values.astype(int),
            "Persentase (%)": (vc.values / len(seri) * 100).round(2),
        }
    )
    t.loc[len(t)] = ["Total", int(vc.sum()), 100.0]
    return t


# ── Pemetaan dokumen ke hari bursa ───────────────────────────
bursa = (
    ihsg[["tanggal"]]
    .assign(tanggal=lambda d: pd.to_datetime(d["tanggal"]))
    .dropna()
    .drop_duplicates()
    .sort_values("tanggal")
    .rename(columns={"tanggal": "tgl_bursa"})
)

dok = pd.merge_asof(
    dok.sort_values("tanggal"),
    bursa,
    left_on="tanggal",
    right_on="tgl_bursa",
    direction="forward",
)

n_tanpa_bursa = int(dok["tgl_bursa"].isna().sum())
dok["tgl_efektif"] = dok["tgl_bursa"].fillna(dok["tanggal"])
dok["subperiode"] = dok["tgl_efektif"].apply(periode)
panel["subperiode"] = panel["tanggal"].apply(periode)

if n_tanpa_bursa:
    print(
        f"PERINGATAN: {n_tanpa_bursa} dokumen tanpa hari bursa berikutnya; "
        f"subperiode memakai tanggal kalender."
    )

# ── Rekonsiliasi dengan panel ────────────────────────────────
dok_per_hari = dok.groupby("tgl_bursa").size()
panel_per_hari = panel.set_index("tanggal")["n_total"]
rekon = pd.DataFrame(
    {
        "dok": dok_per_hari.reindex(panel_per_hari.index).fillna(0).astype(int),
        "panel": panel_per_hari.fillna(0).astype(int),
    }
)
n_hari_beda = int((rekon["dok"] != rekon["panel"]).sum())
n_dok_luar_panel = int(
    (dok["tgl_bursa"].notna() & ~dok["tgl_bursa"].isin(panel["tanggal"])).sum()
)

# ── Tabel 4.1  Tahapan penyaringan data ──────────────────────
n_korpus = len(pd.read_csv(IN_KORPUS, encoding="utf-8-sig", low_memory=False))
n_dikeluarkan = n_korpus - len(dok)
if n_dikeluarkan < 0:
    print(
        f"PERINGATAN: dokumen berlabel ({len(dok)}) melebihi korpus awal "
        f"({n_korpus}). Periksa apakah {IN_KORPUS.name} korpus yang benar."
    )

t1 = pd.DataFrame(
    {
        "Tahapan Penyaringan Data": [
            "Dokumen korpus awal",
            "Dokumen tanpa mayoritas tiga model (dikeluarkan)",
            "Dokumen berlabel mayoritas",
            "Hari kalender dengan teks",
            "Hari perdagangan pada panel akhir",
            "Dokumen masuk panel akhir",
        ],
        "Jumlah": [
            n_korpus,
            n_dikeluarkan,
            len(dok),
            int(nst["tanggal"].nunique()),
            len(panel),
            int(panel["n_total"].sum()),
        ],
    }
)
t1.to_csv(OUT_T1, index=False, encoding="utf-8-sig")

# ── Tabel 4.2  Distribusi sub-periode ────────────────────────
n_luar = int((dok["subperiode"] == LUAR).sum())
if n_luar:
    print(f"PERINGATAN: {n_luar} dokumen di luar ketiga sub-periode.")

rows_t2 = []
for s in URUT + ([LUAR] if n_luar else []):
    rows_t2.append(
        {
            "Sub-Periode": s,
            "Jumlah Unggahan": int((dok["subperiode"] == s).sum()),
            "Hari Perdagangan": int((panel["subperiode"] == s).sum()),
        }
    )
t2 = pd.DataFrame(rows_t2)
t2["Persentase (%)"] = (t2["Jumlah Unggahan"] / len(dok) * 100).round(2)
t2["Rata-rata per Hari"] = (
    t2["Jumlah Unggahan"] / t2["Hari Perdagangan"].replace(0, np.nan)
).round(1)
t2.loc[len(t2)] = [
    "Total",
    int(t2["Jumlah Unggahan"].sum()),
    int(t2["Hari Perdagangan"].sum()),
    round(t2["Jumlah Unggahan"].sum() / len(dok) * 100, 2),
    np.nan,
]
t2.to_csv(OUT_T2, index=False, encoding="utf-8-sig")

# ── Tabel 4.3  Komposisi sumber label ────────────────────────
t3 = tabel_frekuensi(dok[COL_SUMBER], "Sumber Label")
t3.to_csv(OUT_T3, index=False, encoding="utf-8-sig")


# ── Tabel 4.4  Statistik deskriptif ──────────────────────────
def q1(x):
    return x.quantile(0.25)


def q3(x):
    return x.quantile(0.75)


vars_desc = panel[["NSt", "sigma", "R", "n_total"]].copy()
vars_desc.columns = [
    "Net Sentimen Harian (X)",
    "Volatilitas IHSG (Y)",
    "Return Harian",
    "Volume Unggahan",
]

desc = vars_desc.agg(
    ["count", "mean", "std", "min", q1, "median", q3, "max", "skew", "kurt"]
).T
desc.columns = [
    "N",
    "Mean",
    "Std",
    "Min",
    "Q1",
    "Median",
    "Q3",
    "Max",
    "Skewness",
    "Excess Kurtosis",
]
desc = desc.round(6)
desc["N"] = desc["N"].astype(int)
desc = desc.rename_axis("Variabel")
desc.to_csv(OUT_T4, encoding="utf-8-sig")

# ── Tabel 4.5  Rata-rata per sub-periode ─────────────────────
t5 = (
    panel.groupby("subperiode")[["NSt", "sigma", "R"]]
    .agg(["mean", "std", "count"])
    .reindex(URUT)
)
t5.columns = ["_".join(c) for c in t5.columns]
t5 = t5.round(6).rename_axis("Sub-Periode").reset_index()
t5.to_csv(OUT_T5, index=False, encoding="utf-8-sig")

# ── Tabel 4.6  Distribusi sentimen ───────────────────────────
t6 = tabel_frekuensi(
    dok[COL_LABEL], "Kelas Sentimen", peta=PETA, urutan=list(PETA)
).rename(columns={"Jumlah": "Jumlah Unggahan"})
t6.to_csv(OUT_T6, index=False, encoding="utf-8-sig")

# ── Tabel 4.7  Korelasi NSt vs sigma ─────────────────────────
pair = panel.dropna(subset=["NSt", "sigma"])
r_pearson, p_pearson = sp_stats.pearsonr(pair["NSt"], pair["sigma"])
r_spearman, p_spearman = sp_stats.spearmanr(pair["NSt"], pair["sigma"])
t7 = pd.DataFrame(
    {
        "Metode": ["Pearson", "Spearman"],
        "N": [len(pair), len(pair)],
        "r": [round(float(r_pearson), 6), round(float(r_spearman), 6)],
        "p-value": [f"{float(p_pearson):.4e}", f"{float(p_spearman):.4e}"],
        f"Signifikan (α={ALPHA})": [
            "Ya" if p_pearson < ALPHA else "Tidak",
            "Ya" if p_spearman < ALPHA else "Tidak",
        ],
    }
)
t7.to_csv(OUT_T7, index=False, encoding="utf-8-sig")

# ── Tabel 4.8  Uji normalitas Jarque-Bera ────────────────────
jb_rows = []
for col, label in [("NSt", "Net Sentimen"), ("sigma", "Volatilitas"), ("R", "Return")]:
    series = panel[col].dropna()
    jb_stat, jb_p = sp_stats.jarque_bera(series)
    jb_rows.append(
        {
            "Variabel": label,
            "N": len(series),
            "JB Statistic": round(float(jb_stat), 4),
            "p-value": f"{float(jb_p):.4e}",
            f"Normal (α={ALPHA})": "Ya" if float(jb_p) >= ALPHA else "Tidak",
        }
    )
t8 = pd.DataFrame(jb_rows)
t8.to_csv(OUT_T8, index=False, encoding="utf-8-sig")

# ── Gambar 4.1  Volume unggahan harian ───────────────────────
vol = dok.groupby("tanggal").size()

fig, ax = plt.subplots(figsize=(11, 3.8))
garis_peristiwa(ax)
ax.bar(vol.index, vol.values, width=0.8, color="#1f4e79", edgecolor="none")
ax.set_xlabel("Tanggal")
ax.set_ylabel("Jumlah Unggahan")
ax.xaxis.set_major_locator(mdates.MonthLocator())
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.autofmt_xdate(rotation=30)
fig.tight_layout()
fig.savefig(OUT_G1)
plt.close(fig)

# ── Gambar 4.2  NSt vs volatilitas (dua sumbu) ───────────────
fig, ax1 = plt.subplots(figsize=(11, 4.5))
garis_peristiwa(ax1)
ax1.plot(
    panel["tanggal"], panel["NSt"], color="#1f4e79", lw=1.4, label="Net Sentimen (NSt)"
)
ax1.axhline(0, color="0.5", lw=0.8, ls="--")
ax1.set_ylabel("Net Sentimen Harian", color="#1f4e79")
ax1.tick_params(axis="y", labelcolor="#1f4e79")

ax2 = ax1.twinx()
ax2.grid(False)
ax2.plot(
    panel["tanggal"], panel["sigma"], color="#c00000", lw=1.4, label="Volatilitas (σ)"
)
ax2.set_ylabel("Volatilitas IHSG", color="#c00000")
ax2.tick_params(axis="y", labelcolor="#c00000")

h1, l1 = ax1.get_legend_handles_labels()
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left", frameon=False, fontsize=9)
ax1.set_xlabel("Tanggal")
ax1.xaxis.set_major_locator(mdates.MonthLocator())
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
fig.autofmt_xdate(rotation=30)
fig.tight_layout()
fig.savefig(OUT_G2)
plt.close(fig)

# ── Gambar 4.3  Histogram NSt ────────────────────────────────
nst_valid = panel["NSt"].dropna()

fig, ax = plt.subplots(figsize=(7, 4))
ax.hist(nst_valid, bins=25, color="#1f4e79", edgecolor="white", alpha=0.85)
ax.axvline(
    nst_valid.mean(),
    color="#c00000",
    lw=1.5,
    ls="--",
    label=f"Rata-rata = {nst_valid.mean():.4f}",
)
ax.axvline(0, color="0.4", lw=1, ls=":", label="NSt = 0")
ax.set_xlabel("Net Sentimen Harian")
ax.set_ylabel("Jumlah Hari Perdagangan")
ax.legend(fontsize=9, frameon=False)
fig.tight_layout()
fig.savefig(OUT_G3)
plt.close(fig)

# ── Gambar 4.4  Diagram pencar NSt vs sigma ──────────────────
WARNA_SP = {
    "Pra-peristiwa": "#2196F3",
    "Peristiwa/Demonstrasi": "#F44336",
    "Pasca-peristiwa": "#4CAF50",
}

fig, ax = plt.subplots(figsize=(6.5, 5))
for sp in URUT:
    sub = pair[pair["subperiode"] == sp]
    ax.scatter(
        sub["NSt"],
        sub["sigma"],
        label=sp,
        alpha=0.7,
        s=40,
        color=WARNA_SP[sp],
        edgecolors="white",
        linewidth=0.5,
    )
z = np.polyfit(pair["NSt"], pair["sigma"], 1)
x_line = np.linspace(pair["NSt"].min(), pair["NSt"].max(), 100)
ax.plot(
    x_line,
    np.polyval(z, x_line),
    "k--",
    lw=1,
    label=f"Garis tren linear (r = {float(r_pearson):.3f})",
)
ax.set_xlabel("Net Sentimen Harian (NSt)")
ax.set_ylabel("Volatilitas IHSG (σ)")
ax.legend(fontsize=8, frameon=False)
fig.tight_layout()
fig.savefig(OUT_G4)
plt.close(fig)

# ── Keluaran konsol ──────────────────────────────────────────
print("\n" + "=" * 60)
print("  RINGKASAN STATISTIK DESKRIPTIF, BAB IV")
print("=" * 60)

print("\nCek konsistensi pemetaan hari bursa")
print(
    f"  Komponen jam pada tanggal dokumen          : {'Ada' if ada_jam else 'Tidak ada'}"
)
print(f"  Dokumen tanpa hari bursa berikutnya        : {n_tanpa_bursa}")
print(f"  Dokumen pada hari bursa di luar panel      : {n_dok_luar_panel}")
print(f"  Hari panel dengan jumlah dokumen != n_total: {n_hari_beda} dari {len(rekon)}")

for nama, tbl in [
    ("Tabel 4.1  Tahapan penyaringan data", t1),
    ("Tabel 4.2  Distribusi sub-periode", t2),
    ("Tabel 4.3  Komposisi sumber label", t3),
    ("Tabel 4.5  Rata-rata per sub-periode", t5),
    ("Tabel 4.6  Distribusi sentimen", t6),
    ("Tabel 4.7  Korelasi NSt vs volatilitas", t7),
    ("Tabel 4.8  Uji normalitas (Jarque-Bera)", t8),
]:
    print(f"\n{nama}\n{tbl.to_string(index=False)}")

print(f"\nTabel 4.4  Deskriptif variabel\n{desc.to_string()}")

print(
    f"\nVolume unggahan harian tertinggi: {int(vol.max())} pada {vol.idxmax().date()}"
)
print(f"\n  Tabel  4.1-4.8 tersimpan di: {DIR_TBL}")
print(f"  Gambar 4.1-4.4 tersimpan di: {DIR_GBR}")
print("\n  Langkah 4 selesai. Lanjut -> 5_model_h1.py")

INFO  Kolom label  : label_final
INFO  Kolom sumber : sumber

  RINGKASAN STATISTIK DESKRIPTIF, BAB IV

Cek konsistensi pemetaan hari bursa
  Komponen jam pada tanggal dokumen          : Tidak ada
  Dokumen tanpa hari bursa berikutnya        : 0
  Dokumen pada hari bursa di luar panel      : 85
  Hari panel dengan jumlah dokumen != n_total: 0 dari 104

Tabel 4.1  Tahapan penyaringan data
                        Tahapan Penyaringan Data  Jumlah
                             Dokumen korpus awal   14050
Dokumen tanpa mayoritas tiga model (dikeluarkan)       0
                      Dokumen berlabel mayoritas   14050
                       Hari kalender dengan teks     153
               Hari perdagangan pada panel akhir     104
                       Dokumen masuk panel akhir   13965

Tabel 4.2  Distribusi sub-periode
          Sub-Periode  Jumlah Unggahan  Hari Perdagangan  Persentase (%)  Rata-rata per Hari
        Pra-peristiwa             7329                55           52.16          

5. Model H1

In [ ]:
"""5 | Random Forest M0 vs M1, pengujian H1.

Input      : Hasil Akhir/panel_final.csv
Output CSV : hasil_visualisasi/Tabel Model/tabel_4_8_hasil_h1.csv, tabel_4_9_feature_importance.csv
Output JSON: hasil_visualisasi/Tabel Model/hyperparams.json
"""
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV, TimeSeriesSplit

# ── Jalur ────────────────────────────────────────────────────
BASE = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil")
DIR = BASE / "Hasil Akhir"
DIR_VIS = BASE / "hasil_visualisasi"
OUT_DIR = DIR_VIS / "Tabel Model"

OUT_DIR.mkdir(parents=True, exist_ok=True)

IN_PANEL = DIR / "panel_final.csv"

OUT_HASIL_H1 = OUT_DIR / "tabel_4_8_hasil_h1.csv"
OUT_FEAT_IMP = OUT_DIR / "tabel_4_9_feature_importance.csv"
OUT_HYPERPARAM = OUT_DIR / "hyperparams.json"


def _cek(p):
    if not p.exists():
        raise SystemExit(f"GAGAL: {p.name} belum ada. Jalankan langkah sebelumnya.")
    if p.suffix == ".csv":
        with open(p, encoding="utf-8-sig") as fh:
            if sum(1 for _ in fh) < 2:
                raise SystemExit(
                    f"GAGAL: {p.name} kosong, hanya berisi header. Ulangi langkah yang menghasilkannya."
                )


# Validasi ketersediaan input
_cek(IN_PANEL)

# Muat data panel
panel = pd.read_csv(IN_PANEL, encoding="utf-8-sig", parse_dates=["tanggal"])

LAG_VOL, LAG_RET = 5, 2
TEST_SIZE, SEED, N_SPLITS = 0.2, 42, 5
SEEDS = (42, 7, 2024, 99, 1234)

N_TREES = 300
GRID = {
    "max_depth": [3, 5, None],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", 0.5, 1.0],
}

fitur_m0 = [f"sigma_lag{i}" for i in range(1, LAG_VOL + 1)] + [
    f"R_lag{i}" for i in range(1, LAG_RET + 1)
]
fitur_m1 = ["NSt"] + fitur_m0

# Pastikan terurut berdasarkan waktu & bebas NaN agar tidak ada kebocoran data (Data Leakage)
panel = (
    panel.sort_values("tanggal")
    .dropna(subset=fitur_m1 + ["sigma"])
    .reset_index(drop=True)
)

n_test = max(int(round(len(panel) * TEST_SIZE)), 1)
train, test = panel.iloc[:-n_test], panel.iloc[-n_test:]


def tune(fitur):
    gs = GridSearchCV(
        RandomForestRegressor(random_state=SEED, n_estimators=N_TREES, n_jobs=1),
        GRID,
        cv=TimeSeriesSplit(n_splits=N_SPLITS),
        scoring="neg_root_mean_squared_error",
        n_jobs=-1,
    )
    gs.fit(train[fitur], train["sigma"])
    return {**gs.best_params_, "n_estimators": N_TREES}, float(-gs.best_score_)


def nilai(fitur, hp, nama):
    rm, ma, r2 = [], [], []
    for s in SEEDS:
        m = RandomForestRegressor(random_state=s, n_jobs=-1, **hp).fit(
            train[fitur], train["sigma"]
        )
        p = m.predict(test[fitur])
        rm.append(np.sqrt(mean_squared_error(test["sigma"], p)))
        ma.append(mean_absolute_error(test["sigma"], p))
        r2.append(r2_score(test["sigma"], p))
    return {
        "Spesifikasi Model": nama,
        "RMSE": np.mean(rm),
        "RMSE_sd": np.std(rm, ddof=1),
        "MAE": np.mean(ma),
        "R2": np.mean(r2),
        "_rm": np.array(rm),
    }, m


# Jalankan Tuning Hyperparameter
hp1, cv1 = tune(fitur_m1)
hp0, cv0 = tune(fitur_m0)

# Evaluasi Model
b0, _ = nilai(fitur_m0, hp0, "M0 (Baseline Terbaik)")
c0, _ = nilai(fitur_m0, hp1, "M0 hyperparameter M1 (Pembanding Terkendali)")
c1, _ = nilai(fitur_m1, hp1, "M1 (Baseline + NSt)")

hasil = [b0, c0, c1]
res = pd.DataFrame(hasil)

# Menggunakan b0 sebagai dasar agar kita membandingkan Model Terbaik (M1) vs Model Terbaik (M0)
res["delta_RMSE_vs_M0_Terbaik"] = res["RMSE"] - b0["RMSE"]
res["menang_per_seed_vs_M0_Terbaik"] = [
    f"{int((h['_rm'] < b0['_rm']).sum())}/{len(SEEDS)}" if h is not b0 else "-"
    for h in hasil
]
res = res.drop(columns="_rm")

# Simpan tabel hasil uji H1 dan Hyperparameters
res.round(6).to_csv(OUT_HASIL_H1, index=False, encoding="utf-8-sig")
with open(OUT_HYPERPARAM, "w", encoding="utf-8") as f:
    json.dump({"M0": hp0, "M1": hp1}, f, indent=2)

# Hitung Feature Importance (diurutkan berdasar Permutation di Test Set)
mdi_s, perm_s = [], []
for s in SEEDS:
    m = RandomForestRegressor(random_state=s, n_jobs=-1, **hp1).fit(
        train[fitur_m1], train["sigma"]
    )
    mdi_s.append(m.feature_importances_)
    perm_s.append(
        permutation_importance(
            m, test[fitur_m1], test["sigma"], n_repeats=30, random_state=s, n_jobs=-1
        ).importances_mean
    )

fi = pd.DataFrame(
    {
        "Fitur": fitur_m1,
        "MDI": np.mean(mdi_s, axis=0),
        "MDI_sd": np.std(mdi_s, axis=0, ddof=1),
        "Permutation": np.mean(perm_s, axis=0),
        "Permutation_sd": np.std(perm_s, axis=0, ddof=1),
    }
).sort_values("Permutation", ascending=False)

fi["Peringkat_Permutation"] = range(1, len(fi) + 1)
fi.round(6).to_csv(OUT_FEAT_IMP, index=False, encoding="utf-8-sig")

# Putusan Evaluasi H1
utama = c1["RMSE"] < b0["RMSE"]
dukung = [c1["MAE"] < b0["MAE"], c1["R2"] > b0["R2"]]
status = (
    "DITERIMA" if utama and all(dukung) else ("DUKUNGAN LEMAH" if utama else "DITOLAK")
)
menang = int((c1["_rm"] < b0["_rm"]).sum())

# Cetak Laporan di Terminal
print(f"n_train: {len(train)} | n_test: {len(test)} | seed: {list(SEEDS)}")
print(f"cv_rmse tuning  M0: {cv0:.6f} | M1: {cv1:.6f}")
print(f"Params M0: {hp0}")
print(f"Params M1: {hp1}")
print(
    f"\nTABEL 4.8 Hasil Perbandingan Model (Rata-rata {len(SEEDS)} seed)\n{res.round(6).to_string(index=False)}"
)
print(
    f"\nTABEL 4.9 Feature Importance (Diurutkan berdasar Permutation Test)\n{fi.round(6).to_string(index=False)}"
)

_perm_nst = float(fi.loc[fi["Fitur"] == "NSt", "Permutation"].iloc[0])
print(
    f"\nPeringkat NSt pada Permutation Test: {int(fi.loc[fi['Fitur'] == 'NSt', 'Peringkat_Permutation'].iloc[0])} dari {len(fitur_m1)}"
)

if _perm_nst <= 0:
    print("PERHATIAN: Permutation NSt bernilai nol atau negatif.")
    print(
        "Ini berarti fitur sentimen tidak menambah akurasi pada data uji (test set), bahkan bisa menambah noise."
    )

print("\n" + "=" * 70)
print("PUTUSAN H1 (Dasar: M1 Terbaik vs M0 Terbaik)")
print("=" * 70)
print(
    f"  Metrik utama RMSE : {b0['RMSE']:.6f} -> {c1['RMSE']:.6f}   {'TURUN (MEMBAIK)' if utama else 'NAIK (MEMBURUK)'}"
)
print(
    f"  Pendukung MAE     : {b0['MAE']:.6f} -> {c1['MAE']:.6f}   {'TURUN (MEMBAIK)' if dukung[0] else 'NAIK (MEMBURUK)'}"
)
print(
    f"  Pendukung R2      : {b0['R2']:.6f} -> {c1['R2']:.6f}   {'NAIK (MEMBAIK)' if dukung[1] else 'TURUN (MEMBURUK)'}"
)
print(f"  Pendukung membaik : {sum(dukung)} dari 2")
print(f"  Menang per seed   : {menang}/{len(SEEDS)} seed acak")
print(f"  STATUS H1         : {status}")
print("======================================================================")
print(f"\n  Tabel hasil H1 dan feature importance tersimpan di: {OUT_DIR}")
print("Lanjut -> 09_model_h2.py")

n_train: 83 | n_test: 21 | seed: [42, 7, 2024, 99, 1234]
cv_rmse tuning  M0: 0.002686 | M1: 0.002693
Params M0: {'max_depth': None, 'max_features': 1.0, 'min_samples_leaf': 1, 'n_estimators': 300}
Params M1: {'max_depth': None, 'max_features': 1.0, 'min_samples_leaf': 4, 'n_estimators': 300}

TABEL 4.8 Hasil Perbandingan Model (Rata-rata 5 seed)
                           Spesifikasi Model     RMSE  RMSE_sd      MAE       R2  delta_RMSE_vs_M0_Terbaik menang_per_seed_vs_M0_Terbaik
                       M0 (Baseline Terbaik) 0.005152 0.000038 0.003839 0.215528                  0.000000                             -
M0 hyperparameter M1 (Pembanding Terkendali) 0.004583 0.000039 0.003204 0.379311                 -0.000569                           5/5
                         M1 (Baseline + NSt) 0.004566 0.000042 0.003259 0.383927                 -0.000586                           5/5

TABEL 4.9 Feature Importance (Diurutkan berdasar Permutation Test)
     Fitur      MDI   MDI_sd  Permut

H2 versi 1

In [2]:
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score
from sklearn.model_selection import TimeSeriesSplit

# ── Jalur ────────────────────────────────────────────────────
BASE = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil")
DIR = BASE / "Hasil Akhir"
DIR_VIS = BASE / "hasil_visualisasi"
OUT_DIR = DIR_VIS / "Tabel Model"

OUT_DIR.mkdir(parents=True, exist_ok=True)

IN_PANEL = DIR / "panel_final.csv"
IN_HYPERPARAMS = OUT_DIR / "hyperparams.json"

OUT_TABEL_H2 = OUT_DIR / "pembanding_h2_dekomposisi_N104.csv"


def _cek(path_file):
    if not path_file.exists():
        raise SystemExit(
            f"GAGAL: {path_file.name} belum ada. Jalankan langkah sebelumnya."
        )


_cek(IN_PANEL)
_cek(IN_HYPERPARAMS)

LAG_VOL, LAG_RET = 5, 2
fitur_lag = [f"sigma_lag{i}" for i in range(1, LAG_VOL + 1)] + [
    f"R_lag{i}" for i in range(1, LAG_RET + 1)
]

# 1. Muat Data & Hyperparameters
panel = pd.read_csv(IN_PANEL, encoding="utf-8-sig", parse_dates=["tanggal"])
# Sampel sama dengan langkah 5: dropna hanya pada kolom yang dipakai model
panel = (
    panel.sort_values("tanggal")
    .dropna(subset=["NSt", "sigma"] + fitur_lag)
    .reset_index(drop=True)
)

with open(IN_HYPERPARAMS, encoding="utf-8-sig") as fh:
    # Ekstrak param terbaik dari M1 (Langkah 5)
    params = {k: v for k, v in json.load(fh)["M1"].items() if k != "n_jobs"}

# 2. OPTIMALISASI: Feature Splitting untuk Efek Asimetris (Pembanding Split-Sample)
panel["NSt_Neg"] = panel["NSt"].apply(lambda x: x if x < 0 else 0)
panel["NSt_Pos"] = panel["NSt"].apply(lambda x: x if x >= 0 else 0)

fitur_m2 = ["NSt_Neg", "NSt_Pos"] + fitur_lag

SEEDS = (42, 7, 2024, 99, 1234)
N_PERM = 30  # Iterasi permutasi untuk stabilitas skor
N_SPLITS = 5


def rf(seed):
    return RandomForestRegressor(random_state=seed, n_jobs=-1, **params)


def perm_importance(m, X, y, col_name, rng):
    """Menghitung penurunan R2 jika kolom tertentu diacak (Permutation Importance)."""
    dasar = r2_score(y, m.predict(X))
    turun = []
    for _ in range(N_PERM):
        Xp = X.copy()
        Xp[col_name] = rng.permutation(Xp[col_name].values)
        turun.append(dasar - r2_score(y, m.predict(Xp)))
    return float(np.mean(turun))


# 3. Evaluasi Blocked TimeSeries Cross-Validation
tscv = TimeSeriesSplit(n_splits=N_SPLITS)
hasil_eval = []

for seed in SEEDS:
    rng = np.random.RandomState(seed)
    perm_neg_folds, perm_pos_folds = [], []
    mdi_neg_folds, mdi_pos_folds = [], []

    for train_idx, test_idx in tscv.split(panel):
        train, test = panel.iloc[train_idx], panel.iloc[test_idx]

        m = rf(seed).fit(train[fitur_m2], train["sigma"])

        # MDI (In-Sample / Impurity based)
        idx_neg, idx_pos = fitur_m2.index("NSt_Neg"), fitur_m2.index("NSt_Pos")
        mdi_neg_folds.append(m.feature_importances_[idx_neg])
        mdi_pos_folds.append(m.feature_importances_[idx_pos])

        # Permutation Importance (Out-of-Fold / Objective metric)
        perm_neg_folds.append(
            perm_importance(m, test[fitur_m2], test["sigma"], "NSt_Neg", rng)
        )
        perm_pos_folds.append(
            perm_importance(m, test[fitur_m2], test["sigma"], "NSt_Pos", rng)
        )

    hasil_eval.append(
        {
            "Seed": seed,
            "MDI_NSt_Neg": np.mean(mdi_neg_folds),
            "MDI_NSt_Pos": np.mean(mdi_pos_folds),
            "Perm_NSt_Neg": np.mean(perm_neg_folds),
            "Perm_NSt_Pos": np.mean(perm_pos_folds),
        }
    )

# Simpan Tabel Hasil H2
res = pd.DataFrame(hasil_eval)
res.round(6).to_csv(OUT_TABEL_H2, index=False, encoding="utf-8-sig")

# 4. Ringkasan & Putusan
rata_perm_neg = res["Perm_NSt_Neg"].mean()
rata_perm_pos = res["Perm_NSt_Pos"].mean()
rata_mdi_neg = res["MDI_NSt_Neg"].mean()
rata_mdi_pos = res["MDI_NSt_Pos"].mean()

# Hitung dari 5 seed, berapa kali NSt_Neg mengalahkan NSt_Pos
konsisten_perm = sum(res["Perm_NSt_Neg"] > res["Perm_NSt_Pos"])
konsisten_mdi = sum(res["MDI_NSt_Neg"] > res["MDI_NSt_Pos"])

print("\n" + "=" * 70)
print("PEMBANDING H2: EFEK ASIMETRIS SENTIMEN (DEKOMPOSISI FITUR)")
print("=" * 70)
print(
    f"Data diproses utuh ({len(panel)} baris) dengan TimeSeriesSplit ({N_SPLITS} fold)."
)
print(f"Metrik objektif: Permutation Importance (Penurunan Daya Jelas / R2)\n")

print(f"Rata-rata Permutasi NSt_Negatif : {rata_perm_neg:.6f}")
print(f"Rata-rata Permutasi NSt_Positif : {rata_perm_pos:.6f}")
print(
    f"Konsistensi Permutasi (Neg > Pos): Menang di {konsisten_perm} dari {len(SEEDS)} Seed.\n"
)

print(f"[Opsional] Rata-rata MDI NSt_Negatif : {rata_mdi_neg:.6f}")
print(f"[Opsional] Rata-rata MDI NSt_Positif : {rata_mdi_pos:.6f}")
print("-" * 70)

# Arah pembanding: bukan dasar putusan H2
if (rata_perm_neg > rata_perm_pos) and (konsisten_perm >= 3):
    print("  ARAH PEMBANDING: NEGATIF > POSITIF, KONSISTEN LINTAS SEED")
elif rata_perm_neg > rata_perm_pos:
    print("  ARAH PEMBANDING: NEGATIF > POSITIF, TIDAK KONSISTEN LINTAS SEED")
else:
    print("  ARAH PEMBANDING: NEGATIF TIDAK LEBIH BESAR DARI POSITIF")
print("=" * 70)
print("  Status H2 mengikuti sel split-sample (H2 v2).")
print(f"\n  Tabel pembanding H2 tersimpan di: {OUT_TABEL_H2}")
print("\nLanjut -> 7_robustness.py")


PEMBANDING H2: EFEK ASIMETRIS SENTIMEN (DEKOMPOSISI FITUR)
Data diproses utuh (104 baris) dengan TimeSeriesSplit (5 fold).
Metrik objektif: Permutation Importance (Penurunan Daya Jelas / R2)

Rata-rata Permutasi NSt_Negatif : 0.007981
Rata-rata Permutasi NSt_Positif : -0.000302
Konsistensi Permutasi (Neg > Pos): Menang di 5 dari 5 Seed.

[Opsional] Rata-rata MDI NSt_Negatif : 0.026811
[Opsional] Rata-rata MDI NSt_Positif : 0.008961
----------------------------------------------------------------------
  ARAH PEMBANDING: NEGATIF > POSITIF, KONSISTEN LINTAS SEED
  Status H2 mengikuti sel split-sample (H2 v2).

  Tabel pembanding H2 tersimpan di: G:\My Drive\skripsi\proje\proje\data\hasil\hasil_visualisasi\Tabel Model\pembanding_h2_dekomposisi_N104.csv

Lanjut -> 7_robustness.py


H2 v2

In [ ]:
"""6 | Pengujian H2: split-sample analysis Subset A dan Subset B (Bab III 3.6.3 dan 3.6.6).

Input      : Hasil Akhir/panel_final.csv
             hasil_visualisasi/Tabel Model/hyperparams.json (keluaran langkah 5)
Output CSV : hasil_visualisasi/Tabel Model/H2 Split-Sample/tabel_4_10_h2_splitsample_N*.csv
             hasil_visualisasi/Tabel Model/H2 Split-Sample/tabel_4_10_h2_splitsample_per_seed_N*.csv
"""
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ─── path ───────────────────────────────────────────────────────────
BASE = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil")
DIR = BASE / "Hasil Akhir"
DIR_VIS = BASE / "hasil_visualisasi"
DIR_MODEL = DIR_VIS / "Tabel Model"

IN_PANEL = DIR / "panel_final.csv"
IN_HYPERPARAMS = DIR_MODEL / "hyperparams.json"
OUT_DIR = DIR_MODEL / "H2 Split-Sample"
BERKAS_LAMA = DIR_MODEL / "tabel_4_10_hasil_h2.csv"

# ─── parameter ──────────────────────────────────────────────────────
LAG_VOL, LAG_RET = 5, 2
TEST_SIZE = 0.20
SEEDS = (42, 7, 2024, 99, 1234)
HP_SUMBER = "M1"
AMBANG_MIN_TEST = 10

OUT_DIR.mkdir(parents=True, exist_ok=True)


def _cek(p):
    if not p.exists():
        raise SystemExit(f"GAGAL: {p.name} belum ada. Jalankan langkah sebelumnya.")


_cek(IN_PANEL)
_cek(IN_HYPERPARAMS)

fitur = (
    ["NSt"]
    + [f"sigma_lag{i}" for i in range(1, LAG_VOL + 1)]
    + [f"R_lag{i}" for i in range(1, LAG_RET + 1)]
)

panel = pd.read_csv(IN_PANEL, encoding="utf-8-sig", parse_dates=["tanggal"])
panel = (
    panel.sort_values("tanggal").dropna(subset=fitur + ["sigma"]).reset_index(drop=True)
)

with open(IN_HYPERPARAMS, encoding="utf-8-sig") as fh:
    params = {k: v for k, v in json.load(fh)[HP_SUMBER].items() if k != "n_jobs"}

TANDA = f"N{len(panel)}"
OUT_TABEL = OUT_DIR / f"tabel_4_10_h2_splitsample_{TANDA}.csv"
OUT_PER_SEED = OUT_DIR / f"tabel_4_10_h2_splitsample_per_seed_{TANDA}.csv"

SUBSET = [
    ("Subset A (Sentimen Negatif, NSt < 0)", panel["NSt"] < 0),
    ("Subset B (Sentimen Positif, NSt >= 0)", panel["NSt"] >= 0),
]

idx_nst = fitur.index("NSt")
baris, per_seed = [], []

for nama, mask in SUBSET:
    sub = panel.loc[mask].sort_values("tanggal").reset_index(drop=True)
    n_test = max(int(round(len(sub) * TEST_SIZE)), 1)
    if len(sub) - n_test < 2:
        raise SystemExit(
            f"GAGAL: {nama} hanya {len(sub)} observasi, training set tidak terbentuk."
        )
    train, test = sub.iloc[:-n_test], sub.iloc[-n_test:]

    mdi, rmse, mae, r2 = [], [], [], []
    for s in SEEDS:
        m = RandomForestRegressor(random_state=s, n_jobs=-1, **params)
        m.fit(train[fitur], train["sigma"])
        p = m.predict(test[fitur])
        mdi.append(float(m.feature_importances_[idx_nst]))
        rmse.append(float(np.sqrt(mean_squared_error(test["sigma"], p))))
        mae.append(float(mean_absolute_error(test["sigma"], p)))
        r2.append(float(r2_score(test["sigma"], p)) if len(test) > 1 else np.nan)
        per_seed.append(
            {
                "Subset": nama,
                "Seed": s,
                "MDI_NSt": mdi[-1],
                "RMSE": rmse[-1],
                "MAE": mae[-1],
                "R2": r2[-1],
            }
        )

    sd_test = float(test["sigma"].std(ddof=1)) if len(test) > 1 else np.nan
    baris.append(
        {
            "Subset Kondisi": nama,
            "Kepentingan Net Sentimen": float(np.mean(mdi)),
            "RMSE": float(np.mean(rmse)),
            "MDI_sd": float(np.std(mdi, ddof=1)),
            "RMSE_sd": float(np.std(rmse, ddof=1)),
            "MAE": float(np.mean(mae)),
            "R2": float(np.nanmean(r2)),
            "N": len(sub),
            "n_train": len(train),
            "n_test": len(test),
            "rasio_latih_per_fitur": round(len(train) / len(fitur), 2),
            "sd_sigma_test": sd_test,
            "RMSE_dibagi_sd_test": float(np.mean(rmse)) / sd_test
            if sd_test and sd_test > 0
            else np.nan,
            "_mdi": np.array(mdi),
            "_rmse": np.array(rmse),
        }
    )

A, B = baris
res = pd.DataFrame(baris).drop(columns=["_mdi", "_rmse"])
res.round(6).to_csv(OUT_TABEL, index=False, encoding="utf-8-sig")
pd.DataFrame(per_seed).round(6).to_csv(OUT_PER_SEED, index=False, encoding="utf-8-sig")

mdi_arah = A["Kepentingan Net Sentimen"] > B["Kepentingan Net Sentimen"]
mdi_menang = int((A["_mdi"] > B["_mdi"]).sum())
rmse_arah = A["RMSE"] < B["RMSE"]
rmse_menang = int((A["_rmse"] < B["_rmse"]).sum())
lapis1 = mdi_arah and mdi_menang == len(SEEDS)
lapis2 = rmse_arah

if AMBANG_MIN_TEST is None:
    tak_layak = []
else:
    tak_layak = [b["Subset Kondisi"] for b in baris if b["n_test"] < AMBANG_MIN_TEST]

if tak_layak:
    status = "TIDAK KONKLUSIF"
elif lapis1 and lapis2:
    status = "DITERIMA"
else:
    status = "DITOLAK"

nol_di_B = int((panel["NSt"] == 0).sum())
tanpa_dok = (
    int((panel.loc[panel["NSt"] >= 0, "n_total"] == 0).sum())
    if "n_total" in panel.columns
    else -1
)

print("=" * 78)
print("PUTUSAN H2: EFEK ASIMETRIS SENTIMEN (SPLIT-SAMPLE, BAB III 3.6.3 DAN 3.6.6)")
print("=" * 78)
print(
    f"Panel     : {len(panel)} observasi, {panel['tanggal'].min().date()} s.d. {panel['tanggal'].max().date()}"
)
print(f"Fitur     : {len(fitur)} kolom, {', '.join(fitur)}")
print(f"Hyperpar. : diwarisi dari {HP_SUMBER} hasil tuning dataset utuh, {params}")
print(
    f"Pembagian : {int(TEST_SIZE * 100)} persen test set kronologis di dalam tiap subset, tanpa partisi validasi"
)
print(f"Seed      : {list(SEEDS)}")
print(
    f"Baris NSt = 0 pada panel : {nol_di_B}"
    + (
        f", di antaranya {tanpa_dok} baris tanpa dokumen (n_total = 0)"
        if tanpa_dok >= 0
        else ""
    )
)

print("\nTABEL 4.10 Hasil Pengujian Asimetri Kontribusi Sentimen")
print(
    res[["Subset Kondisi", "Kepentingan Net Sentimen", "RMSE"]]
    .round(6)
    .to_string(index=False)
)

print("\nLampiran Tabel 4.10 (bukan untuk badan Bab IV)")
print(
    res[
        [
            "Subset Kondisi",
            "N",
            "n_train",
            "n_test",
            "rasio_latih_per_fitur",
            "MDI_sd",
            "RMSE_sd",
            "MAE",
            "R2",
            "sd_sigma_test",
            "RMSE_dibagi_sd_test",
        ]
    ]
    .round(6)
    .to_string(index=False)
)

print("\n" + "-" * 78)
print("Lapis 1  MDI net sentimen, Subset A lebih tinggi dari Subset B")
print(
    f"  A {A['Kepentingan Net Sentimen']:.6f}  vs  B {B['Kepentingan Net Sentimen']:.6f}"
    f"   selisih {A['Kepentingan Net Sentimen'] - B['Kepentingan Net Sentimen']:+.6f}"
)
print(f"  Arah terpenuhi   : {mdi_arah}")
print(
    f"  Konsisten lintas : {mdi_menang}/{len(SEEDS)} random_state (Bab III menuntut {len(SEEDS)}/{len(SEEDS)})"
)
print("Lapis 2  RMSE, Subset A lebih rendah dari Subset B")
print(
    f"  A {A['RMSE']:.6f}  vs  B {B['RMSE']:.6f}   selisih {A['RMSE'] - B['RMSE']:+.6f}"
)
print(f"  Arah terpenuhi   : {rmse_arah}")
print(
    f"  Konsisten lintas : {rmse_menang}/{len(SEEDS)} random_state (pelengkap, bukan syarat Bab III)"
)
print("-" * 78)

if AMBANG_MIN_TEST is None:
    print("KELAYAKAN SUBSET : belum dapat diputuskan, AMBANG_MIN_TEST masih None.")
    print(
        f"                   Ukuran test set aktual: A {A['n_test']}, B {B['n_test']}."
    )
    print(
        "                   Putusan di bawah bersifat sementara sampai ambang Bab III ditetapkan."
    )
elif tak_layak:
    print("KELAYAKAN SUBSET : tidak memenuhi syarat -> " + "; ".join(tak_layak))
    print(
        f"                   Ambang berlaku: test set minimal {AMBANG_MIN_TEST} observasi."
    )
else:
    print(
        f"KELAYAKAN SUBSET : kedua subset memenuhi ambang {AMBANG_MIN_TEST} observasi pada test set."
    )

print(f"STATUS H2        : {status}")
if status == "DITOLAK":
    gagal = []
    if not mdi_arah:
        gagal.append("arah MDI tidak sesuai")
    elif mdi_menang < len(SEEDS):
        gagal.append(f"arah MDI hanya konsisten {mdi_menang}/{len(SEEDS)} random_state")
    if not rmse_arah:
        gagal.append("RMSE Subset A tidak lebih rendah")
    print("                   Sebab: " + "; ".join(gagal) + ".")
print("=" * 78)
print(f"Disimpan di {OUT_DIR}")
print(f"  {OUT_TABEL.name}")
print(f"  {OUT_PER_SEED.name}")
print(
    f"Berkas metode lama {'ada dan tidak disentuh' if BERKAS_LAMA.exists() else 'tidak ditemukan'}: {BERKAS_LAMA}"
)
print("Lanjut -> 6b_banding_h2.py, lalu 7_robustness.py")

PUTUSAN H2: EFEK ASIMETRIS SENTIMEN (SPLIT-SAMPLE, BAB III 3.6.3 DAN 3.6.6)
Panel     : 104 observasi, 2025-06-03 s.d. 2025-10-31
Fitur     : 8 kolom, NSt, sigma_lag1, sigma_lag2, sigma_lag3, sigma_lag4, sigma_lag5, R_lag1, R_lag2
Hyperpar. : diwarisi dari M1 hasil tuning dataset utuh, {'max_depth': None, 'max_features': 1.0, 'min_samples_leaf': 4, 'n_estimators': 300}
Pembagian : 20 persen test set kronologis di dalam tiap subset, tanpa partisi validasi
Seed      : [42, 7, 2024, 99, 1234]
Baris NSt = 0 pada panel : 2, di antaranya 0 baris tanpa dokumen (n_total = 0)

TABEL 4.10 Hasil Pengujian Asimetri Kontribusi Sentimen
                       Subset Kondisi  Kepentingan Net Sentimen     RMSE
 Subset A (Sentimen Negatif, NSt < 0)                  0.028457 0.004222
Subset B (Sentimen Positif, NSt >= 0)                  0.010330 0.005022

Lampiran Tabel 4.10 (bukan untuk badan Bab IV)
                       Subset Kondisi  N  n_train  n_test  rasio_latih_per_fitur   MDI_sd  RMSE_sd    

In [ ]:
"""6b | Pembanding metode H2: dekomposisi fitur (kode lama) dan split-sample Bab III (kode baru).

Input      : hasil_visualisasi/Tabel Model/tabel_4_10_hasil_h2.csv (6_model_h2.py, dekomposisi) dan 
hasil_visualisasi/Tabel Model/H2 Split-Sample/tabel_4_10_h2_splitsample*_N*.csv
(6_model_h2_splitsample.py)
Output CSV : hasil_visualisasi/Tabel Model/H2 Split-Sample/tabel_banding_metode_h2.csv
"""
import numpy as np
import pandas as pd
from pathlib import Path

# ─── path ───────────────────────────────────────────────────────────
BASE = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil")
DIR_VIS = BASE / "hasil_visualisasi"
DIR_MODEL = DIR_VIS / "Tabel Model"

IN_LAMA = DIR_MODEL / "tabel_4_10_hasil_h2.csv"
DIR_BARU = DIR_MODEL / "H2 Split-Sample"
OUT_BANDING = DIR_BARU / "tabel_banding_metode_h2.csv"

# ─── parameter ──────────────────────────────────────────────────────
N_PANEL_LAMA = 10  # isi jumlah baris panel saat metode lama dijalankan


def _pilih(pola):
    f = sorted(DIR_BARU.glob(pola), key=lambda p: p.stat().st_mtime)
    if not f:
        raise SystemExit(
            f"GAGAL: berkas {pola} tidak ada di {DIR_BARU}. "
            f"Jalankan 6_model_h2_splitsample.py lebih dulu."
        )
    return f[-1]


f_baru = _pilih("tabel_4_10_h2_splitsample_N*.csv")
f_seed = _pilih("tabel_4_10_h2_splitsample_per_seed_N*.csv")
baru = pd.read_csv(f_baru, encoding="utf-8-sig")
seedb = pd.read_csv(f_seed, encoding="utf-8-sig")

A = baru[baru["Subset Kondisi"].str.startswith("Subset A")].iloc[0]
B = baru[baru["Subset Kondisi"].str.startswith("Subset B")].iloc[0]

piv = seedb.pivot(index="Seed", columns="Subset", values="MDI_NSt")
kol_a = next(c for c in piv.columns if c.startswith("Subset A"))
kol_b = next(c for c in piv.columns if c.startswith("Subset B"))
menang_baru = int((piv[kol_a] > piv[kol_b]).sum())
n_seed_baru = len(piv)

arah_baru = bool(A["Kepentingan Net Sentimen"] > B["Kepentingan Net Sentimen"])
rmse_baru = bool(A["RMSE"] < B["RMSE"])
status_baru = (
    "DITERIMA"
    if (arah_baru and menang_baru == n_seed_baru and rmse_baru)
    else "DITOLAK"
)
n_baru = int(A["N"] + B["N"])

PERLU = {"Perm_NSt_Neg", "Perm_NSt_Pos", "MDI_NSt_Neg", "MDI_NSt_Pos"}
lama = pd.read_csv(IN_LAMA, encoding="utf-8-sig") if IN_LAMA.exists() else None
ada_lama = lama is not None and PERLU <= set(lama.columns)

f6 = lambda x: f"{x:.6f}"

if ada_lama:
    pn, pp = lama["Perm_NSt_Neg"].mean(), lama["Perm_NSt_Pos"].mean()
    mn, mp = lama["MDI_NSt_Neg"].mean(), lama["MDI_NSt_Pos"].mean()
    menang_lama = int((lama["Perm_NSt_Neg"] > lama["Perm_NSt_Pos"]).sum())
    n_seed_lama = len(lama)
    arah_lama = bool(pn > pp)
    status_lama = (
        "DITERIMA"
        if arah_lama and menang_lama >= 3
        else "TIDAK KONKLUSIF"
        if arah_lama
        else "DITOLAK"
    )
    v = f6
else:
    pn = pp = mn = mp = np.nan
    menang_lama = n_seed_lama = 0
    arah_lama = None
    status_lama = "tidak tersedia"
    v = lambda x: "tidak tersedia"

TD = "tidak tersedia"
rows = [
    (
        "Dasar metode",
        "satu model, kolom NSt dipecah jadi NSt_Neg dan NSt_Pos" if ada_lama else TD,
        "dua model Random Forest terpisah, Subset A dan Subset B",
        "tidak",
    ),
    (
        "Landasan Bab III",
        "tidak ada, kode menyatakan menggantikan split-sample" if ada_lama else TD,
        "3.6.3 dan 3.6.6",
        "tidak",
    ),
    ("Jumlah fitur", "9" if ada_lama else TD, "8", "tidak"),
    (
        "Skema evaluasi",
        "TimeSeriesSplit 5 lipatan pada data utuh" if ada_lama else TD,
        "80 dan 20 persen kronologis di dalam tiap subset",
        "tidak",
    ),
    (
        "Jumlah observasi",
        (str(N_PANEL_LAMA) if N_PANEL_LAMA else "tidak tercatat pada berkas")
        if ada_lama
        else TD,
        f"{n_baru} ({int(A['N'])} pada A, {int(B['N'])} pada B)",
        "wajib sama, periksa manual",
    ),
    (
        "Alat ukur utama",
        "permutation importance, penurunan R2 out-of-fold" if ada_lama else TD,
        "MDI, penurunan impuritas in-sample",
        "tidak",
    ),
    (
        "Ukuran kelompok negatif",
        v(pn),
        f6(A["Kepentingan Net Sentimen"]),
        "tidak, skala berbeda",
    ),
    (
        "Ukuran kelompok positif",
        v(pp),
        f6(B["Kepentingan Net Sentimen"]),
        "tidak, skala berbeda",
    ),
    (
        "MDI kelompok negatif",
        v(mn),
        f6(A["Kepentingan Net Sentimen"]),
        "tidak, basis model berbeda",
    ),
    (
        "MDI kelompok positif",
        v(mp),
        f6(B["Kepentingan Net Sentimen"]),
        "tidak, basis model berbeda",
    ),
    (
        "Arah, negatif lebih tinggi",
        str(arah_lama) if ada_lama else TD,
        str(arah_baru),
        "ya",
    ),
    (
        "Konsistensi lintas seed",
        f"{menang_lama}/{n_seed_lama}" if ada_lama else TD,
        f"{menang_baru}/{n_seed_baru}",
        "ya",
    ),
    (
        "RMSE kelompok negatif",
        "tidak pernah dihitung" if ada_lama else TD,
        f6(A["RMSE"]),
        "tidak",
    ),
    (
        "RMSE kelompok positif",
        "tidak pernah dihitung" if ada_lama else TD,
        f6(B["RMSE"]),
        "tidak",
    ),
    (
        "Arah, RMSE negatif lebih rendah",
        "tidak dapat diuji" if ada_lama else TD,
        str(rmse_baru),
        "tidak",
    ),
    (
        "Kriteria penerimaan",
        "rata-rata permutasi negatif lebih besar dan menang minimal 3 dari 5 seed"
        if ada_lama
        else TD,
        f"MDI A lebih tinggi konsisten {n_seed_baru} dari {n_seed_baru} seed dan RMSE A lebih rendah",
        "ya",
    ),
    ("Status H2", status_lama, status_baru, "ya"),
    (
        "Dapat mengisi Tabel 4.10 draf",
        "tidak, kolom RMSE kosong" if ada_lama else TD,
        "ya",
        "ya",
    ),
]

res = pd.DataFrame(
    rows,
    columns=[
        "Aspek",
        "Dekomposisi fitur (kode lama)",
        "Split-sample (Bab III, kode baru)",
        "Sebanding",
    ],
)
res.to_csv(OUT_BANDING, index=False, encoding="utf-8-sig")

print("=" * 100)
print("PEMBANDING METODE PENGUJIAN H2")
print("=" * 100)
print(
    f"Berkas lama : {IN_LAMA if ada_lama else 'tidak ditemukan atau formatnya bukan keluaran metode dekomposisi'}"
)
print(f"Berkas baru : {f_baru.name}")
print(f"              {f_seed.name}\n")
print(res.to_string(index=False))
print("\n" + "-" * 100)
print("BATAS TAFSIR")
print(
    "1. Baris berlabel tidak sebanding hanya boleh dibaca sebagai keterangan, bukan sebagai selisih."
)
print(
    "   MDI metode lama berasal dari satu model berisi dua kolom sentimen, MDI metode baru berasal"
)
print(
    "   dari dua model berbeda yang masing-masing berjumlah satu. Kedua angka tidak sekala."
)
print("2. Perbandingan hanya sah bila kedua metode dijalankan pada panel yang sama.")
print(
    "   Berkas metode lama tidak mencatat jumlah observasi, isi N_PANEL_LAMA secara manual"
)
print("   atau jalankan ulang metode lama pada panel yang berlaku sekarang.")
print("3. Status pada kolom kode baru belum memperhitungkan ambang kelayakan subset.")
print("   Rujuk keluaran 6_model_h2_splitsample.py untuk status akhir.")
print("4. Yang sah dijadikan bahan pembahasan hanya baris arah, kriteria, dan status.")
print("-" * 100)
print(f"Disimpan: {OUT_BANDING}")

PEMBANDING METODE PENGUJIAN H2
Berkas lama : G:\My Drive\skripsi\proje\proje\data\hasil\hasil_visualisasi\Tabel Model\tabel_4_10_hasil_h2.csv
Berkas baru : tabel_4_10_h2_splitsample_N157.csv
              tabel_4_10_h2_splitsample_per_seed_N157.csv

                          Aspek                                            Dekomposisi fitur (kode lama)                                  Split-sample (Bab III, kode baru)                  Sebanding
                   Dasar metode                   satu model, kolom NSt dipecah jadi NSt_Neg dan NSt_Pos            dua model Random Forest terpisah, Subset A dan Subset B                      tidak
               Landasan Bab III                     tidak ada, kode menyatakan menggantikan split-sample                                                    3.6.3 dan 3.6.6                      tidak
                   Jumlah fitur                                                                        9                                                 

In [ ]:
"""7a | Varian NSt untuk uji ketahanan, dibentuk dari keluaran Langkah 1 (FINAL 3)."""
import sys

import pandas as pd
from pathlib import Path

DIR = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil\Hasil Akhir")
F_DOK = DIR / "01_dataset_skor_sentimen.csv"
F_IHSG = DIR / "ihsg_harian.csv"
F_PANEL = DIR / "panel_final.csv"
F_OUT = DIR / "panel_varian_nst.csv"

SUMBER_VOTING = "voting_3model"
SKOR = {"pos": 1.0, "neu": 0.0, "neg": -1.0}
TOL = 1e-9

for f in (F_DOK, F_IHSG, F_PANEL):
    if not f.exists():
        sys.exit(f"GAGAL: {f} tidak ditemukan.")

dok = pd.read_csv(
    F_DOK, encoding="utf-8-sig", usecols=["tanggal", "label_final", "sumber"]
)
dok["label_final"] = dok["label_final"].astype(str).str.strip().str.lower()
dok["sumber"] = dok["sumber"].astype(str).str.strip().str.lower()
dok["tgl"] = (
    pd.to_datetime(dok["tanggal"], errors="coerce")
    .dt.normalize()
    .astype("datetime64[ns]")
)
if (~dok["label_final"].isin(SKOR)).any() or dok["tgl"].isna().any():
    sys.exit(
        f"GAGAL: {F_DOK.name} memuat label atau tanggal tidak sah. Jalankan ulang Langkah 1."
    )
if not dok["sumber"].eq(SUMBER_VOTING).any():
    sys.exit(
        f"GAGAL: tidak ada dokumen bersumber '{SUMBER_VOTING}'. Nilai sumber: {sorted(dok['sumber'].unique())}"
    )

dok["s_hibrida"] = dok["label_final"].map(SKOR)
dok["s_diskret"] = dok["s_hibrida"].where(dok["sumber"].eq(SUMBER_VOTING))

kal = (
    pd.read_csv(F_IHSG, parse_dates=["tanggal"])[["tanggal"]]
    .assign(tanggal=lambda x: x["tanggal"].dt.normalize().astype("datetime64[ns]"))
    .sort_values("tanggal")
    .rename(columns={"tanggal": "tgl_bursa"})
)
dok = pd.merge_asof(
    dok.sort_values("tgl"),
    kal,
    left_on="tgl",
    right_on="tgl_bursa",
    direction="forward",
)
n_gugur = int(dok["tgl_bursa"].isna().sum())

harian = (
    dok.dropna(subset=["tgl_bursa"])
    .groupby("tgl_bursa")
    .agg(
        n_cek=("s_hibrida", "size"),
        NSt_hibrida=("s_hibrida", "mean"),
        n_diskret=("s_diskret", "count"),
        NSt_diskret=("s_diskret", "mean"),
    )
    .reset_index()
    .rename(columns={"tgl_bursa": "tanggal"})
)

panel = pd.read_csv(F_PANEL, parse_dates=["tanggal"])
panel["tanggal"] = panel["tanggal"].astype("datetime64[ns]")
for k in ("tanggal", "NSt", "n_total"):
    if k not in panel.columns:
        sys.exit(f"GAGAL: kolom '{k}' tidak ada di {F_PANEL.name}.")

VARIAN = ["NSt_hibrida", "NSt_diskret"]
out = panel[["tanggal", "NSt", "n_total"]].merge(
    harian, on="tanggal", how="left", validate="one_to_one"
)
kosong = out["n_cek"].isna()
out[["n_cek", "n_diskret"]] = out[["n_cek", "n_diskret"]].fillna(0).astype(int)
out[VARIAN] = out[VARIAN].fillna(0.0)
tanpa_voting = int(((out["n_cek"] > 0) & (out["n_diskret"] == 0)).sum())

beda_n = int((out["n_cek"] != out["n_total"]).sum())
if beda_n:
    sys.exit(
        f"GAGAL: jumlah dokumen per hari bursa berbeda dengan n_total panel pada {beda_n} hari. "
        "ihsg_harian.csv atau 01_dataset_skor_sentimen.csv bukan yang membentuk panel_final.csv."
    )
selisih_h = float((out["NSt_hibrida"] - out["NSt"]).abs().max())
if selisih_h > TOL:
    sys.exit(
        f"GAGAL: NSt panel tidak sama dengan label_final Langkah 1 (selisih maks {selisih_h:.6f}). "
        "Panel belum dibentuk ulang dari keluaran Langkah 1 terbaru."
    )

ringkas = pd.DataFrame(
    [
        {
            "varian": k,
            "mean": out[k].mean(),
            "sd": out[k].std(ddof=1),
            "r_dengan_NSt": out[k].corr(out["NSt"]),
            "selisih_maks_thd_NSt": (out[k] - out["NSt"]).abs().max(),
        }
        for k in VARIAN
    ]
)

print("=" * 90)
print("7a | VARIAN NSt UNTUK UJI KETAHANAN (sumber: 01_dataset_skor_sentimen.csv)")
print("=" * 90)
print(f"Dokumen : {len(dok)} | gugur tanpa hari bursa berikutnya: {n_gugur}")
print(
    f"\nSumber label x label_final:\n{pd.crosstab(dok['sumber'], dok['label_final'], margins=True, margins_name='Jumlah').to_string()}"
)
print(
    f"\nPanel   : {len(panel)} baris, {panel['tanggal'].min().date()} s.d. {panel['tanggal'].max().date()}"
)
print(f"n_total per hari cocok dengan panel : {len(panel)} dari {len(panel)} hari")
print(f"NSt_hibrida = NSt panel             : ya (selisih maks {selisih_h:.1e})")
print(
    f"Hari bursa panel tanpa dokumen      : {int(kosong.sum())} (varian diisi 0, sama dengan Langkah 3)"
)
print(
    f"Hari bursa tanpa dokumen voting     : {tanpa_voting} (NSt_diskret diisi 0, aturan yang sama)"
)
print(
    f"Dokumen pembentuk NSt_diskret       : {int(dok['s_diskret'].notna().sum())} dari {len(dok)} "
    f"(hanya sumber '{SUMBER_VOTING}')"
)
print(
    f"\nRingkasan varian pada hari bursa panel:\n{ringkas.round(6).to_string(index=False)}"
)
print(
    "\nNSt_prob TIDAK dapat dibentuk: 01_dataset_skor_sentimen.csv dan FINAL 3 tidak memuat probabilitas model."
)

out[["tanggal"] + VARIAN + ["n_diskret"]].to_csv(
    F_OUT, index=False, encoding="utf-8-sig"
)
print(f"\nDisimpan: {F_OUT}")
print("Lanjut -> 7_robustness.py")

7a | VARIAN NSt UNTUK UJI KETAHANAN (sumber: 01_dataset_skor_sentimen.csv)
Dokumen : 14050 | gugur tanpa hari bursa berikutnya: 0

Sumber label x label_final:
label_final     neg   neu   pos  Jumlah
sumber                                 
leksikon       4467   232  4501    9200
voting_3model  1463  2774   613    4850
Jumlah         5930  3006  5114   14050

Panel   : 104 baris, 2025-06-03 s.d. 2025-10-31
n_total per hari cocok dengan panel : 104 dari 104 hari
NSt_hibrida = NSt panel             : ya (selisih maks 1.7e-16)
Hari bursa panel tanpa dokumen      : 0 (varian diisi 0, sama dengan Langkah 3)
Hari bursa tanpa dokumen voting     : 0 (NSt_diskret diisi 0, aturan yang sama)
Dokumen pembentuk NSt_diskret       : 4850 dari 14050 (hanya sumber 'voting_3model')

Ringkasan varian pada hari bursa panel:
     varian      mean       sd  r_dengan_NSt  selisih_maks_thd_NSt
NSt_hibrida -0.029540 0.236597      1.000000              0.000000
NSt_diskret -0.164952 0.167369      0.603977        

7 Robusness test 

In [ ]:
"""
Langkah 7 | Robustness Check
Menguji ketahanan kesimpulan H1 terhadap definisi NSt (hibrida, probabilitas, diskret)
dan lag sentimen, dengan kriteria yang sama persis dengan Langkah 5.

Input      : Hasil Akhir/panel_final.csv
             Hasil Akhir/panel_varian_nst.csv (keluaran 7a)
             hasil_visualisasi/Tabel Model/hyperparams.json (keluaran langkah 5)
Output CSV : hasil_visualisasi/Tabel Model/tabel_4_12_robustness.csv
"""
import json
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

BASE = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil")
DIR = BASE / "Hasil Akhir"
DIR_VIS = BASE / "hasil_visualisasi"
OUT_DIR = DIR_VIS / "Tabel Model"
OUT_DIR.mkdir(parents=True, exist_ok=True)

IN_PANEL = DIR / "panel_final.csv"
IN_VARIAN = DIR / "panel_varian_nst.csv"
IN_HP = OUT_DIR / "hyperparams.json"
IN_H1 = OUT_DIR / "tabel_4_8_hasil_h1.csv"
OUT_ROBUSTNESS = OUT_DIR / "tabel_4_12_robustness.csv"

LAG_VOL, LAG_RET = 5, 2
TEST_SIZE = 0.2
SEEDS = (42, 7, 2024, 99, 1234)
N_PERM = 10
TOL = 1e-9
TOL_SILANG = 5e-7

VERSI = ["NSt_hibrida", "NSt_prob", "NSt_diskret"]
SEBAB = {
    "NSt_prob": "FINAL 3 dan 01_dataset_skor_sentimen.csv tidak memuat probabilitas model"
}


def _cek(p: Path, nama: str):
    if not p.exists():
        raise SystemExit(
            f"GAGAL: {nama} belum ada di direktori {p.parent}. Jalankan langkah sebelumnya."
        )
    if p.suffix == ".csv":
        with open(p, encoding="utf-8-sig") as fh:
            if sum(1 for _ in fh) < 2:
                raise SystemExit(f"GAGAL: {nama} kosong, hanya berisi header.")


_cek(IN_PANEL, "panel_final.csv")
_cek(IN_VARIAN, "panel_varian_nst.csv (keluaran 7a_varian_nst.py)")
_cek(IN_HP, "hyperparams.json")
for p, nama in (
    (IN_HP, "hyperparams.json (Langkah 5)"),
    (IN_VARIAN, "panel_varian_nst.csv (7a)"),
):
    if p.stat().st_mtime < IN_PANEL.stat().st_mtime:
        raise SystemExit(
            f"GAGAL: {nama} lebih tua daripada panel_final.csv. Jalankan ulang langkah tersebut."
        )

panel = pd.read_csv(IN_PANEL, encoding="utf-8-sig", parse_dates=["tanggal"])
varian = pd.read_csv(IN_VARIAN, encoding="utf-8-sig", parse_dates=["tanggal"])
for d in (panel, varian):
    d["tanggal"] = d["tanggal"].astype("datetime64[ns]")
kol_v = [k for k in VERSI if k in varian.columns]
panel = panel.drop(columns=kol_v, errors="ignore").merge(
    varian[["tanggal"] + kol_v],
    on="tanggal",
    how="left",
    validate="one_to_one",
    indicator=True,
)
if (panel["_merge"] != "both").any():
    raise SystemExit(
        "GAGAL: ada tanggal panel yang tidak ada di panel_varian_nst.csv. Jalankan ulang 7a."
    )
panel = panel.drop(columns="_merge")

with open(IN_HP, encoding="utf-8-sig") as f:
    HP = {
        m: {k: v for k, v in h.items() if k != "n_jobs"}
        for m, h in json.load(f).items()
    }

fitur_m0 = [f"sigma_lag{i}" for i in range(1, LAG_VOL + 1)] + [
    f"R_lag{i}" for i in range(1, LAG_RET + 1)
]

aktif = next(
    (k for k in kol_v if np.allclose(panel[k], panel["NSt"], rtol=0, atol=TOL)), None
)
kol_hilang = [k for k in VERSI if k not in panel.columns]

M0_TERBAIK = "M0 hyperparameter sendiri"
BASIS = "M0 hyperparameter M1 (pembanding terkendali)"
SPEK = [
    (None, M0_TERBAIK, "M0"),
    (None, BASIS, "M1"),
    ("NSt_prob", "M1 versi ansambel probabilitas (soft voting)", "M1"),
    (
        "NSt_diskret",
        "M1 versi majority voting (dokumen berlabel voting tiga model saja)",
        "M1",
    ),
    ("NSt_hibrida", "M1 versi hibrida leksikon", "M1"),
    ("NSt_lag1", "M1 sentimen lag 1 hari", "M1"),
]
SPEK = [
    (
        k,
        "NSt" if k is not None and k == aktif else k,
        n + (" [SPESIFIKASI UTAMA]" if k is not None and k == aktif else ""),
        h,
    )
    for k, n, h in SPEK
    if k is None or k in panel.columns
]

kol_h1 = ["NSt", "sigma"] + fitur_m0
n_sampel_h1 = len(panel.dropna(subset=kol_h1))
kol_data = [kd for _, kd, _, _ in SPEK if kd]
panel = (
    panel.sort_values("tanggal").dropna(subset=kol_h1 + kol_data).reset_index(drop=True)
)

n_test = max(int(round(len(panel) * TEST_SIZE)), 1)
train, test = panel.iloc[:-n_test], panel.iloc[-n_test:]


def jalankan(kol_label, kol, nama, hp):
    fitur = ([kol] if kol else []) + fitur_m0
    rmse, mae, r2, pi_scores = [], [], [], []
    for s in SEEDS:
        m = RandomForestRegressor(random_state=s, n_jobs=-1, **hp).fit(
            train[fitur], train["sigma"]
        )
        pred = m.predict(test[fitur])
        rmse.append(np.sqrt(mean_squared_error(test["sigma"], pred)))
        mae.append(mean_absolute_error(test["sigma"], pred))
        r2.append(r2_score(test["sigma"], pred))
        if kol:
            pi = permutation_importance(
                m,
                test[fitur],
                test["sigma"],
                n_repeats=N_PERM,
                random_state=s,
                n_jobs=-1,
            )
            pi_scores.append(pi.importances_mean[fitur.index(kol)])
    return {
        "Spesifikasi": nama,
        "fitur_sentimen": kol_label or "(tidak ada)",
        "RMSE_mean": np.mean(rmse),
        "RMSE_sd": np.std(rmse, ddof=1),
        "MAE_mean": np.mean(mae),
        "R2_mean": np.mean(r2),
        "PI_sentimen_mean": np.mean(pi_scores) if pi_scores else np.nan,
        "PI_sentimen_sd": np.std(pi_scores, ddof=1) if pi_scores else np.nan,
        "_rmse": np.array(rmse),
    }


def status_h1(r, acuan):
    if r["RMSE_mean"] >= acuan["RMSE_mean"]:
        return "DITOLAK"
    if r["MAE_mean"] < acuan["MAE_mean"] and r["R2_mean"] > acuan["R2_mean"]:
        return "DITERIMA"
    return "DUKUNGAN LEMAH"


hasil = [jalankan(*s[:3], HP.get(s[3], {})) for s in SPEK]
res = pd.DataFrame(hasil)
m0 = res.loc[res["Spesifikasi"] == M0_TERBAIK].iloc[0]
bs = res.loc[res["Spesifikasi"] == BASIS].iloc[0]
ada_sent = (res["fitur_sentimen"] != "(tidak ada)").tolist()

res["delta_RMSE_vs_M0_terbaik"] = res["RMSE_mean"] - m0["RMSE_mean"]
res["rasio_delta_thd_sd"] = (
    res["delta_RMSE_vs_M0_terbaik"].abs() / res["RMSE_sd"]
).round(2)
res["status_H1_vs_M0_terbaik"] = [
    status_h1(r, m0) if a else "" for (_, r), a in zip(res.iterrows(), ada_sent)
]
res["menang_per_seed_vs_M0_terbaik"] = [
    f"{int((h['_rmse'] < m0['_rmse']).sum())}/{len(SEEDS)}" if a else ""
    for h, a in zip(hasil, ada_sent)
]
res["delta_RMSE_vs_terkendali"] = res["RMSE_mean"] - bs["RMSE_mean"]
res["status_vs_terkendali"] = [
    status_h1(r, bs) if a else "" for (_, r), a in zip(res.iterrows(), ada_sent)
]
res = res.drop(columns="_rmse")
res.round(6).to_csv(OUT_ROBUSTNESS, index=False, encoding="utf-8-sig")

silang = "tidak dapat diperiksa, tabel_4_8_hasil_h1.csv tidak ditemukan"
if IN_H1.exists() and aktif:
    h1 = pd.read_csv(IN_H1, encoding="utf-8-sig")
    utama = res.loc[res["fitur_sentimen"] == aktif, "Spesifikasi"].iloc[0]
    beda = []
    for pola, nama in (("M0 (Baseline", M0_TERBAIK), ("M1 (Baseline", utama)):
        b = h1[h1["Spesifikasi Model"].str.contains(pola, regex=False, na=False)]
        if len(b):
            sel = abs(
                float(b["RMSE"].iloc[0])
                - float(res.loc[res["Spesifikasi"] == nama, "RMSE_mean"].iloc[0])
            )
            if sel > TOL_SILANG:
                beda.append(f"{pola.split()[0]} selisih {sel:.9f}")
    silang = (
        "cocok pada presisi 6 desimal"
        if not beda
        else "BERBEDA (" + "; ".join(beda) + "), periksa N dan hyperparams"
    )

print(
    f"N: {len(panel)} | n_train: {len(train)} | n_test: {len(test)} | seed: {list(SEEDS)}"
)
if len(panel) != n_sampel_h1:
    print(
        f"PERINGATAN: sampel robustness {len(panel)} baris, sampel H1 {n_sampel_h1} baris."
    )
print(f"Spesifikasi utama (NSt identik dengan): {aktif or 'tidak terdeteksi'}")
print(f"\nTABEL 4.12 ROBUSTNESS\n{res.round(6).to_string(index=False)}")
print(
    f"\nPembanding penentu    : {M0_TERBAIK}, RMSE {m0['RMSE_mean']:.6f} (sama dengan putusan H1 Langkah 5)"
)
print(
    f"Pembanding terkendali : {BASIS}, RMSE {bs['RMSE_mean']:.6f} (diagnostik, tidak memutuskan)"
)
print(
    "Aturan status sama dengan Langkah 5: RMSE turun, MAE turun, dan R2 naik = DITERIMA;"
)
print(
    "RMSE turun tanpa seluruh pendukung = DUKUNGAN LEMAH; RMSE tidak turun = DITOLAK."
)
print(f"Silang periksa terhadap Tabel 4.8 Langkah 5: {silang}")

if kol_hilang:
    print("\nSPESIFIKASI TIDAK DIUJI")
    for k in kol_hilang:
        print(f"  {k}: {SEBAB.get(k, 'kolom tidak dibentuk oleh 7a_varian_nst.py')}")

versi = res[res["fitur_sentimen"].isin(VERSI)]
if len(versi) >= 2:
    konsisten = versi["status_H1_vs_M0_terbaik"].nunique() == 1
    print(
        f"\nStatus H1 sama pada {len(versi)} definisi NSt yang diuji ({', '.join(versi['fitur_sentimen'])}): {konsisten}"
    )
    if not konsisten:
        print(
            "Kesimpulan berubah antardefinisi. Wajib dilaporkan di Bab IV dan diangkat di keterbatasan Bab V."
        )
else:
    print(
        "\nKurang dari dua definisi NSt yang diuji; ketahanan terhadap definisi NSt belum dapat dinilai."
    )

lag = res[res["fitur_sentimen"] == "NSt_lag1"]
if len(lag):
    st = lag["status_H1_vs_M0_terbaik"].iloc[0]
    rs = float(lag["rasio_delta_thd_sd"].iloc[0])
    print(
        f"\nNSt_lag1 terhadap M0 terbaik: {st} | rasio |delta RMSE| thd sd antarseed: {rs}"
    )
    if rs < 1:
        print(
            "-> selisih lebih kecil daripada variasi antarseed Random Forest (heuristik, bukan uji statistik)."
        )
    if st == "DITERIMA":
        print(
            "-> sentimen hari sebelumnya memenuhi kriteria yang sama dengan H1 pada data uji ini."
        )
    elif st == "DUKUNGAN LEMAH":
        print(
            "-> RMSE turun, tetapi metrik pendukung tidak seluruhnya membaik. Jangan disebut kontribusi prediktif penuh."
        )
    else:
        print(
            "-> sentimen hari sebelumnya tidak mengungguli M0 terbaik. Tidak ada dasar untuk klaim kontribusi prediktif."
        )

print(f"\nTabel robustness tersimpan di: {OUT_ROBUSTNESS}")

N: 104 | n_train: 83 | n_test: 21 | seed: [42, 7, 2024, 99, 1234]
Spesifikasi utama (NSt identik dengan): NSt_hibrida

TABEL 4.12 ROBUSTNESS
                                                       Spesifikasi fitur_sentimen  RMSE_mean  RMSE_sd  MAE_mean  R2_mean  PI_sentimen_mean  PI_sentimen_sd  delta_RMSE_vs_M0_terbaik  rasio_delta_thd_sd status_H1_vs_M0_terbaik menang_per_seed_vs_M0_terbaik  delta_RMSE_vs_terkendali status_vs_terkendali
                                         M0 hyperparameter sendiri    (tidak ada)   0.005152 0.000038  0.003839 0.215528               NaN             NaN                  0.000000                0.00                                                                        0.000569                     
                      M0 hyperparameter M1 (pembanding terkendali)    (tidak ada)   0.004583 0.000039  0.003204 0.379311               NaN             NaN                 -0.000569               14.75                                                       

UJI KONTROL

In [ ]:
"""8 | Uji arah maju pelabelan: tarik sampel buta dari FINAL 3 untuk label manual."""
import hashlib
import json
import re
import sys
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from openpyxl import Workbook, load_workbook
from openpyxl.styles import Alignment, Font, Protection
from openpyxl.worksheet.datavalidation import DataValidation

BASE = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil")
FILE_IN = BASE / "hasil_visualisasi" / "FINAL 3.csv"
DIR_UJI = BASE / "Hasil Akhir" / "Uji Label"
LEMBAR = DIR_UJI / "lembar_label_buta.xlsx"
LEMBAR_2 = DIR_UJI / "lembar_label_buta_anotator2.xlsx"
KUNCI = DIR_UJI / "kunci_sampel.csv"
PROTOKOL = DIR_UJI / "protokol_uji_label.json"
LEMBAR_LAMA = Path(r"c:\Users\EZRA\Downloads\tempat\Uji Label\lembar_label_buta.xlsx")
PERNAH_TAMPIL = [Path(r"c:\Users\EZRA\Downloads\tempat\sampel_uji_kontrol.xlsx")]

N_SAMPEL, N_ANOTATOR2, SEED = 300, 300, 42
HANYA_SUMBER = None
JENDELA = ("2025-06-01", "2025-10-31")
BATAS_PRA, BATAS_PUNCAK = "2025-08-24", "2025-09-09"
STATUS_LEKSIKON = {"dominan_pos", "dominan_neg", "imbang", "hanya_netral"}
AMBANG = {
    "kappa_memadai": 0.61,
    "kappa_layak": 0.41,
    "aturan_kappa": "kappa >= 0,61 memadai; 0,41 <= kappa < 0,61 layak dengan catatan keterbatasan; "
    "kappa < 0,41 gagal; diputus dari nilai titik tanpa pembulatan",
    "aturan_accuracy": "accuracy > proporsi kelas terbanyak pada label manual sampel",
    "aturan_f1_confusion": "tanpa ambang baku; tukar negatif-positif dilaporkan dan harus sedikit",
    "rujukan_kategori_kappa": "Landis & Koch (1977), diverifikasi dari artikel asli sebelum dikutip",
}
PEDOMAN = [
    "Tentukan opini atau suasana hati penulis terhadap pasar modal, IHSG, saham, atau kondisi ekonomi.",
    "pos: optimistis, misalnya harapan kenaikan, kabar baik, keuntungan, atau keyakinan terhadap pasar.",
    "neg: pesimistis, misalnya kekhawatiran, kerugian, penurunan, kritik, ancaman, atau kekecewaan.",
    "neu: tidak condong ke optimisme maupun pesimisme, misalnya informasi faktual, pertanyaan, "
    "promosi tanpa sikap, atau muatan positif dan negatif yang seimbang.",
    "Satu dokumen satu label. Pilih pos, neu, atau neg dari daftar pada kolom label_manual.",
    "Jangan membuka FINAL 3.csv, 01_dataset_skor_sentimen.csv, kunci_sampel.csv, atau keluaran skrip 9 "
    "sebelum seluruh baris terisi.",
    "Kolom id_sampel dan teks dikunci. Keraguan boleh dicatat di kolom catatan_anotator.",
]
PETA = {
    "positive": "pos",
    "positif": "pos",
    "neutral": "neu",
    "netral": "neu",
    "negative": "neg",
    "negatif": "neg",
}
ILEGAL = re.compile(r"[\x00-\x08\x0b\x0c\x0e-\x1f]")


def tulis_lembar(path, data):
    wb = Workbook()
    ws = wb.active
    ws.title = "label"
    ws.append(["id_sampel", "teks", "label_manual", "catatan_anotator"])
    for i, t in zip(data["id_sampel"], data["teks"]):
        ws.append([i, t, None, None])
        r = ws.max_row
        ws.cell(r, 2).data_type = "s"
        ws.cell(r, 2).alignment = Alignment(wrap_text=True, vertical="top")
        ws.row_dimensions[r].height = 15 * min(27, max(1, -(-len(t) // 95)))
        for k in (3, 4):
            ws.cell(r, k).protection = Protection(locked=False)
    for c in ws[1]:
        c.font = Font(bold=True)
    for kol, lebar in zip("ABCD", (11, 100, 15, 30)):
        ws.column_dimensions[kol].width = lebar
    ws.freeze_panes = "A2"
    dv = DataValidation(
        type="list",
        formula1='"pos,neu,neg"',
        allow_blank=True,
        showErrorMessage=True,
        error="Isi pos, neu, atau neg.",
    )
    ws.add_data_validation(dv)
    dv.add(f"C2:C{ws.max_row}")
    ws.protection.sheet = True
    ws.protection.formatColumns = False
    ws.protection.formatRows = False
    wp = wb.create_sheet("pedoman")
    for baris in PEDOMAN:
        wp.append([baris])
    wp.column_dimensions["A"].width = 130
    wb.save(path)


if LEMBAR.exists():
    sys.exit(
        f"GAGAL: {LEMBAR} sudah ada. Sampel hanya ditarik sekali; lembar lama tidak ditimpa."
    )
if not FILE_IN.exists():
    sys.exit(f"GAGAL: {FILE_IN} tidak ditemukan.")
if LEMBAR_LAMA.exists():
    ws = load_workbook(LEMBAR_LAMA, read_only=True)["label"]
    terisi = sum(
        1
        for r in ws.iter_rows(min_row=2, min_col=3, max_col=3, values_only=True)
        if r[0] not in (None, "")
    )
    if terisi:
        sys.exit(
            f"BERHENTI: {LEMBAR_LAMA} (sampel FINAL 2) sudah berisi {terisi} label manual. "
            "Jangan tarik sampel baru sebelum diputuskan nasib label tersebut."
        )
    print(
        f"INFO: {LEMBAR_LAMA.name} lama (sampel FINAL 2) masih kosong dan digantikan; pindahkan ke arsip."
    )

df = pd.read_csv(FILE_IN, encoding="utf-8-sig", low_memory=False)
df.columns = [c.strip().lstrip("\ufeff") for c in df.columns]
hilang = [
    k
    for k in [
        "row_key",
        "tanggal_wib",
        "text_clean_bert",
        "label_final",
        "sumber",
        "status",
    ]
    if k not in df.columns
]
if hilang:
    sys.exit(f"GAGAL: kolom {hilang} tidak ada di {FILE_IN.name}.")
if df["row_key"].duplicated().any():
    sys.exit("GAGAL: row_key tidak unik.")

df["label_final"] = df["label_final"].astype(str).str.strip().str.lower().replace(PETA)
df["sumber"] = df["sumber"].astype(str).str.strip().str.lower()
df["status"] = df["status"].astype(str).str.strip().str.lower()
df["tgl"] = pd.to_datetime(df["tanggal_wib"], errors="coerce").dt.normalize()
df["jalur"] = np.select(
    [
        df["sumber"].eq("voting_3model"),
        df["sumber"].eq("leksikon") & df["status"].isin(STATUS_LEKSIKON),
    ],
    ["voting_3model", "leksikon_" + df["status"]],
    "lain",
)
df["periode"] = np.select(
    [df["tgl"] <= pd.Timestamp(BATAS_PRA), df["tgl"] <= pd.Timestamp(BATAS_PUNCAK)],
    ["Pra", "Puncak"],
    "Pasca",
)

sah = (
    df["label_final"].isin(["pos", "neu", "neg"])
    & df["tgl"].between(pd.Timestamp(JENDELA[0]), pd.Timestamp(JENDELA[1]))
    & df["text_clean_bert"].fillna("").astype(str).str.strip().ne("")
)
if HANYA_SUMBER:
    sah &= df["sumber"].eq(HANYA_SUMBER)
if (df.loc[sah, "jalur"] == "lain").any():
    x = df.loc[sah & df["jalur"].eq("lain"), ["sumber", "status"]].drop_duplicates()
    sys.exit(
        f"GAGAL: kombinasi sumber dan status tak dikenal:\n{x.to_string(index=False)}"
    )

tampil = set()
for f in PERNAH_TAMPIL:
    if f.exists():
        lama = pd.read_excel(f, dtype=str)
        if "row_key" in lama:
            tampil |= set(lama["row_key"].dropna())
        print(
            f"PERINGATAN: {f.name} pernah menampilkan label model. Dokumennya dikeluarkan dari kerangka."
        )
n_tampil = int((sah & df["row_key"].isin(tampil)).sum())
kerangka = (
    df[sah & ~df["row_key"].isin(tampil)].sort_values("row_key").reset_index(drop=True)
)

ukuran = kerangka.groupby(["jalur", "label_final"]).size()
kuota = ukuran * N_SAMPEL / ukuran.sum()
alok = np.floor(kuota).astype(int)
alok.loc[
    (kuota - alok)
    .sort_values(ascending=False, kind="stable")
    .index[: N_SAMPEL - alok.sum()]
] += 1

rs = np.random.RandomState(SEED)
grup = dict(tuple(kerangka.groupby(["jalur", "label_final"])))
sampel = pd.concat(
    [grup[s].sample(n=int(k), random_state=rs) for s, k in alok.items() if k > 0]
)
sampel = sampel.sample(frac=1, random_state=rs).reset_index(drop=True)
sampel["id_sampel"] = [f"S{i:03d}" for i in range(1, len(sampel) + 1)]
sampel["teks"] = sampel["text_clean_bert"].astype(str).map(lambda x: ILEGAL.sub(" ", x))

DIR_UJI.mkdir(parents=True, exist_ok=True)
tulis_lembar(LEMBAR, sampel)
tulis_lembar(LEMBAR_2, sampel.head(N_ANOTATOR2))
sampel[["id_sampel", "row_key"]].to_csv(KUNCI, index=False, encoding="utf-8-sig")
protokol = {
    "dibuat": datetime.now().isoformat(timespec="seconds"),
    "berkas_sumber": FILE_IN.name,
    "path_berkas_sumber": str(FILE_IN),
    "sha256_berkas_sumber": hashlib.sha256(FILE_IN.read_bytes()).hexdigest(),
    "n_berkas": len(df),
    "n_kerangka": len(kerangka),
    "n_tidak_sah": int((~sah).sum()),
    "n_dikeluarkan_pernah_tampil": n_tampil,
    "hanya_sumber": HANYA_SUMBER,
    "desain": "acak berlapis, lapisan jalur pelabelan (sumber x status) x label_final, "
    "alokasi proporsional sisa terbesar",
    "alokasi": {f"{j}|{l}": int(k) for (j, l), k in alok.items()},
    "seed": SEED,
    "n_sampel": len(sampel),
    "n_anotator2": int(min(N_ANOTATOR2, len(sampel))),
    "ambang": AMBANG,
    "pedoman": PEDOMAN,
}
PROTOKOL.write_text(
    json.dumps(protokol, ensure_ascii=False, indent=2), encoding="utf-8"
)

print(f"Berkas sumber   : {FILE_IN.name} ({len(df)} baris)")
print(
    f"Kerangka sampel : {len(kerangka)} dokumen | tidak sah {int((~sah).sum())} | pernah tampil {n_tampil}"
    + (f" | hanya sumber {HANYA_SUMBER}" if HANYA_SUMBER else "")
)
print("\nSumber label x sub-periode (basis kalender), kerangka sampel:")
print(
    pd.crosstab(
        kerangka["sumber"], kerangka["periode"], margins=True, margins_name="Total"
    )
    .reindex(columns=["Pra", "Puncak", "Pasca", "Total"])
    .to_string()
)
print("\nJalur pelabelan x label_final, kerangka sampel:")
print(
    pd.crosstab(
        kerangka["jalur"], kerangka["label_final"], margins=True, margins_name="Total"
    ).to_string()
)
print(f"\nAlokasi sampel (N = {len(sampel)}, seed {SEED}):")
print(alok.unstack(fill_value=0).to_string())
print(
    "\nSampel per sub-periode:",
    sampel["periode"].value_counts().reindex(["Pra", "Puncak", "Pasca"]).to_dict(),
)
print(f"\nDisimpan di {DIR_UJI}:")
print(f"  {LEMBAR.name} ({len(sampel)} baris)")
print(
    f"  {LEMBAR_2.name} ({min(N_ANOTATOR2, len(sampel))} baris, opsional untuk pelabel kedua)"
)
print(f"  {KUNCI.name}, {PROTOKOL.name}")
print("\nIsi label_manual tanpa membuka berkas yang memuat label model.")
print("Lanjut -> 9_evaluasi_uji_kontrol.py setelah seluruh baris terisi.")

INFO: lembar_label_buta.xlsx lama (sampel FINAL 2) masih kosong dan digantikan; pindahkan ke arsip.
Berkas sumber   : FINAL 3.csv (14050 baris)
Kerangka sampel : 14050 dokumen | tidak sah 0 | pernah tampil 0

Sumber label x sub-periode (basis kalender), kerangka sampel:
periode         Pra  Puncak  Pasca  Total
sumber                                   
leksikon       4930    1394   2876   9200
voting_3model  2675     604   1571   4850
Total          7605    1998   4447  14050

Jalur pelabelan x label_final, kerangka sampel:
label_final             neg   neu   pos  Total
jalur                                         
leksikon_dominan_neg   3560     0     0   3560
leksikon_dominan_pos      0     0  4501   4501
leksikon_hanya_netral     0   232     0    232
leksikon_imbang         907     0     0    907
voting_3model          1463  2774   613   4850
Total                  5930  3006  5114  14050

Alokasi sampel (N = 300, seed 42):
label_final            neg  neu  pos
jalur                

In [ ]:
"""9 | Uji arah maju pelabelan: label_final FINAL 3 terhadap label manual buta."""
import hashlib
import json
import sys
import warnings
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    cohen_kappa_score,
    confusion_matrix,
    precision_recall_fscore_support,
)

warnings.filterwarnings("ignore", category=RuntimeWarning)

BASE = Path(r"G:\My Drive\skripsi\proje\proje\data\hasil")
FILE_LABEL, KOL_LABEL = BASE / "hasil_visualisasi" / "FINAL 3.csv", "label_final"
DIR_UJI = BASE / "Hasil Akhir" / "Uji Label"
LEMBAR = DIR_UJI / "lembar_label_buta.xlsx"
LEMBAR_2 = DIR_UJI / "lembar_label_buta_anotator2.xlsx"
KONSENSUS = DIR_UJI / "lembar_konsensus.xlsx"
KUNCI = DIR_UJI / "kunci_sampel.csv"
PROTOKOL = DIR_UJI / "protokol_uji_label.json"
OUT = BASE / "hasil_visualisasi" / "Tabel Validasi Label"

N_BOOT, SEED, MIN_ANOTATOR2 = 2000, 42, 30
IZINKAN_LABEL_BERUBAH = False
BATAS_PRA, BATAS_PUNCAK = "2025-08-24", "2025-09-09"
STATUS_LEKSIKON = {"dominan_pos", "dominan_neg", "imbang", "hanya_netral"}
URUT_JALUR = [
    "voting_3model",
    "leksikon_dominan_pos",
    "leksikon_dominan_neg",
    "leksikon_imbang",
    "leksikon_hanya_netral",
]

KELAS = ["neg", "neu", "pos"]
KODE = {k: i for i, k in enumerate(KELAS)}
SKOR = {"neg": -1, "neu": 0, "pos": 1}
PETA = {
    "negatif": "neg",
    "negative": "neg",
    "-1": "neg",
    "-1.0": "neg",
    "netral": "neu",
    "neutral": "neu",
    "0": "neu",
    "0.0": "neu",
    "positif": "pos",
    "positive": "pos",
    "1": "pos",
    "1.0": "pos",
}


def norm_label(s):
    return s.fillna("").astype(str).str.strip().str.lower().replace(PETA)


def norm_teks(s):
    return (
        s.fillna("")
        .astype(str)
        .str.replace(r"[\x00-\x1f]", " ", regex=True)
        .str.split()
        .str.join(" ")
    )


def baca_lembar(path):
    s = pd.read_excel(path, sheet_name="label", dtype=str)
    s["id_sampel"] = s["id_sampel"].fillna("").str.strip()
    s["label_manual"] = norm_label(s["label_manual"])
    return s


def simpan_protokol():
    PROTOKOL.write_text(
        json.dumps(prot, ensure_ascii=False, indent=2), encoding="utf-8"
    )


def bekukan(nama, s):
    kini = dict(zip(s["id_sampel"], s["label_manual"]))
    if nama not in prot:
        prot[nama] = {
            "waktu": datetime.now().isoformat(timespec="seconds"),
            "label": kini,
        }
        simpan_protokol()
        return prot[nama]["waktu"], []
    berubah = sorted(k for k in kini if prot[nama]["label"].get(k) != kini[k])
    if berubah and not IZINKAN_LABEL_BERUBAH:
        sys.exit(
            f"GAGAL: {len(berubah)} label pada {nama} berubah sesudah evaluasi pertama "
            f"({prot[nama]['waktu']}), contoh {berubah[:10]}. Label model sudah terlihat pada evaluasi itu, "
            "jadi perubahan ini tidak lagi buta. Kembalikan label semula."
        )
    return prot[nama]["waktu"], berubah


def cepat(y, p):
    cm = np.bincount(3 * y + p, minlength=9).reshape(3, 3)
    n, tp, kol, bar = cm.sum(), np.diag(cm), cm.sum(0), cm.sum(1)
    po, pe = tp.sum() / n, bar @ kol / n**2
    kap = (
        (po - pe) / (1 - pe) if (kol > 0).sum() > 1 and (bar > 0).sum() > 1 else np.nan
    )
    pr = np.divide(tp, kol, out=np.zeros(3), where=kol > 0)
    rc = np.divide(tp, bar, out=np.zeros(3), where=bar > 0)
    f1 = np.divide(2 * pr * rc, pr + rc, out=np.zeros(3), where=pr + rc > 0)
    return po, kap, f1.mean(), (p - y).mean()


def metrik(d, rng):
    y, p = d["manual"].to_numpy(), d["model"].to_numpy()
    sy, sp = d["manual"].map(SKOR).to_numpy(), d["model"].map(SKOR).to_numpy()
    pr, rc, f1, _ = precision_recall_fscore_support(
        y, p, labels=KELAS, average="macro", zero_division=0
    )
    satu_kelas = len(set(y)) < 2 or len(set(p)) < 2
    vc = d["manual"].value_counts(normalize=True)
    tukar, polar, galat = (
        int((sy * sp == -1).sum()),
        int(((sy != 0) & (sp != 0)).sum()),
        np.abs(sp - sy).sum(),
    )
    m = {
        "N": len(d),
        "accuracy": accuracy_score(y, p),
        "kelas_terbanyak": vc.idxmax(),
        "proporsi_kelas_terbanyak": vc.max(),
        "precision_macro": pr,
        "recall_macro": rc,
        "f1_macro": f1,
        "kappa": np.nan if satu_kelas else cohen_kappa_score(y, p, labels=KELAS),
        "tukar_neg_pos": tukar,
        "tukar_pct_N": 100 * tukar / len(d),
        "tukar_pct_polar": 100 * tukar / polar if polar else np.nan,
        "porsi_galat_tukar_pct": 100 * 2 * tukar / galat if galat else 0.0,
        "selisih_skor": (sp - sy).mean(),
    }
    yk, pk = d["manual"].map(KODE).to_numpy(), d["model"].map(KODE).to_numpy()
    if not np.allclose(
        cepat(yk, pk),
        [m["accuracy"], m["kappa"], m["f1_macro"], m["selisih_skor"]],
        equal_nan=True,
    ):
        sys.exit("GAGAL: silang periksa numpy terhadap scikit-learn tidak cocok.")
    lapis = [
        np.asarray(v) for v in d.groupby(["jalur", "label_strata"]).indices.values()
    ]
    boot = np.array(
        [
            cepat(yk[i], pk[i])
            for i in (
                np.concatenate([rng.choice(s, s.size) for s in lapis])
                for _ in range(N_BOOT)
            )
        ]
    )
    m["ci"] = np.nanpercentile(boot, [2.5, 97.5], axis=0)
    return m


for f in (PROTOKOL, KUNCI, LEMBAR, FILE_LABEL):
    if not f.exists():
        sys.exit(f"GAGAL: {f} tidak ditemukan.")
prot = json.loads(PROTOKOL.read_text(encoding="utf-8"))
A = prot["ambang"]
FILE_SUMBER = Path(prot.get("path_berkas_sumber", DIR_UJI / prot["berkas_sumber"]))
if not FILE_SUMBER.exists():
    sys.exit(f"GAGAL: berkas sumber sampel {FILE_SUMBER} tidak ditemukan.")
kunci = pd.read_csv(KUNCI, dtype=str, encoding="utf-8-sig")
lembar = baca_lembar(LEMBAR)

if (
    sorted(lembar["id_sampel"]) != sorted(kunci["id_sampel"])
    or lembar["id_sampel"].duplicated().any()
):
    sys.exit(
        "GAGAL: id_sampel pada lembar tidak sama dengan kunci (baris hilang, bertambah, atau ganda)."
    )
kosong = int((lembar["label_manual"] == "").sum())
salah = lembar.loc[
    ~lembar["label_manual"].isin(KELAS + [""]), ["id_sampel", "label_manual"]
]
if kosong or len(salah):
    sys.exit(
        f"GAGAL: {kosong} baris belum berlabel dan {len(salah)} label tidak valid "
        f"{salah.values.tolist()[:10]}. Evaluasi baru dijalankan setelah seluruh baris terisi."
    )

l2 = baca_lembar(LEMBAR_2)
gab = lembar.merge(l2, on="id_sampel", suffixes=("_1", "_2"))
if (
    len(gab) != len(lembar)
    or len(l2) != len(lembar)
    or (~l2["label_manual"].isin(KELAS)).any()
    or (norm_teks(gab["teks_1"]) != norm_teks(gab["teks_2"])).any()
):
    sys.exit(
        f"GAGAL: {LEMBAR_2.name} harus berisi baris dan teks yang sama dengan {LEMBAR.name} dan terisi seluruhnya."
    )
beda_a = gab[gab["label_manual_1"] != gab["label_manual_2"]]
if len(beda_a) and not KONSENSUS.exists():
    beda_a[["id_sampel", "teks_1", "label_manual_1", "label_manual_2"]].set_axis(
        ["id_sampel", "teks", "label_anotator1", "label_anotator2"], axis=1
    ).assign(label_konsensus="").to_excel(
        KONSENSUS, sheet_name="konsensus", index=False
    )
    sys.exit(
        f"BERHENTI: {KONSENSUS.name} dibuat berisi {len(beda_a)} baris yang label kedua anotatornya berbeda. "
        "Isi label_konsensus bersama tanpa membuka FINAL 3, lalu jalankan ulang skrip ini."
    )
ks = (
    pd.read_excel(KONSENSUS, sheet_name="konsensus", dtype=str)
    if len(beda_a)
    else pd.DataFrame(
        columns=["id_sampel", "label_anotator1", "label_anotator2", "label_konsensus"]
    )
)
ks["id_sampel"] = ks["id_sampel"].fillna("").str.strip()
for k in ("label_anotator1", "label_anotator2", "label_konsensus"):
    ks[k] = norm_label(ks[k])
cek = beda_a.merge(ks, on="id_sampel")
if (
    len(cek) != len(beda_a)
    or len(ks) != len(beda_a)
    or (cek["label_anotator1"] != cek["label_manual_1"]).any()
    or (cek["label_anotator2"] != cek["label_manual_2"]).any()
):
    sys.exit(f"GAGAL: {KONSENSUS.name} tidak sesuai lagi dengan label kedua anotator.")
if not ks["label_konsensus"].isin(KELAS).all():
    sys.exit(f"GAGAL: label_konsensus di {KONSENSUS.name} belum terisi seluruhnya.")
rujukan = lembar.copy()
rujukan["label_manual"] = (
    rujukan["id_sampel"]
    .map(dict(zip(ks["id_sampel"], ks["label_konsensus"])))
    .fillna(rujukan["label_manual"])
)

sumber = pd.read_csv(
    FILE_SUMBER,
    encoding="utf-8-sig",
    dtype=str,
    usecols=[
        "row_key",
        "tanggal_wib",
        "text_clean_bert",
        "label_final",
        "sumber",
        "status",
    ],
)
sumber = sumber.rename(columns={"label_final": "label_strata"})
label = pd.read_csv(
    FILE_LABEL, encoding="utf-8-sig", dtype=str, usecols=["row_key", KOL_LABEL]
)
data = (
    kunci.merge(rujukan.rename(columns={"label_manual": "manual"}), on="id_sampel")
    .merge(sumber, on="row_key", how="left", validate="one_to_one")
    .merge(
        label.rename(columns={KOL_LABEL: "model"}),
        on="row_key",
        how="left",
        validate="one_to_one",
    )
)
for k in ("model", "label_strata"):
    data[k] = norm_label(data[k])
if not data["model"].isin(KELAS).all():
    sys.exit(
        f"GAGAL: {int((~data['model'].isin(KELAS)).sum())} row_key tidak ditemukan atau "
        f"{KOL_LABEL} tidak valid di {FILE_LABEL.name}."
    )
beda = data.loc[
    norm_teks(data["teks"]) != norm_teks(data["text_clean_bert"]), "id_sampel"
]
if len(beda):
    sys.exit(
        f"GAGAL: teks {len(beda)} baris tidak cocok dengan {FILE_SUMBER.name}, contoh {beda.tolist()[:10]}. "
        "Lembar tersunting atau barisnya bergeser."
    )

sb = data["sumber"].fillna("").str.strip().str.lower()
st = data["status"].fillna("").str.strip().str.lower()
data["sumber"] = sb
data["jalur"] = np.select(
    [sb.eq("voting_3model"), sb.eq("leksikon") & st.isin(STATUS_LEKSIKON)],
    ["voting_3model", "leksikon_" + st],
    "lain",
)
tgl = pd.to_datetime(data["tanggal_wib"], errors="coerce").dt.normalize()
data["periode"] = np.select(
    [tgl <= pd.Timestamp(BATAS_PRA), tgl <= pd.Timestamp(BATAS_PUNCAK)],
    ["Pra", "Puncak"],
    "Pasca",
)

waktu_beku, berubah = bekukan("beku_anotator1", lembar)
berubah += bekukan(
    "beku_konsensus", ks.rename(columns={"label_konsensus": "label_manual"})
)[1]
info_berkas = []
if hashlib.sha256(FILE_SUMBER.read_bytes()).hexdigest() != prot["sha256_berkas_sumber"]:
    info_berkas.append(
        f"{FILE_SUMBER.name} berubah sejak sampel ditarik; jalur dan teks dibaca dari versi sekarang."
    )
if FILE_LABEL.resolve() != FILE_SUMBER.resolve():
    info_berkas.append(
        f"Label yang dinilai berasal dari {FILE_LABEL.name}, kolom {KOL_LABEL}."
    )

rng = np.random.default_rng(SEED)
m = metrik(data, rng)
lo, hi = m["ci"]

y, p = data["manual"], data["model"]
pr_k, rc_k, f1_k, sup = precision_recall_fscore_support(
    y, p, labels=KELAS, average=None, zero_division=0
)
qy = y.value_counts(normalize=True).reindex(KELAS, fill_value=0.0)
qp = p.value_counts(normalize=True).reindex(KELAS, fill_value=0.0)
f1_acak = (2 * qy * qp / (qy + qp)).fillna(0.0)
kelas = pd.DataFrame(
    {
        "kelas": KELAS,
        "precision": pr_k,
        "recall": rc_k,
        "f1": f1_k,
        "f1_pembanding_acak": f1_acak.to_numpy(),
        "n_manual": sup,
        f"n_{KOL_LABEL}": p.value_counts().reindex(KELAS, fill_value=0).to_numpy(),
    }
)
kelas.loc[len(kelas)] = [
    "macro",
    m["precision_macro"],
    m["recall_macro"],
    m["f1_macro"],
    f1_acak.mean(),
    int(sup.sum()),
    int(sup.sum()),
]

cm = pd.DataFrame(
    confusion_matrix(y, p, labels=KELAS),
    index=[f"manual_{k}" for k in KELAS],
    columns=[f"{KOL_LABEL}_{k}" for k in KELAS],
)
cm["total"] = cm.sum(axis=1)
cm.loc["total"] = cm.sum()


def koma(x):
    return f"{x:.2f}".replace(".", ",")


if m["kappa"] >= A["kappa_memadai"]:
    putus_kappa = "MEMADAI"
elif m["kappa"] >= A["kappa_layak"]:
    putus_kappa = "LAYAK, CATAT SEBAGAI KETERBATASAN"
else:
    putus_kappa = "GAGAL"
putus_acc = "LOLOS" if m["accuracy"] > m["proporsi_kelas_terbanyak"] else "GAGAL"
nol = "CI 95% memuat 0" if lo[3] <= 0 <= hi[3] else "CI 95% tidak memuat 0"
ambang_kappa = (
    f">= {koma(A['kappa_memadai'])} memadai; {koma(A['kappa_layak'])} s.d. < "
    f"{koma(A['kappa_memadai'])} layak; < {koma(A['kappa_layak'])} gagal"
)
baris = [
    ("N sampel", m["N"], None, "-", "-"),
    (
        "Accuracy",
        m["accuracy"],
        0,
        f"> {m['proporsi_kelas_terbanyak']:.4f} (proporsi kelas terbanyak label manual: {m['kelas_terbanyak']})",
        putus_acc,
    ),
    (
        "Proporsi kelas terbanyak (label manual)",
        m["proporsi_kelas_terbanyak"],
        None,
        "pembanding accuracy",
        "-",
    ),
    ("Precision (macro)", m["precision_macro"], None, "tanpa ambang baku", "-"),
    ("Recall (macro)", m["recall_macro"], None, "tanpa ambang baku", "-"),
    (
        "F1-score (macro)",
        m["f1_macro"],
        2,
        f"tanpa ambang baku; pembanding acak {f1_acak.mean():.4f}",
        "-",
    ),
    ("Cohen kappa", m["kappa"], 1, ambang_kappa, putus_kappa),
    (
        "Tukar negatif-positif (dokumen)",
        m["tukar_neg_pos"],
        None,
        "tanpa ambang baku, harus sedikit",
        "-",
    ),
    (
        "Tukar negatif-positif (% dari N)",
        m["tukar_pct_N"],
        None,
        "tanpa ambang baku",
        "-",
    ),
    (
        "Tukar negatif-positif (% dari pasangan polar)",
        m["tukar_pct_polar"],
        None,
        "tanpa ambang baku",
        "-",
    ),
    (
        "Porsi galat skor dari tukar polaritas (%)",
        m["porsi_galat_tukar_pct"],
        None,
        "tanpa ambang baku",
        "-",
    ),
    (
        f"Selisih rata-rata skor ({KOL_LABEL} - manual)",
        m["selisih_skor"],
        3,
        "0 = tanpa bias arah",
        nol,
    ),
]
utama = pd.DataFrame(
    {
        "Metrik": [b[0] for b in baris],
        "Nilai": pd.Series(
            [
                b[1] if isinstance(b[1], (int, np.integer)) else round(float(b[1]), 4)
                for b in baris
            ],
            dtype=object,
        ),
        "CI95_bawah": [np.nan if b[2] is None else round(lo[b[2]], 4) for b in baris],
        "CI95_atas": [np.nan if b[2] is None else round(hi[b[2]], 4) for b in baris],
        "Ambang atau pembanding": [b[3] for b in baris],
        "Putusan": [b[4] for b in baris],
    }
)

rincian = []
for kolom, urutan in (
    ("sumber", ["voting_3model", "leksikon"]),
    ("jalur", URUT_JALUR),
    ("periode", ["Pra", "Puncak", "Pasca"]),
):
    for g in urutan:
        d = data[data[kolom] == g].reset_index(drop=True)
        if d.empty:
            continue
        r = metrik(d, rng)
        rincian.append(
            {
                "kelompok": f"{kolom}={g}",
                "N": r["N"],
                "accuracy": r["accuracy"],
                "acc_CI95_bawah": r["ci"][0][0],
                "acc_CI95_atas": r["ci"][1][0],
                "kappa": r["kappa"],
                "kappa_CI95_bawah": r["ci"][0][1],
                "kappa_CI95_atas": r["ci"][1][1],
                "f1_macro": np.nan if np.isnan(r["kappa"]) else r["f1_macro"],
                "tukar_neg_pos": r["tukar_neg_pos"],
                "selisih_skor": r["selisih_skor"],
                "selisih_CI95_bawah": r["ci"][0][3],
                "selisih_CI95_atas": r["ci"][1][3],
            }
        )
rincian = pd.DataFrame(rincian)

antar = None
if LEMBAR_2.exists():
    l2 = baca_lembar(LEMBAR_2)
    n_isi = int(l2["label_manual"].isin(KELAS).sum())
    if n_isi == len(l2) >= MIN_ANOTATOR2 and set(l2["id_sampel"]) <= set(
        lembar["id_sampel"]
    ):
        bekukan("beku_anotator2", l2)
        g = l2.merge(lembar, on="id_sampel", suffixes=("_2", "_1"))
        a1, a2 = (
            g["label_manual_1"].map(KODE).to_numpy(),
            g["label_manual_2"].map(KODE).to_numpy(),
        )
        bt = np.array(
            [
                cepat(a1[i], a2[i])
                for i in (rng.integers(0, len(g), len(g)) for _ in range(N_BOOT))
            ]
        )
        c2 = np.nanpercentile(bt, [2.5, 97.5], axis=0)
        antar = {
            "N": len(g),
            "kesepakatan": (a1 == a2).mean(),
            "kappa": cohen_kappa_score(
                g["label_manual_1"], g["label_manual_2"], labels=KELAS
            ),
            "kappa_CI95": (c2[0][1], c2[1][1]),
        }
    else:
        print(
            f"INFO: {LEMBAR_2.name} terisi {n_isi} dari {len(l2)} baris; kesepakatan antarpelabel dilewati."
        )

skor = data["manual"].map(SKOR) * data["model"].map(SKOR)
tidak = data[data["manual"] != data["model"]].assign(
    jenis=lambda d: np.where(skor[d.index] == -1, "tukar_neg_pos", "netral_vs_polar")
)
tidak = tidak.sort_values(
    ["jenis", "jalur", "id_sampel"], ascending=[False, True, True]
)[["id_sampel", "jenis", "jalur", "periode", "manual", "model", "teks"]].rename(
    columns={"manual": "label_manual", "model": KOL_LABEL}
)

OUT.mkdir(parents=True, exist_ok=True)
sfx = f"N{m['N']}"
berkas = {
    "utama": OUT / f"tabel_validasi_label_{sfx}.csv",
    "kelas": OUT / f"tabel_validasi_per_kelas_{sfx}.csv",
    "cm": OUT / f"tabel_validasi_confusion_{sfx}.csv",
    "rincian": OUT / f"tabel_validasi_rincian_{sfx}.csv",
    "tidak": OUT / f"daftar_tidak_sepakat_{sfx}.csv",
}
utama.to_csv(berkas["utama"], index=False, encoding="utf-8-sig")
kelas.round(4).to_csv(berkas["kelas"], index=False, encoding="utf-8-sig")
cm.to_csv(berkas["cm"], encoding="utf-8-sig")
rincian.round(4).to_csv(berkas["rincian"], index=False, encoding="utf-8-sig")
tidak.to_csv(berkas["tidak"], index=False, encoding="utf-8-sig")

pd.set_option(
    "display.width", 200, "display.max_columns", 30, "display.max_colwidth", 70
)
print("=" * 78)
print(
    f"UJI ARAH MAJU PELABELAN | {KOL_LABEL} ({FILE_LABEL.name}) terhadap label manual buta"
)
print("=" * 78)
print(f"Protokol dan ambang dibuat : {prot['dibuat']}")
print(f"Label manual dibekukan     : {waktu_beku}")
for s in info_berkas:
    print(f"INFO: {s}")
if berubah:
    print(
        f"PERINGATAN KRITIS: {len(berubah)} label manual berubah sesudah evaluasi pertama dan tetap dipakai "
        f"karena IZINKAN_LABEL_BERUBAH = True. Wajib dilaporkan: {berubah}"
    )
print(
    f"Silang periksa numpy vs scikit-learn: cocok | bootstrap berlapis {N_BOOT} ulangan, seed {SEED}"
)
print(f"\nTABEL VALIDASI LABEL\n{utama.to_string(index=False)}")
print(f"\nCONFUSION MATRIX (baris = manual, kolom = {KOL_LABEL})\n{cm.to_string()}")
print(f"\nMETRIK PER KELAS\n{kelas.round(4).to_string(index=False)}")
print(f"\nRINCIAN PER KELOMPOK\n{rincian.round(4).to_string(index=False)}")
if antar:
    print(
        f"\nKESEPAKATAN ANTARPELABEL MANUSIA (N = {antar['N']}): kesepakatan {antar['kesepakatan']:.4f}, "
        f"kappa {antar['kappa']:.4f} [CI 95% {antar['kappa_CI95'][0]:.4f}; {antar['kappa_CI95'][1]:.4f}]"
    )

print("\n" + "=" * 78)
print("PUTUSAN (ambang dari protokol, diputus dari nilai titik)")
print("=" * 78)
print(
    f"  Cohen kappa : {m['kappa']:.4f} [CI 95% {lo[1]:.4f}; {hi[1]:.4f}] -> {putus_kappa}"
)
for t in (A["kappa_memadai"], A["kappa_layak"]):
    if lo[1] < t <= hi[1]:
        print(
            f"                CI 95% melintasi {koma(t)}; putusan tetap dari nilai titik, rentang dilaporkan."
        )
print(
    f"  Accuracy    : {m['accuracy']:.4f} [CI 95% {lo[0]:.4f}; {hi[0]:.4f}] vs proporsi kelas terbanyak "
    f"{m['proporsi_kelas_terbanyak']:.4f} ({m['kelas_terbanyak']}) -> {putus_acc}"
)
if putus_acc == "LOLOS" and lo[0] <= m["proporsi_kelas_terbanyak"]:
    print(
        "                Batas bawah CI 95% accuracy tidak melampaui pembanding; catat sebagai keterbatasan."
    )
print(
    f"  F1 macro    : {m['f1_macro']:.4f} [CI 95% {lo[2]:.4f}; {hi[2]:.4f}], pembanding acak "
    f"{f1_acak.mean():.4f}; tanpa ambang baku"
)
print(
    f"  Tukar neg-pos: {m['tukar_neg_pos']} dokumen ({m['tukar_pct_N']:.2f}% dari N; "
    f"{m['tukar_pct_polar']:.2f}% dari pasangan polar); menyumbang {m['porsi_galat_tukar_pct']:.2f}% galat skor"
)
print(
    f"  Arah label  : selisih rata-rata skor {m['selisih_skor']:+.4f} [CI 95% {lo[3]:+.4f}; {hi[3]:+.4f}] -> {nol}"
)
print(f"\nDisimpan di {OUT}:")
for f in berkas.values():
    print(f"  {f.name}")

UJI ARAH MAJU PELABELAN | label_final (FINAL 3.csv) terhadap label manual buta
Protokol dan ambang dibuat : 2026-09-28T18:14:10
Label manual dibekukan     : 2026-09-28T23:03:18
Silang periksa numpy vs scikit-learn: cocok | bootstrap berlapis 2000 ulangan, seed 42

TABEL VALIDASI LABEL
                                       Metrik    Nilai  CI95_bawah  CI95_atas                                Ambang atau pembanding         Putusan
                                     N sampel      300         NaN        NaN                                                     -               -
                                     Accuracy     0.43      0.3800     0.4800 > 0.6267 (proporsi kelas terbanyak label manual: neu)           GAGAL
      Proporsi kelas terbanyak (label manual)   0.6267         NaN        NaN                                   pembanding accuracy               -
                            Precision (macro)   0.4911         NaN        NaN                                     tanpa am

Visualisasi untuk karakteristik sentimen 

In [ ]:
from pathlib import Path
import pandas as pd

D = Path(r"G:\My Drive\skripsi\proje\proje\data")
F_KORPUS = D / r"interim\dataset_multilabel_final.csv"
F_AKUN = D / r"interim\hasil 1,\6. dataset_final_bersih.csv"
F_TINJAU = D / r"hasil\Hasil Akhir\tinjau_jenis_akun.csv"
N_TERATAS = 200
POLA_BERITA = (
    r"news|berita|kabar|warta|media|harian|koran|times|daily|tv$|fm$|com$|dotco|"
    r"kompas|detik|tribun|tempo|cnn|cnbc|kontan|bisnis|antara|liputan|okezone|sindo|"
    r"republika|kumparan|tirto|viva|suara|merdeka|katadata|idntimes|jpnn|rmol|metro|"
    r"inews|bloomberg|investor|beritasatu|jawapos"
)


def baca(f):
    d = pd.read_csv(f, dtype=str, encoding="utf-8-sig", low_memory=False)
    d.columns = [c.strip().lstrip("\ufeff") for c in d.columns]
    return d


def muat():
    k = baca(F_KORPUS)
    perlu = [c for c in ("username", "verified", "followers") if c not in k.columns]
    if perlu:
        k = k.merge(
            baca(F_AKUN)[["id"] + perlu].drop_duplicates("id"), on="id", how="left"
        )
    k["username"] = k["username"].str.strip().str.lower()
    k["v"] = k["verified"].str.strip().str.lower().map({"true": 1.0, "false": 0.0})
    return k


k = muat()
n = len(k)
akun = (
    k.groupby("username", dropna=False)
    .agg(
        n_unggahan=("id", "size"),
        porsi_verified=("v", "mean"),
        followers=("followers", lambda s: pd.to_numeric(s, errors="coerce").max()),
    )
    .reset_index()
    .sort_values("n_unggahan", ascending=False)
)
akun["peringkat"] = range(1, len(akun) + 1)
akun["kandidat_pola"] = akun["username"].fillna("").str.contains(POLA_BERITA)
tinjau = akun[akun["kandidat_pola"] | (akun["peringkat"] <= N_TERATAS)].copy()
tinjau["jenis_akun"] = ""
tinjau.to_csv(F_TINJAU, index=False, encoding="utf-8-sig")

campur = int(((akun["porsi_verified"] > 0) & (akun["porsi_verified"] < 1)).sum())
print(
    f"unggahan {n} | akun unik {len(akun)} | username kosong {int(k['username'].isna().sum())}"
)
print(f"akun dengan status verifikasi campuran : {campur}")
print(f"kandidat pola {int(akun['kandidat_pola'].sum())} | akun ditinjau {len(tinjau)}")
print(
    f"cakupan tinjauan : {int(tinjau['n_unggahan'].sum())} unggahan ({tinjau['n_unggahan'].sum() / n:.2%})"
)
print(f"Tersimpan: {F_TINJAU.name}")

unggahan 14050 | akun unik 4075 | username kosong 0
akun dengan status verifikasi campuran : 2
kandidat pola 124 | akun ditinjau 286
cakupan tinjauan : 8707 unggahan (61.97%)
Tersimpan: tinjau_jenis_akun.csv


In [ ]:
POLA_KUAT = (
    r"kompas|detik|tribun|tempo|cnn|cnbc|kontan|antara|liputan6|okezone|sindo|republika|"
    r"kumparan|tirto|viva|merdeka|katadata|idntimes|jpnn|rmol|metrotv|inews|bloomberg|"
    r"beritasatu|jawapos|news|berita|tv$|dotco|com$"
)
KOREKSI_BERITA = []
KOREKSI_NON = []

t = baca(F_TINJAU)
u = t["username"].fillna("")
t["jenis_akun"] = u.str.contains(POLA_KUAT).map({True: "berita", False: "non_berita"})
t.loc[u.isin(KOREKSI_BERITA), "jenis_akun"] = "berita"
t.loc[u.isin(KOREKSI_NON), "jenis_akun"] = "non_berita"
t.to_csv(F_TINJAU, index=False, encoding="utf-8-sig")

for j in ("berita", "non_berita"):
    s = t[t["jenis_akun"] == j]
    print(
        f"\n{j.upper()} | {len(s)} akun | {int(pd.to_numeric(s['n_unggahan']).sum())} unggahan"
    )
    print(
        ", ".join(
            f"{a}({b})"
            for a, b in zip(s["username"].fillna("(kosong)"), s["n_unggahan"])
        )
    )


BERITA | 91 akun | 3432 unggahan
cnbcindonesia(1272), detikfinance(344), kontannews(236), kompascom(171), bloombergtz(133), antaranews(118), beritasatu(102), kumparan(97), bisniscom(96), rmol_id(87), hariankompas(86), cnnindonesia(80), detikcom(73), metro_tv(61), officialinews_(47), tirtoid(42), vivacoid(34), inilahdotcom(25), tvonenews(23), jawapos(22), sindonews(18), liputan6dotcom(16), tempodotco(13), kompastv(13), idntimes(11), informasicom(10), bolasaljucom(10), mikirduitcom(10), detikinet(9), katadatacoid(8), suaramerdeka(8), kompasiana(8), softwarnews(7), keuangannews_id(7), nusantarahukum(7), merdekadotcom(6), narasinewsroom(6), republikaonline(6), tribunnews(6), serambinews(5), energijuangnews(5), bolasportcom(5), okezonenews(5), insertlivettv(5), tribunpontianak(4), detikoto(4), infosawitcom(4), beritakbr(3), mynewshub(3), asmurcom(3), genznusantara(3), garudatvnews(3), teddynewsid(3), kompastekno(3), kompasngidol(3), majunusantaraku(2), mojokdotco(2), kabarbursacom(2), blib

In [ ]:
F_TABEL = D / r"hasil\Hasil Akhir\tabel_karakteristik_sumber_jenis_akun.csv"
SAH = {"berita", "non_berita"}

t = baca(F_TINJAU)
t["jenis_akun"] = t["jenis_akun"].fillna("").str.strip().str.lower()
belum = t[~t["jenis_akun"].isin(SAH)]
if len(belum):
    raise SystemExit(
        f"{len(belum)} akun belum berisi berita/non_berita, contoh: {belum['username'].head(5).tolist()}"
    )

k = muat()
n = len(k)
peta = dict(zip(t["username"], t["jenis_akun"]))
k["ditinjau"] = k["username"].isin(peta)
k["jenis"] = k["username"].map(peta).fillna("non_berita")
k["status"] = (
    k["v"]
    .map({1.0: "terverifikasi", 0.0: "tidak_terverifikasi"})
    .fillna("tidak_diketahui")
)

print(
    f"unggahan {n} | dari akun yang ditinjau {int(k['ditinjau'].sum())} ({k['ditinjau'].mean():.2%})"
)
print(
    f"akun berlabel berita {int((t['jenis_akun'] == 'berita').sum())} dari {len(t)} akun yang ditinjau"
)
print("\nUNGGAHAN (jenis x status)")
print(pd.crosstab(k["jenis"], k["status"], margins=True).to_string())
print("\nPERSEN TERHADAP SELURUH UNGGAHAN")
print(
    (pd.crosstab(k["jenis"], k["status"], margins=True, normalize="all") * 100)
    .round(2)
    .to_string()
)

tabel = (
    k.groupby(["jenis", "status"])
    .agg(jumlah_akun=("username", "nunique"), jumlah_unggahan=("id", "size"))
    .reset_index()
)
tabel["persen_unggahan"] = (tabel["jumlah_unggahan"] / n * 100).round(2)
tabel.to_csv(F_TABEL, index=False, encoding="utf-8-sig")
print(f"\n{tabel.to_string(index=False)}\nTersimpan: {F_TABEL.name}")

unggahan 14050 | dari akun yang ditinjau 8707 (61.97%)
akun berlabel berita 91 dari 286 akun yang ditinjau

UNGGAHAN (jenis x status)
status      terverifikasi  tidak_terverifikasi    All
jenis                                                
berita               2549                  883   3432
non_berita           5140                 5478  10618
All                  7689                 6361  14050

PERSEN TERHADAP SELURUH UNGGAHAN
status      terverifikasi  tidak_terverifikasi     All
jenis                                                 
berita              18.14                 6.28   24.43
non_berita          36.58                38.99   75.57
All                 54.73                45.27  100.00

     jenis              status  jumlah_akun  jumlah_unggahan  persen_unggahan
    berita       terverifikasi           41             2549            18.14
    berita tidak_terverifikasi           50              883             6.28
non_berita       terverifikasi          766         

Visualisasi untuk kalimat sentimen yang sering muncul. 

In [ ]:
"""Tahap 7 | Kata kunci, konteks kata kunci, dan visualisasinya pada korpus 14.050 dokumen."""
import random
import re
from collections import Counter, defaultdict
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

D = Path(r"G:\My Drive\skripsi\proje\proje\data")
F_KORPUS = D / r"interim\dataset_multilabel_final.csv"
OUT = D / r"hasil\Hasil Akhir\kata_kunci"

KOL_TEKS = "text_clean_bert"
KOL_TGL_WIB = "tanggal_wib"
KOL_KUNCI = "id"
KOL_TGL = "tanggal"
KOL_PERIODE = "periode"
N_HARAP = 14050
BATAS_PERIODE = [
    ("Pra-peristiwa", "2025-06-01", "2025-08-24"),
    ("Puncak peristiwa", "2025-08-25", "2025-09-09"),
    ("Pasca-peristiwa", "2025-09-10", "2025-10-31"),
]
PERIODE = [b[0] for b in BATAS_PERIODE]
PUNCAK = BATAS_PERIODE[1][1:]

KATA_PASAR = r"\b(?:ihsg|saham|investasi)\b"
KATA_POLITIK = r"\b(?:demo|unjuk rasa|pemerintah)\b"
BASIS = ("bab3", "pipeline", "diperluas")
BASIS_GAMBAR = "bab3"
JENDELA = 5
N_KOLOKASI = 15
N_KWIC = 5
LEBAR_KWIC = 70
N_UNIGRAM = 30
N_BIGRAM = 20
N_SILANG = 10
MIN_DOK_PETA = 30
PANJANG_MIN = 3
BENIH = 42

KAMUS = {
    "Pasar modal": {
        "bab3": ["ihsg", "saham", "investasi"],
        "kueri": ["bursa", "bei", "pasar modal", "investor", "modal keluar"],
        "saringan": [
            "emiten",
            "rupiah",
            "jkse",
            "jci",
            "idx",
            "lq45",
            "idx30",
            "reksa dana",
            "obligasi",
            "sbn",
            "sukuk",
            "kode emiten",
        ],
        "varian": ["trading", "portofolio", "dividen", "ipo"],
    },
    "Demonstrasi": {
        "bab3": ["demo", "unjuk rasa"],
        "kueri": ["rusuh"],
        "varian": [
            "demonstrasi",
            "demonstran",
            "pendemo",
            "kerusuhan",
            "ricuh",
            "kericuhan",
            "aksi massa",
            "penjarahan",
        ],
    },
    "Politik dan pemerintahan": {
        "bab3": ["pemerintah"],
        "kueri": ["dpr", "politik", "ketidakpastian", "putusan"],
        "varian": ["pemerintahan", "politisi"],
    },
    "Ekonomi makro": {
        "kueri": ["ekonomi", "inflasi", "krisis"],
    },
}

KELOMPOK = {
    "Pasar modal": ["Pasar modal"],
    "Peristiwa politik": ["Demonstrasi", "Politik dan pemerintahan"],
    "Demonstrasi": ["Demonstrasi"],
    "Politik dan pemerintahan": ["Politik dan pemerintahan"],
    "Ekonomi makro": ["Ekonomi makro"],
}

KODE_EMITEN = (
    "bbca|bbri|bmri|bbni|bris|tlkm|asii|goto|brpt|adro|aadi|antm|mdka|"
    "ptba|pgas|smgr|unvr|icbp|indf|klbf|inco|tins|itmg|untr|cpin|excl|"
    "isat|towr|amrt|hmsp|ggrm|buka|emtk|arto|bbtn|pgeo|nckl"
)
POLA_KHUSUS = {"reksa dana": r"reksa ?dana", "kode emiten": KODE_EMITEN}

STOP_TAMBAHAN = set()
STOP = set(
    """
    yang dan di ke dari ini itu untuk dengan pada adalah akan atau juga karena
    tidak tak bukan sudah telah belum masih lagi saja hanya bisa dapat harus
    ada jadi menjadi oleh dalam atas bawah sebagai seperti kalau jika bila maka
    agar supaya tapi tetapi namun sedangkan serta hingga sampai sejak selama
    saat ketika waktu setelah sebelum kemudian lalu lebih paling sangat banyak
    sedikit semua setiap para antara tanpa bagi kepada terhadap tentang secara
    apa siapa mana kapan kenapa mengapa bagaimana berapa saya aku kami kita
    kamu anda dia mereka ia nya pun lah kah dong deh sih nih tuh kok kan ya
    yaa iya gak ga nggak enggak ngga tdk bgt banget aja doang udah udh sdh
    lagi lg jg juga sama dgn utk yg krn tp klo kalo gitu gini begitu begini
    biar buat bikin mau ingin baru lama hari kali tahun bulan minggu kemarin
    besok nanti sekarang tadi sini situ sana hal cara pihak orang ini itu
    tersebut yakni yaitu ialah merupakan bahwa per dll dsb rt amp via the of
    to in and is for on its ada adanya jadi makin semakin terus tetap malah
    justru memang mungkin kayak kaya seperti sekali pernah selalu sering
    langsung masuk keluar punya milik guna usai pasca oleh nya bakal lewat
    antara sekitar sebesar senilai mencapai sebanyak dua tiga satu
    """.split()
)

WARNA = {
    "Pasar modal": "#1f4e79",
    "Demonstrasi": "#c00000",
    "Politik dan pemerintahan": "#e08e45",
    "Ekonomi makro": "#7a7973",
}
RE_TOKEN = re.compile(r"[a-z][a-z0-9_]*")

pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 60)
pd.set_option("display.max_colwidth", 80)
plt.rcParams.update(
    {
        "figure.dpi": 100,
        "savefig.dpi": 300,
        "savefig.bbox": "tight",
        "font.size": 9,
        "axes.spines.top": False,
        "axes.spines.right": False,
    }
)


def judul(teks):
    print("\n" + "=" * 78)
    print(teks)
    print("=" * 78)


def simpan(df, nama):
    df.to_csv(OUT / nama, index=False, encoding="utf-8-sig")
    print(f"Disimpan: {nama} ({len(df)} baris)")


def baca():
    if not F_KORPUS.exists():
        raise SystemExit(f"GAGAL: {F_KORPUS} tidak ditemukan.")
    df = pd.read_csv(F_KORPUS, dtype=str, encoding="utf-8-sig", low_memory=False)
    df.columns = [c.strip().lstrip("\ufeff") for c in df.columns]
    kurang = [k for k in (KOL_TEKS, KOL_TGL_WIB, KOL_KUNCI) if k not in df.columns]
    if kurang:
        raise SystemExit(f"GAGAL: kolom tidak ada: {kurang}")
    if len(df) != N_HARAP:
        raise SystemExit(f"GAGAL: {len(df)} baris, seharusnya {N_HARAP}.")
    if df[KOL_KUNCI].duplicated().any():
        raise SystemExit(f"GAGAL: {KOL_KUNCI} ganda.")
    df[KOL_TEKS] = df[KOL_TEKS].fillna("").str.lower()
    df[KOL_TGL] = pd.to_datetime(df[KOL_TGL_WIB].str[:10], errors="coerce")
    if df[KOL_TGL].isna().any():
        raise SystemExit(
            f"GAGAL: {int(df[KOL_TGL].isna().sum())} tanggal gagal dibaca."
        )
    df[KOL_PERIODE] = None
    for nama, mulai, selesai in BATAS_PERIODE:
        m = df[KOL_TGL].between(pd.Timestamp(mulai), pd.Timestamp(selesai))
        df.loc[m, KOL_PERIODE] = nama
    if df[KOL_PERIODE].isna().any():
        raise SystemExit(
            f"GAGAL: {int(df[KOL_PERIODE].isna().sum())} dokumen di luar ketiga sub-periode."
        )
    return df.reset_index(drop=True)


def daftar_kata_kunci():
    baris = [
        (kat, k, asal)
        for kat, grup in KAMUS.items()
        for asal, isi in grup.items()
        for k in isi
    ]
    kk = pd.DataFrame(baris, columns=["kategori", "kata_kunci", "asal"])
    ganda = kk.loc[kk["kata_kunci"].duplicated(), "kata_kunci"].tolist()
    if ganda:
        raise SystemExit(f"GAGAL: kata kunci ganda di KAMUS: {ganda}")
    return kk


def susun_pola(k):
    return re.compile(rf"\b(?:{POLA_KHUSUS.get(k) or re.escape(k)})\b")


def anggota(kk, kategori, basis):
    asal = {"bab3": ["bab3"], "pipeline": ["bab3", "kueri", "saringan"]}.get(basis)
    m = kk["kategori"].isin(kategori)
    if asal:
        m &= kk["asal"].isin(asal)
    return kk.loc[m, "kata_kunci"].tolist()


def bendera_kelompok(ada, kk, basis):
    return pd.DataFrame(
        {
            nama: ada[anggota(kk, kat, basis)].any(axis=1)
            for nama, kat in KELOMPOK.items()
        }
    )


def tabel_cakupan(ada, kk):
    n = len(ada)
    tunggal = {
        "Memuat kata kunci pasar modal": "Pasar modal",
        "Memuat kata kunci peristiwa politik": "Peristiwa politik",
        "   di antaranya: demonstrasi": "Demonstrasi",
        "   di antaranya: politik dan pemerintahan": "Politik dan pemerintahan",
        "Memuat kata kunci ekonomi makro": "Ekonomi makro",
    }
    kol = {}
    for basis in BASIS:
        g = bendera_kelompok(ada, kk, basis)
        pm, pol, demo = g["Pasar modal"], g["Peristiwa politik"], g["Demonstrasi"]
        baris = {k: g[v] for k, v in tunggal.items()}
        baris.update(
            {
                "Memuat pasar modal dan peristiwa politik sekaligus": pm & pol,
                "Memuat pasar modal dan demonstrasi sekaligus": pm & demo,
                "Tanpa kata kunci pasar modal": ~pm,
                "Tanpa kata kunci pasar modal maupun peristiwa politik": ~pm & ~pol,
                "Tanpa kata kunci dari kelompok mana pun": ~g.any(axis=1),
                "Total dokumen": pd.Series(True, index=ada.index),
            }
        )
        hampa = {k for k, v in tunggal.items() if not anggota(kk, KELOMPOK[v], basis)}
        kol[f"n_{basis}"] = {
            k: pd.NA if k in hampa else int(v.sum()) for k, v in baris.items()
        }
        kol[f"persen_{basis}"] = {
            k: pd.NA if k in hampa else round(100 * v.sum() / n, 2)
            for k, v in baris.items()
        }
    t = pd.DataFrame(kol)
    for basis in BASIS:
        t[f"n_{basis}"] = t[f"n_{basis}"].astype("Int64")
        t[f"persen_{basis}"] = t[f"persen_{basis}"].astype("Float64").astype(float)
    return t.rename_axis("karakteristik").reset_index()


def tabel_cakupan_periode(ada, kk, periode):
    n_per = periode.value_counts().reindex(PERIODE)
    baris = []
    for basis in BASIS:
        g = bendera_kelompok(ada, kk, basis)
        g = g[[k for k in g.columns if anggota(kk, KELOMPOK[k], basis)]].copy()
        g["Pasar modal dan demonstrasi"] = g["Pasar modal"] & g["Demonstrasi"]
        g["Pasar modal dan peristiwa politik"] = (
            g["Pasar modal"] & g["Peristiwa politik"]
        )
        for nama in g.columns:
            n = g[nama].groupby(periode).sum().reindex(PERIODE)
            for p in PERIODE:
                baris.append(
                    {
                        "basis": basis,
                        "kelompok": nama,
                        "periode": p,
                        "n_dokumen_periode": int(n_per[p]),
                        "n_memuat": int(n[p]),
                        "persen": round(100 * n[p] / n_per[p], 2),
                    }
                )
    return pd.DataFrame(baris)


def tabel_kata_kunci(jml, kk, periode):
    ada = jml > 0
    n = len(jml)
    n_per = periode.value_counts().reindex(PERIODE)
    t = kk.copy()
    t["n_dokumen"] = ada.sum().reindex(t["kata_kunci"]).to_numpy()
    t["persen_dokumen"] = (100 * t["n_dokumen"] / n).round(2)
    t["n_kemunculan"] = jml.sum().reindex(t["kata_kunci"]).to_numpy()
    per = ada.groupby(periode).sum().reindex(PERIODE)
    for p in PERIODE:
        t[f"n_{p}"] = per.loc[p].reindex(t["kata_kunci"]).to_numpy()
        t[f"persen_{p}"] = (100 * t[f"n_{p}"] / n_per[p]).round(2)
    return t.sort_values(
        ["kategori", "n_dokumen"], ascending=[True, False]
    ).reset_index(drop=True)


def susun_token(teks, kk, pola):
    frasa = sorted(
        (k for k in kk["kata_kunci"] if " " in k and k != "kode emiten"),
        key=len,
        reverse=True,
    )
    peta = {t: "kode_emiten" for t in KODE_EMITEN.split("|")}
    semua = []
    for t in teks:
        for k in frasa:
            t = pola[k].sub(" " + k.replace(" ", "_") + " ", t)
        tok = []
        for w in RE_TOKEN.findall(t):
            tok.append(peta.get(w, w))
        semua.append(tok)
    target = {k.replace(" ", "_") for k in kk["kata_kunci"]}
    henti = STOP | STOP_TAMBAHAN

    def sah(w):
        return w not in henti and (len(w) >= PANJANG_MIN or w in target)

    isi = [[w for w in tok if sah(w)] for tok in semua]
    return semua, isi, target, sah


def tabel_kolokasi(isi, target, kk):
    frek = Counter()
    sekitar = defaultdict(Counter)
    for tok in isi:
        frek.update(tok)
        for i, w in enumerate(tok):
            if w in target:
                kiri = tok[max(0, i - JENDELA) : i]
                kanan = tok[i + 1 : i + 1 + JENDELA]
                sekitar[w].update(x for x in kiri + kanan if x != w)
    total = sum(frek.values())
    kat = dict(zip(kk["kata_kunci"], kk["kategori"]))
    baris = []
    for k in kk["kata_kunci"]:
        c = sekitar.get(k.replace(" ", "_"))
        if not c:
            continue
        tot_c = sum(c.values())
        for r, (x, n) in enumerate(c.most_common(N_KOLOKASI), 1):
            baris.append(
                {
                    "kategori": kat[k],
                    "kata_kunci": k,
                    "peringkat": r,
                    "kata_konteks": x.replace("_", " "),
                    "n_bersama": n,
                    "rasio_terhadap_korpus": round((n / tot_c) / (frek[x] / total), 2),
                }
            )
    return pd.DataFrame(baris), frek


def tabel_kata_umum(semua, isi, sah):
    df_uni = Counter()
    bi = Counter()
    df_bi = Counter()
    uni = Counter()
    for tok_semua, tok_isi in zip(semua, isi):
        uni.update(tok_isi)
        df_uni.update(set(tok_isi))
        pasangan = [
            (a, b)
            for a, b in zip(tok_semua, tok_semua[1:])
            if sah(a) and sah(b) and a != b
        ]
        bi.update(pasangan)
        df_bi.update(set(pasangan))
    n = len(isi)
    t_uni = pd.DataFrame(
        [
            {
                "kata": w.replace("_", " "),
                "n_kemunculan": c,
                "n_dokumen": df_uni[w],
                "persen_dokumen": round(100 * df_uni[w] / n, 2),
            }
            for w, c in uni.most_common(200)
        ]
    )
    t_bi = pd.DataFrame(
        [
            {
                "pasangan_kata": f"{a} {b}".replace("_", " "),
                "n_kemunculan": c,
                "n_dokumen": df_bi[(a, b)],
                "persen_dokumen": round(100 * df_bi[(a, b)] / n, 2),
            }
            for (a, b), c in bi.most_common(200)
        ]
    )
    return t_uni, t_bi, uni


def tabel_kwic(df, ada, kk, pola):
    acak = random.Random(BENIH)
    baris = []
    for kat, k in zip(kk["kategori"], kk["kata_kunci"]):
        idx = ada.index[ada[k]].tolist()
        for i in sorted(acak.sample(idx, min(N_KWIC, len(idx)))):
            t = df.at[i, KOL_TEKS]
            m = pola[k].search(t)
            baris.append(
                {
                    "kategori": kat,
                    "kata_kunci": k,
                    KOL_KUNCI: df.at[i, KOL_KUNCI],
                    "tanggal": df.at[i, KOL_TGL].date(),
                    "periode": df.at[i, KOL_PERIODE],
                    "kiri": t[max(0, m.start() - LEBAR_KWIC) : m.start()],
                    "kata": m.group(0),
                    "kanan": t[m.end() : m.end() + LEBAR_KWIC],
                }
            )
    return pd.DataFrame(baris)


def tabel_silang(ada, t_kk):
    def teratas(kategori):
        s = t_kk[t_kk["kategori"].isin(kategori) & (t_kk["n_dokumen"] > 0)]
        return s.nlargest(N_SILANG, "n_dokumen")["kata_kunci"].tolist()

    brs = teratas(["Pasar modal"])
    kol = teratas(["Demonstrasi", "Politik dan pemerintahan"])
    a = ada[brs].to_numpy(dtype=int)
    b = ada[kol].to_numpy(dtype=int)
    return pd.DataFrame(a.T @ b, index=brs, columns=kol)


def tabel_harian(df, ada, kk):
    g = bendera_kelompok(ada, kk, BASIS_GAMBAR)
    h = pd.DataFrame(
        {
            "n_dokumen": 1,
            "n_demonstrasi": g["Demonstrasi"].astype(int),
            "n_peristiwa_politik": g["Peristiwa politik"].astype(int),
        }
    )
    h = h.groupby(df[KOL_TGL]).sum()
    h = h.reindex(pd.date_range(h.index.min(), h.index.max(), freq="D"), fill_value=0)
    tot7 = h["n_dokumen"].rolling(7, min_periods=1).sum()
    for nama in ("demonstrasi", "peristiwa_politik"):
        h[f"persen_{nama}"] = (
            100 * h[f"n_{nama}"] / h["n_dokumen"].where(h["n_dokumen"] > 0)
        ).round(2)
        h[f"persen_{nama}_7h"] = (
            100 * h[f"n_{nama}"].rolling(7, min_periods=1).sum() / tot7.where(tot7 > 0)
        ).round(2)
    return h.rename_axis("tanggal").reset_index()


def gambar_frekuensi(t_kk):
    d = t_kk[t_kk["n_dokumen"] > 0].sort_values(
        ["kategori", "n_dokumen"], ascending=[False, True]
    )
    label = [
        k + ("*" if a == "varian" else "") for k, a in zip(d["kata_kunci"], d["asal"])
    ]
    tebal = d["asal"].eq("bab3").tolist()
    fig, ax = plt.subplots(figsize=(8, 0.26 * len(d) + 1.2))
    b = ax.barh(
        label, d["n_dokumen"], color=[WARNA[k] for k in d["kategori"]], height=0.72
    )
    ax.bar_label(
        b,
        labels=[f"{v:,}".replace(",", ".") for v in d["n_dokumen"]],
        padding=2,
        fontsize=8,
    )
    ax.set_xlabel(
        "Jumlah dokumen yang memuat kata kunci (tebal: kata kunci Bab III; *: varian)"
    )
    for teks_label, t in zip(ax.get_yticklabels(), tebal):
        teks_label.set_fontweight("bold" if t else "normal")
    ax.xaxis.set_major_formatter(lambda v, _: f"{int(v):,}".replace(",", "."))
    ax.margins(y=0.01)
    ax.legend(
        handles=[plt.Rectangle((0, 0), 1, 1, color=WARNA[k]) for k in KAMUS],
        labels=list(KAMUS),
        frameon=False,
        fontsize=8,
        ncol=len(KAMUS),
        loc="lower left",
        bbox_to_anchor=(0, 1.0),
    )
    fig.savefig(OUT / "gambar_kk_1_frekuensi_kata_kunci.png")
    plt.close(fig)


def gambar_kata_umum(t_uni, t_bi):
    fig, axes = plt.subplots(1, 2, figsize=(12, 0.24 * max(N_UNIGRAM, N_BIGRAM) + 1.2))
    for ax, t, kol, n, ket in (
        (axes[0], t_uni, "kata", N_UNIGRAM, "Kata tunggal"),
        (axes[1], t_bi, "pasangan_kata", N_BIGRAM, "Pasangan kata berurutan"),
    ):
        d = t.head(n).iloc[::-1]
        ax.barh(d[kol], d["n_dokumen"], color="#1f4e79", height=0.72)
        ax.set_xlabel("Jumlah dokumen")
        ax.xaxis.set_major_formatter(lambda v, _: f"{int(v):,}".replace(",", "."))
        ax.set_title(ket, fontsize=9, loc="left")
        ax.margins(y=0.01)
    fig.tight_layout()
    fig.savefig(OUT / "gambar_kk_2_kata_sering_muncul.png")
    plt.close(fig)


def peta_panas(ax, nilai, baris, kolom, fmt, cmap):
    im = ax.imshow(nilai, aspect="auto", cmap=cmap)
    ax.set_xticks(range(len(kolom)), kolom)
    ax.set_yticks(range(len(baris)), baris)
    batas = np.nanmax(nilai) * 0.6 if nilai.size else 0
    for i in range(nilai.shape[0]):
        for j in range(nilai.shape[1]):
            ax.text(
                j,
                i,
                fmt(nilai[i, j]),
                ha="center",
                va="center",
                fontsize=7.5,
                color="white" if nilai[i, j] > batas else "black",
            )
    for s in ax.spines.values():
        s.set_visible(False)
    return im


def gambar_periode(t_kk):
    d = t_kk[t_kk["n_dokumen"] >= MIN_DOK_PETA]
    if d.empty:
        return
    nilai = d[[f"persen_{p}" for p in PERIODE]].to_numpy(dtype=float)
    label = [f"{k} ({kat})" for k, kat in zip(d["kata_kunci"], d["kategori"])]
    fig, ax = plt.subplots(figsize=(7.5, 0.27 * len(d) + 1.2))
    im = peta_panas(
        ax, nilai, label, PERIODE, lambda v: f"{v:.1f}".replace(".", ","), "Blues"
    )
    fig.colorbar(
        im, ax=ax, fraction=0.035, pad=0.02, label="% dokumen pada sub-periode"
    )
    fig.savefig(OUT / "gambar_kk_3_kata_kunci_per_periode.png")
    plt.close(fig)


def gambar_harian(h):
    a, b = pd.Timestamp(PUNCAK[0]), pd.Timestamp(PUNCAK[1])
    fig, (ax1, ax2) = plt.subplots(
        2, 1, figsize=(11, 5.6), sharex=True, gridspec_kw={"hspace": 0.1}
    )
    for ax in (ax1, ax2):
        ax.axvspan(
            a,
            b + pd.Timedelta(days=1),
            color="#e0e0e0",
            zorder=0,
            label="Puncak peristiwa",
        )
    ax1.bar(
        h["tanggal"],
        h["n_dokumen"],
        width=0.8,
        color="#7a7973",
        label="Seluruh dokumen",
    )
    ax1.bar(
        h["tanggal"],
        h["n_demonstrasi"],
        width=0.8,
        color="#c00000",
        label="Memuat kata kunci demonstrasi",
    )
    ax1.set_ylabel("Jumlah dokumen")
    ax2.plot(
        h["tanggal"],
        h["persen_peristiwa_politik_7h"],
        color="#e08e45",
        lw=1.3,
        label="Peristiwa politik, jendela 7 hari",
    )
    ax2.plot(
        h["tanggal"],
        h["persen_demonstrasi_7h"],
        color="#c00000",
        lw=1.3,
        label="Demonstrasi, jendela 7 hari",
    )
    ax2.set_ylabel("% dokumen")
    ax2.set_ylim(bottom=0)
    h1, l1 = ax1.get_legend_handles_labels()
    h2, l2 = ax2.get_legend_handles_labels()
    ax1.legend(
        h1 + h2[1:],
        l1 + l2[1:],
        frameon=False,
        fontsize=8,
        ncol=3,
        loc="lower left",
        bbox_to_anchor=(0, 1.0),
    )
    fig.savefig(OUT / "gambar_kk_4_kata_kunci_harian.png")
    plt.close(fig)


def gambar_silang(silang):
    if silang.empty:
        return
    fig, ax = plt.subplots(
        figsize=(1.0 * silang.shape[1] + 2.5, 0.42 * silang.shape[0] + 1.5)
    )
    im = peta_panas(
        ax,
        silang.to_numpy(dtype=float),
        list(silang.index),
        list(silang.columns),
        lambda v: f"{int(v):,}".replace(",", "."),
        "Reds",
    )
    ax.tick_params(axis="x", rotation=35)
    plt.setp(ax.get_xticklabels(), ha="right")
    ax.set_ylabel("Kata kunci pasar modal")
    ax.set_xlabel("Kata kunci peristiwa politik")
    fig.colorbar(
        im, ax=ax, fraction=0.035, pad=0.02, label="Jumlah dokumen yang memuat keduanya"
    )
    fig.savefig(OUT / "gambar_kk_5_silang_pasar_politik.png")
    plt.close(fig)


def gambar_awan(uni):
    try:
        from wordcloud import WordCloud
    except ImportError:
        print(
            "Awan kata dilewati: paket wordcloud belum terpasang (pip install wordcloud)."
        )
        return
    frek = {w.replace("_", " "): c for w, c in uni.most_common(150)}
    wc = WordCloud(
        width=1800,
        height=1000,
        background_color="white",
        colormap="Blues_r",
        prefer_horizontal=1.0,
        random_state=BENIH,
        collocations=False,
    ).generate_from_frequencies(frek)
    fig, ax = plt.subplots(figsize=(9, 5))
    ax.imshow(wc, interpolation="bilinear")
    ax.axis("off")
    fig.savefig(OUT / "gambar_kk_6_awan_kata.png")
    plt.close(fig)
    print("Disimpan: gambar_kk_6_awan_kata.png")


def main():
    OUT.mkdir(parents=True, exist_ok=True)
    df = baca()
    kk = daftar_kata_kunci()
    pola = {k: susun_pola(k) for k in kk["kata_kunci"]}
    teks = df[KOL_TEKS].tolist()
    jml = pd.DataFrame({k: [len(p.findall(t)) for t in teks] for k, p in pola.items()})
    ada = jml > 0
    periode = df[KOL_PERIODE]

    judul("0. DATA, DAFTAR KATA KUNCI, DAN TOTAL KONTROL")
    print(f"Berkas  : {F_KORPUS.name}")
    print(
        f"Dokumen : {len(df)} | {df[KOL_TGL].min().date()} s.d. {df[KOL_TGL].max().date()}"
    )
    print(
        f"Dokumen per sub-periode (basis kalender):\n{periode.value_counts().reindex(PERIODE).to_string()}"
    )
    print("\nJumlah kata kunci per kategori dan asal:")
    print(pd.crosstab(kk["kategori"], kk["asal"], margins=True).to_string())
    g3 = bendera_kelompok(ada, kk, "bab3")
    pm = df[KOL_TEKS].str.contains(KATA_PASAR)
    pol = df[KOL_TEKS].str.contains(KATA_POLITIK)
    print("\nTotal kontrol terhadap sel B.1 (pola KATA_PASAR dan KATA_POLITIK):")
    print(
        f"pasar modal {int(pm.sum())} | politik {int(pol.sum())} | keduanya {int((pm & pol).sum())} | tanpa keduanya {int((~pm & ~pol).sum())}"
    )
    if (pm != g3["Pasar modal"]).any() or (pol != g3["Peristiwa politik"]).any():
        raise SystemExit(
            "GAGAL: kelompok bab3 pada KAMUS tidak sama dengan KATA_PASAR atau KATA_POLITIK."
        )
    simpan(kk, "kk_0_daftar_kata_kunci.csv")

    judul("1. CAKUPAN KATA KUNCI (kolom bab3 adalah acuan Tabel 4.3)")
    t_cak = tabel_cakupan(ada, kk)
    print(t_cak.astype(object).where(t_cak.notna(), "-").to_string(index=False))
    simpan(t_cak, "kk_1_cakupan.csv")

    judul("2. CAKUPAN KELOMPOK KATA KUNCI PER SUB-PERIODE")
    t_per = tabel_cakupan_periode(ada, kk, periode)
    for basis in BASIS:
        print(f"\nBasis {basis} (persen dokumen pada sub-periode):")
        print(
            t_per[t_per["basis"] == basis]
            .pivot(index="kelompok", columns="periode", values="persen")
            .reindex(columns=PERIODE)
            .to_string()
        )
    simpan(t_per, "kk_2_cakupan_per_periode.csv")

    judul("3. FREKUENSI PER KATA KUNCI")
    t_kk = tabel_kata_kunci(jml, kk, periode)
    print(t_kk.to_string(index=False))
    simpan(t_kk, "kk_3_frekuensi_kata_kunci.csv")

    semua, isi, target, sah = susun_token(teks, kk, pola)

    judul(f"4. KONTEKS KATA KUNCI (kata di sekitar, jendela {JENDELA} kata isi)")
    t_kol, _ = tabel_kolokasi(isi, target, kk)
    for k in kk.loc[kk["asal"].isin(["bab3", "kueri"]), "kata_kunci"]:
        s = t_kol[t_kol["kata_kunci"] == k]
        if len(s):
            daftar = ", ".join(
                f"{x} ({n})" for x, n in zip(s["kata_konteks"], s["n_bersama"])
            )
            print(f"\n[{k}] {daftar}")
    simpan(t_kol, "kk_4_konteks_kolokasi.csv")

    judul("5. KATA DAN PASANGAN KATA YANG PALING SERING MUNCUL")
    t_uni, t_bi, uni = tabel_kata_umum(semua, isi, sah)
    print(t_uni.head(N_UNIGRAM).to_string(index=False))
    print()
    print(t_bi.head(N_BIGRAM).to_string(index=False))
    simpan(t_uni, "kk_5a_kata_tunggal.csv")
    simpan(t_bi, "kk_5b_pasangan_kata.csv")

    judul("6. SILANG KATA KUNCI PASAR MODAL DAN PERISTIWA POLITIK (jumlah dokumen)")
    silang = tabel_silang(ada, t_kk)
    print(silang.to_string())
    silang.rename_axis("pasar_modal").reset_index().to_csv(
        OUT / "kk_6_silang_pasar_politik.csv", index=False, encoding="utf-8-sig"
    )
    print("Disimpan: kk_6_silang_pasar_politik.csv")

    judul(f"7. KATA KUNCI HARIAN (basis {BASIS_GAMBAR}, kalender)")
    h = tabel_harian(df, ada, kk)
    print(
        f"Hari kalender: {len(h)} | hari tanpa dokumen: {int((h['n_dokumen'] == 0).sum())}"
    )
    print("\nLima hari dengan dokumen demonstrasi terbanyak:")
    print(
        h.nlargest(5, "n_demonstrasi")[
            ["tanggal", "n_dokumen", "n_demonstrasi", "persen_demonstrasi"]
        ].to_string(index=False)
    )
    simpan(h, "kk_7_kata_kunci_harian.csv")

    judul("8. CUPLIKAN KONTEKS (KWIC)")
    t_kwic = tabel_kwic(df, ada, kk, pola)
    simpan(t_kwic, "kk_8_cuplikan_kwic.csv")

    judul("9. GAMBAR")
    gambar_frekuensi(t_kk)
    gambar_kata_umum(t_uni, t_bi)
    gambar_periode(t_kk)
    gambar_harian(h)
    gambar_silang(silang)
    for nama in (
        "gambar_kk_1_frekuensi_kata_kunci.png",
        "gambar_kk_2_kata_sering_muncul.png",
        "gambar_kk_3_kata_kunci_per_periode.png",
        "gambar_kk_4_kata_kunci_harian.png",
        "gambar_kk_5_silang_pasar_politik.png",
    ):
        print(f"Disimpan: {nama}" if (OUT / nama).exists() else f"TIDAK DIBUAT: {nama}")
    gambar_awan(uni)
    print(f"\nSeluruh keluaran berada di: {OUT}")


main()


0. DATA, DAFTAR KATA KUNCI, DAN TOTAL KONTROL
Berkas  : dataset_multilabel_final.csv
Dokumen : 14050 | 2025-06-01 s.d. 2025-10-31
Dokumen per sub-periode (basis kalender):
periode
Pra-peristiwa       7605
Puncak peristiwa    1998
Pasca-peristiwa     4447

Jumlah kata kunci per kategori dan asal:
asal                      bab3  kueri  saringan  varian  All
kategori                                                    
Demonstrasi                  2      1         0       8   11
Ekonomi makro                0      3         0       0    3
Pasar modal                  3      5        12       4   24
Politik dan pemerintahan     1      4         0       2    7
All                          6     13        12      14   45

Total kontrol terhadap sel B.1 (pola KATA_PASAR dan KATA_POLITIK):
pasar modal 13566 | politik 633 | keduanya 612 | tanpa keduanya 463
Disimpan: kk_0_daftar_kata_kunci.csv (45 baris)

1. CAKUPAN KATA KUNCI (kolom bab3 adalah acuan Tabel 4.3)
                                